# PIC16F877A · USART, I2C, SPI, PWM, interrupciones e ISR, multitarea
## Laboratorio en Colab con 12 ejemplos

Este cuaderno acompaña las diapositivas, los esquemáticos de Proteus y el simulador web. Todo corre en Python:

1. Un **ensamblador** con la sintaxis de MPASM y un **emulador** del PIC16F877A: núcleo de 35 instrucciones con su temporización, interrupciones, TMR0/1/2, PWM de 10 bits (CCP1 y CCP2), ADC, **USART**, **MSSP en SPI e I2C maestro**. Modelos de **LCD 16x2, PCF8574, DS1307, 74HC595 y MAX7219**.
2. Los **12 programas** en ensamblador (y en C para XC8). Aquí se ensamblan y producen los `.hex` para **Proteus**.
3. Cada ejemplo se **simula y se grafica**: tramas UART, transacciones I2C, bytes SPI, formas de onda PWM, la línea de tiempo de las interrupciones y el diagrama de Gantt de las tareas.

> *Entorno de ejecución → Ejecutar todas*. Las celdas `%%writefile` solo guardan archivos.

| # | Tema | Ejemplo | Lo que se aprende |
|---|---|---|---|
| 1 | USART | Terminal de comandos con interrupción de recepción | USART asíncrona 9600 8N1, interrupción RCIF, protocolo de comandos |
| 2 | USART | Telemetría: el PIC envía mediciones cada 250 ms | formato de texto para el PC, ADC de 10 bits, graficar datos seriales en Python |
| 3 | I2C | I2C: 8 LED en un expansor PCF8574 | trama I2C: START, dirección, ACK, dato, STOP, MSSP en I2C maestro, expansión de puertos |
| 4 | I2C | I2C: reloj de tiempo real DS1307 en LCD | lectura y escritura de registros I2C, RESTART y NACK, números BCD |
| 5 | SPI | SPI: 8 LED con un registro de desplazamiento 74HC595 | SPI maestro, registro de desplazamiento y latch, más salidas con 3 pines |
| 6 | SPI | SPI: display de 8 dígitos con MAX7219 | tramas SPI de 16 bits, registros de configuración, decodificación BCD |
| 7 | PWM | PWM: dos canales de 10 bits a 1 kHz | PWM de 10 bits (CCPRxL + CCPxCON<5:4>), periodo con PR2 y TMR2, dos canales a la vez |
| 8 | PWM | PWM: melodía en un parlante | frecuencia variable con PR2, tablas de notas y duraciones, sonido con PWM |
| 9 | Interrupciones | Interrupciones: INT, cambio en RB, ADC y TMR0 | vector 0x004 y contexto (W, STATUS, PCLATH), INTF, RBIF, ADIF, T0IF, copia atómica de datos compartidos |
| 10 | Interrupciones | Interrupciones: reloj HH:MM:SS con TMR1 | cálculo de precarga de TMR1, interrupción de periférico (PEIE), sondeo frente a interrupción |
| 11 | Multitarea | Multitarea: planificador cooperativo con tick de 10 ms | tick de sistema, tareas periódicas sin retardos, por qué un PIC16 no usa un RTOS con desalojo |
| 12 | Multitarea | Multitarea: semáforo, latido y comandos serie | máquinas de estado no bloqueantes, tareas con periodos distintos, integrar USART e interrupciones |

## 1. Herramientas: ensamblador, emulador y dispositivos

In [ ]:
%%writefile picasm.py
"""picasm: ensamblador didáctico para PIC16F877/877A (subconjunto de MPASM).
Directivas: LIST (p=, r=), #include (se ignora: los símbolos del PIC16F877A ya están cargados), #define, __CONFIG, ORG, EQU, SET,
CBLOCK/ENDC, DT, DA, BANKSEL, PAGESEL, ERRORLEVEL, END. Expresiones con + - * / % << >> & | ^ ~ ( ), HIGH, LOW, $.
Números: 0x1F, h'1F', b'0101', d'25', .25, 'A' y la base por defecto de LIST r= (hex si no se indica)."""
import re

SFR = dict(INDF=0x00, TMR0=0x01, PCL=0x02, STATUS=0x03, FSR=0x04, PORTA=0x05, PORTB=0x06, PORTC=0x07, PORTD=0x08, PORTE=0x09,
           PCLATH=0x0A, INTCON=0x0B, PIR1=0x0C, PIR2=0x0D, TMR1L=0x0E, TMR1H=0x0F, T1CON=0x10, TMR2=0x11, T2CON=0x12,
           SSPBUF=0x13, SSPCON=0x14, CCPR1L=0x15, CCPR1H=0x16, CCP1CON=0x17, RCSTA=0x18, TXREG=0x19, RCREG=0x1A,
           CCPR2L=0x1B, CCPR2H=0x1C, CCP2CON=0x1D, ADRESH=0x1E, ADCON0=0x1F,
           OPTION_REG=0x81, TRISA=0x85, TRISB=0x86, TRISC=0x87, TRISD=0x88, TRISE=0x89, PIE1=0x8C, PIE2=0x8D, PCON=0x8E,
           SSPCON2=0x91, PR2=0x92, SSPADD=0x93, SSPSTAT=0x94, TXSTA=0x98, SPBRG=0x99, CMCON=0x9C, CVRCON=0x9D, ADRESL=0x9E, ADCON1=0x9F,
           EEDATA=0x10C, EEADR=0x10D, EEDATH=0x10E, EEADRH=0x10F, EECON1=0x18C, EECON2=0x18D)
BITS = dict(C=0, DC=1, Z=2, NOT_PD=3, NOT_TO=4, RP0=5, RP1=6, IRP=7,
            RBIF=0, INTF=1, T0IF=2, RBIE=3, INTE=4, T0IE=5, PEIE=6, GIE=7, TMR0IF=2, TMR0IE=5,
            TMR1IF=0, TMR2IF=1, CCP1IF=2, SSPIF=3, TXIF=4, RCIF=5, ADIF=6, PSPIF=7,
            TMR1IE=0, TMR2IE=1, CCP1IE=2, SSPIE=3, TXIE=4, RCIE=5, ADIE=6, PSPIE=7,
            TMR1ON=0, TMR1CS=1, NOT_T1SYNC=2, T1OSCEN=3, T1CKPS0=4, T1CKPS1=5, TMR2ON=2, T2CKPS0=0, T2CKPS1=1,
            PS0=0, PS1=1, PS2=2, PSA=3, T0SE=4, T0CS=5, INTEDG=6, NOT_RBPU=7,
            ADON=0, GO=2, GO_DONE=2, NOT_DONE=2, CHS0=3, CHS1=4, CHS2=5, ADCS0=6, ADCS1=7, ADFM=7, PCFG0=0, PCFG1=1, PCFG2=2, PCFG3=3,
            RX9D=0, OERR=1, FERR=2, ADDEN=3, CREN=4, SREN=5, RX9=6, SPEN=7, TX9D=0, TRMT=1, BRGH=2, SYNC=4, TXEN=5, TX9=6, CSRC=7,
            BF=0, CKE=6, SMP=7, SSPM0=0, SSPM1=1, SSPM2=2, SSPM3=3, CKP=4, SSPEN=5, SSPOV=6, WCOL=7,
            RD=0, WR=1, WREN=2, WRERR=3, EEPGD=7, CCP1M0=0, CCP1M1=1, CCP1M2=2, CCP1M3=3, CCP1Y=4, CCP1X=5,
            SEN=0, RSEN=1, PEN=2, RCEN=3, ACKEN=4, ACKDT=5, ACKSTAT=6, GCEN=7, R_W=2, D_A=5, UA=1,
            F=1, W=0)
CONFIG = dict(_CP_OFF=0x3FFF, _CP_ALL=0x1FFF, _DEBUG_OFF=0x3FFF, _DEBUG_ON=0x37FF, _WRT_OFF=0x3FFF, _CPD_OFF=0x3FFF, _CPD_ON=0x3EFF,
              _LVP_OFF=0x3F7F, _LVP_ON=0x3FFF, _BOREN_OFF=0x3FBF, _BOREN_ON=0x3FFF, _BODEN_OFF=0x3FBF, _BODEN_ON=0x3FFF,
              _PWRTE_ON=0x3FF7, _PWRTE_OFF=0x3FFF, _WDT_OFF=0x3FFB, _WDTE_OFF=0x3FFB, _WDT_ON=0x3FFF, _WDTE_ON=0x3FFF,
              _LP_OSC=0x3FFC, _XT_OSC=0x3FFD, _HS_OSC=0x3FFE, _RC_OSC=0x3FFF)
BYTE = dict(ADDWF=0x0700, ANDWF=0x0500, COMF=0x0900, DECF=0x0300, DECFSZ=0x0B00, INCF=0x0A00, INCFSZ=0x0F00, IORWF=0x0400,
            MOVF=0x0800, RLF=0x0D00, RRF=0x0C00, SUBWF=0x0200, SWAPF=0x0E00, XORWF=0x0600)
BIT = dict(BCF=0x1000, BSF=0x1400, BTFSC=0x1800, BTFSS=0x1C00)
LIT = dict(ADDLW=0x3E00, ANDLW=0x3900, IORLW=0x3800, MOVLW=0x3000, RETLW=0x3400, SUBLW=0x3C00, XORLW=0x3A00)
SOLO = dict(NOP=0x0000, CLRW=0x0100, RETURN=0x0008, RETFIE=0x0009, SLEEP=0x0063, CLRWDT=0x0064)


class _Op:
    def __init__(self, f): self.f = f
    def __matmul__(self, v): return self.f(v)


_UNA = {'_hi': _Op(lambda v: (v >> 8) & 0xFF), '_lo': _Op(lambda v: v & 0xFF), '_up': _Op(lambda v: (v >> 16) & 0xFF)}


class ErrorASM(Exception):
    pass


class Ensamblador:
    def __init__(self, fuente):
        self.fuente = fuente
        self.radix = 16
        self.mem = {}
        self.conf = None
        self.lista = []

    # ---------- números y expresiones ----------
    def numero(self, t):
        t = t.strip(); u = t.upper()
        m = re.fullmatch(r"([HDBO])'([0-9A-F]+)'", u)
        if m: return int(m.group(2), {'H': 16, 'D': 10, 'B': 2, 'O': 8}[m.group(1)])
        if u.startswith('0X'): return int(u[2:], 16)
        if u.startswith('.') and u[1:].isdigit(): return int(u[1:])
        if re.fullmatch(r'[01]+B', u) and self.radix != 16: return int(u[:-1], 2)
        if re.fullmatch(r'[0-9A-F]+H', u) and u[0].isdigit(): return int(u[:-1], 16)
        if re.fullmatch(r'[0-9A-F]+', u) and u[0].isdigit(): return int(u, self.radix)
        return None

    def valor(self, expr, pasada, pc=0):
        expr = expr.strip()
        if not expr: raise ErrorASM('operando vacío')
        expr = re.sub(r"'(.)'", lambda m: str(ord(m.group(1))), expr)
        toks = re.findall(r"[HDBOhdbo]'[0-9A-Fa-f]+'|0[xX][0-9A-Fa-f]+|\.\d+|[A-Za-z_][A-Za-z0-9_]*|\d[0-9A-Fa-f]*[hH]?|<<|>>|[-+*/%&|^~()$]", expr)
        out = []
        for tk in toks:
            u = tk.upper()
            if tk in ('<<', '>>') or tk in '+-*/%&|^~()': out.append('//' if tk == '/' else tk); continue
            if tk == '$': out.append(str(pc)); continue
            if u == 'HIGH': out.append('_hi @'); continue
            if u == 'LOW': out.append('_lo @'); continue
            if u == 'UPPER': out.append('_up @'); continue
            n = self.numero(tk)
            if n is not None and not (u in self.simbolos or u in SFR or u in BITS): out.append(str(n)); continue
            if u in self.simbolos: out.append(str(self.simbolos[u])); continue
            if u in SFR: out.append(str(SFR[u])); continue
            if u in BITS: out.append(str(BITS[u])); continue
            if u in CONFIG: out.append(str(CONFIG[u])); continue
            if n is not None: out.append(str(n)); continue
            if pasada == 1: out.append('0'); continue
            raise ErrorASM(f'símbolo no definido: {tk}')
        try:
            return int(eval(' '.join(out), {'__builtins__': {}}, _UNA))
        except ErrorASM:
            raise
        except Exception:
            raise ErrorASM(f'expresión inválida: {expr}')

    @staticmethod
    def partir(ops):
        res, cur, q = [], '', None
        for ch in ops:
            if ch in '"\'' and (q is None or q == ch): q = None if q else ch
            if ch == ',' and not q: res.append(cur.strip()); cur = ''
            else: cur += ch
        if cur.strip(): res.append(cur.strip())
        return res

    # ---------- ensamblado en dos pasadas ----------
    def ensamblar(self):
        lineas = self.fuente.splitlines()
        for pasada in (1, 2):
            self.simbolos = {} if pasada == 1 else self.simbolos
            self.defines = {}
            self.mem = {}; self.lista = []; self.pc = 0; self.cblock = None; self.radix = 16
            for nlin, linea in enumerate(lineas, 1):
                try:
                    if self.linea(linea, pasada, nlin) == 'END': break
                except ErrorASM as e:
                    raise ErrorASM(f'línea {nlin}: {e}\n    {linea.strip()}')
        return self

    def linea(self, linea, pasada, nlin):
        # quitar comentario (respetando comillas)
        s, q = '', False
        for ch in linea:
            if ch in '"\'': q = not q
            if ch == ';' and not q: break
            s += ch
        if not s.strip(): return
        st = s.strip()
        if st.upper().startswith('#DEFINE'):
            p = st.split(None, 2); self.defines[p[1].upper()] = p[2] if len(p) > 2 else '1'; return
        if st.upper().startswith('#INCLUDE') or st.upper().startswith('ERRORLEVEL') or st.upper().startswith('PROCESSOR'): return
        for k, v in self.defines.items():
            s = re.sub(r'(?i)\b' + re.escape(k) + r'\b', v, s)
        etiqueta = None
        if not s[0].isspace():
            m = re.match(r'([A-Za-z_][A-Za-z0-9_]*):?\s*(.*)', s)
            etiqueta, s = m.group(1), m.group(2)
        partes = s.strip().split(None, 1)
        mn = partes[0].upper() if partes else ''
        ops = partes[1] if len(partes) > 1 else ''
        # el modo CBLOCK: cada línea es una lista de nombres
        if self.cblock is not None and mn != 'ENDC':
            nombres = ([etiqueta] if etiqueta else []) + self.partir(s)
            for n in nombres:
                n = n.strip()
                if not n: continue
                m = re.match(r'([A-Za-z_]\w*)\s*(?::\s*(.+))?', n)
                if pasada == 1 and m.group(1).upper() in self.simbolos: raise ErrorASM(f'símbolo repetido: {m.group(1)}')
                self.simbolos[m.group(1).upper()] = self.cblock
                self.cblock += self.valor(m.group(2), pasada) if m.group(2) else 1
            return
        if mn in ('EQU', 'SET'):
            self.simbolos[etiqueta.upper()] = self.valor(ops, pasada, self.pc); return
        if etiqueta:
            if pasada == 1 and etiqueta.upper() in self.simbolos and etiqueta.upper() not in ('_',):
                raise ErrorASM(f'etiqueta repetida: {etiqueta}')
            self.simbolos[etiqueta.upper()] = self.pc
        if not mn: return
        if mn == 'LIST':
            m = re.search(r'(?i)r\s*=\s*(\w+)', ops)
            if m: self.radix = {'DEC': 10, 'HEX': 16, 'OCT': 8}[m.group(1).upper()]
            return
        if mn == 'END': return 'END'
        if mn == 'ORG': self.pc = self.valor(ops, pasada); return
        if mn == 'CBLOCK': self.cblock = self.valor(ops, pasada) if ops.strip() else 0; return
        if mn == 'ENDC': self.cblock = None; return
        if mn == '__CONFIG':
            args = self.partir(ops); self.conf = self.valor(args[-1], pasada) & 0x3FFF; return
        if mn in ('DT', 'DA'):
            for a in self.partir(ops):
                if a.startswith('"'):
                    for ch in a.strip('"'): self.emitir(0x3400 | ord(ch), f'retlw {ch!r}', nlin, pasada, s)
                else:
                    self.emitir(0x3400 | (self.valor(a, pasada, self.pc) & 0xFF), 'retlw', nlin, pasada, s)
            return
        if mn == 'BANKSEL':
            a = self.valor(ops, pasada)
            self.emitir((0x1683 if a & 0x80 else 0x1283), '', nlin, pasada, s)
            self.emitir((0x1703 if a & 0x100 else 0x1303), '', nlin, pasada, s); return
        if mn == 'PAGESEL':
            a = self.valor(ops, pasada)
            self.emitir((0x158A if a & 0x800 else 0x118A), '', nlin, pasada, s)
            self.emitir((0x160A if a & 0x1000 else 0x120A), '', nlin, pasada, s); return
        args = self.partir(ops)
        V = lambda e: self.valor(e, pasada, self.pc)
        if mn in SOLO: w = SOLO[mn]
        elif mn in BYTE:
            d = 1 if len(args) < 2 else (V(args[1]) & 1)
            w = BYTE[mn] | (d << 7) | (V(args[0]) & 0x7F)
        elif mn == 'CLRF': w = 0x0180 | (V(args[0]) & 0x7F)
        elif mn == 'MOVWF': w = 0x0080 | (V(args[0]) & 0x7F)
        elif mn in BIT:
            if len(args) != 2: raise ErrorASM(f'{mn} necesita registro,bit')
            b = V(args[1])
            if not 0 <= b <= 7: raise ErrorASM('bit fuera de rango')
            w = BIT[mn] | (b << 7) | (V(args[0]) & 0x7F)
        elif mn in LIT: w = LIT[mn] | (V(args[0]) & 0xFF)
        elif mn in ('CALL', 'GOTO'):
            k = V(args[0])
            w = (0x2000 if mn == 'CALL' else 0x2800) | (k & 0x7FF)
        else:
            raise ErrorASM(f'instrucción desconocida: {mn}')
        self.emitir(w, '', nlin, pasada, s)

    def emitir(self, w, txt, nlin, pasada, fuente):
        if self.pc in self.mem and pasada == 2: raise ErrorASM(f'dirección 0x{self.pc:04X} usada dos veces')
        self.mem[self.pc] = w & 0x3FFF
        if pasada == 2: self.lista.append((self.pc, w & 0x3FFF, fuente.strip()))
        self.pc += 1

    def hex(self):
        datos = {}
        for a, w in self.mem.items(): datos[2 * a] = w & 0xFF; datos[2 * a + 1] = w >> 8
        if self.conf is not None: datos[0x400E] = self.conf & 0xFF; datos[0x400F] = self.conf >> 8
        out = [':020000040000FA']
        dirs = sorted(datos); i = 0
        while i < len(dirs):
            ini = dirs[i]; bloque = [datos[ini]]; i += 1
            while i < len(dirs) and dirs[i] == ini + len(bloque) and len(bloque) < 16 and (ini + len(bloque)) % 16:
                bloque.append(datos[dirs[i]]); i += 1
            reg = [len(bloque), ini >> 8, ini & 0xFF, 0] + bloque
            out.append(':' + ''.join(f'{b:02X}' for b in reg) + f'{(-sum(reg)) & 0xFF:02X}')
        out.append(':00000001FF')
        return '\n'.join(out) + '\n'

    def palabras(self):
        return len([a for a in self.mem if a < 0x2000])


def leer_hex(texto):
    datos, base = {}, 0
    for l in texto.split():
        if not l.startswith(':'): continue
        b = bytes.fromhex(l[1:])
        if sum(b) & 0xFF: raise ValueError('suma de verificación incorrecta: ' + l)
        n, a, t = b[0], (b[1] << 8) | b[2], b[3]
        if t == 0:
            for k in range(n): datos[base + a + k] = b[4 + k]
        elif t == 4: base = ((b[4] << 8) | b[5]) << 16
        elif t == 1: break
    prog, conf = {}, None
    for a in sorted(datos):
        if a % 2 == 0 and a + 1 in datos:
            w = datos[a] | (datos[a + 1] << 8)
            if a // 2 == 0x2007: conf = w
            elif a // 2 < 0x2000: prog[a // 2] = w & 0x3FFF
    return prog, conf


_INV = {v: k for k, v in SFR.items() if v < 0x80}
_INV1 = {v & 0x7F: k for k, v in SFR.items() if 0x80 <= v < 0x100}


def desensamblar(w):
    if w == 0x0008: return 'return'
    if w == 0x0009: return 'retfie'
    if w == 0x0063: return 'sleep'
    if w == 0x0064: return 'clrwdt'
    if w == 0x0100: return 'clrw'
    top = w & 0x3000; f = w & 0x7F; fn = _INV.get(f, f'0x{f:02X}')
    if top == 0:
        o = w & 0x0F00; d = (w >> 7) & 1
        if o == 0: return f'movwf  {fn}' if w & 0x80 else 'nop'
        if o == 0x0100: return f'clrf   {fn}'
        n = {v: k for k, v in BYTE.items()}.get(o, '??').lower()
        return f'{n:<7s}{fn},{"F" if d else "W"}'
    if top == 0x1000:
        n = ['bcf', 'bsf', 'btfsc', 'btfss'][(w >> 10) & 3]
        return f'{n:<7s}{fn},{(w >> 7) & 7}'
    if top == 0x2000: return f'{"goto" if w & 0x800 else "call":<7s}0x{w & 0x7FF:03X}'
    k = w & 0xFF
    for n, c in (('movlw', 0x3000), ('retlw', 0x3400)):
        if w & 0x3C00 == c: return f'{n:<7s}0x{k:02X}'
    for n, c in (('iorlw', 0x3800), ('andlw', 0x3900), ('xorlw', 0x3A00)):
        if w & 0x3F00 == c: return f'{n:<7s}0x{k:02X}'
    if w & 0x3E00 == 0x3C00: return f'sublw  0x{k:02X}'
    return f'addlw  0x{k:02X}'

In [ ]:
%%writefile pic16sim.py
"""pic16sim: emulador didáctico del PIC16F877/877A.
Núcleo de 35 instrucciones con temporización real, interrupciones (INT/RB0, TMR0, periféricos), TMR0, TMR1, TMR2,
PWM del CCP1/CCP2, ADC de 10 bits, USART y MSSP en modo SPI maestro. Los dispositivos externos (LCD, OLED, teclado...)
están en dispositivos.py y se conectan con escuchas de pines."""

PUERTOS = 'ABCDE'


class PIC16F877:
    def __init__(self, prog, fosc=4e6):
        self.prog = [0x3FFF] * 8192
        for a, w in prog.items(): self.prog[a] = w
        self.fosc = fosc
        self.ent = {'A': 0xFF, 'B': 0xFF, 'C': 0xFF, 'D': 0xFF, 'E': 0xFF}   # nivel externo de las entradas
        self.fn_ent = {}            # puerto -> función(pic) que calcula las entradas (teclado, etc.)
        self.escuchas = []          # función(pic, puerto, anterior, nuevo) al cambiar una salida
        self.spi = []               # función(pic, byte) al transmitir por SPI
        self.i2c = None             # bus I2C (dispositivos.BusI2C) si hay esclavos conectados
        self.analog = [0.0] * 8
        self.reset()

    def reset(self):
        self.ram = [0] * 512
        self.w = 0; self.pc = 0; self.pila = []; self.cy = 0; self.dormido = False
        r = self.ram
        r[0x03] = 0x18; r[0x81] = 0xFF
        for t in (0x85, 0x86, 0x87, 0x88): r[t] = 0xFF
        r[0x89] = 0x07; r[0x92] = 0xFF; r[0x98] = 0x02; r[0x0C] = 0x10
        self.pre0 = 0; self.pre1 = 0; self.pre2 = 0; self.post2 = 0
        self.adc_fin = None; self.ssp_fin = None
        self.tsr = None; self.tsr_fin = 0; self.txreg_lleno = False; self.tx = []; self.rxq = []; self.rx_listo = 0
        self.salidas = {p: self.salida(p) for p in PUERTOS}
        self.rb0 = self.entrada_pin('B', 0)
        self.rbref = (self.valor_b() >> 4) & 0xF
        self.i2c_op = None

    # ---------- memoria ----------
    def dirf(self, f):
        if f == 0: f = self.ram[0x04] | ((self.ram[0x03] & 0x80) << 1)
        else: f |= (self.ram[0x03] & 0x60) << 2
        b = f & 0x7F
        if b in (0x00, 0x02, 0x03, 0x04, 0x0A, 0x0B) or b >= 0x70: return b
        if f in (0x101, 0x106, 0x181, 0x186): return f & 0xFF
        return f

    def entradas(self, p):
        return self.fn_ent[p](self) if p in self.fn_ent else self.ent[p]

    def leer(self, a):
        if 5 <= a <= 9:
            p = PUERTOS[a - 5]; tris = self.ram[a + 0x80]
            v = ((self.ram[a] & ~tris) | (self.entradas(p) & tris)) & 0xFF
            if p == 'B': self.rbref = (v >> 4) & 0xF          # la lectura de PORTB termina la condición de cambio
            return v
        if a == 0x02: return self.pc & 0xFF
        if a == 0x1A: self.ram[0x0C] &= ~0x20
        if a == 0x13: self.ram[0x94] &= ~0x01
        return self.ram[a]

    def valor_b(self):
        tris = self.ram[0x86]
        return ((self.ram[6] & ~tris) | (self.entradas('B') & tris)) & 0xFF

    def salida(self, p):
        a = 5 + PUERTOS.index(p); tris = self.ram[a + 0x80]
        return self.ram[a] & ~tris & 0xFF   # pines en alta impedancia se ven en 0

    def entrada_pin(self, p, b):
        a = 5 + PUERTOS.index(p)
        return (self.leer(a) >> b) & 1

    def poner_entrada(self, p, v):
        """Cambia el nivel externo de un puerto (pulsadores) y detecta el flanco de RB0/INT."""
        self.ent[p] = v & 0xFF
        self.revisar_int()

    def revisar_int(self):
        v = self.valor_b(); n = v & 1
        if (((v >> 4) ^ self.rbref) & (self.ram[0x86] >> 4) & 0xF): self.ram[0x0B] |= 0x01   # RBIF
        if n != self.rb0 and (self.ram[0x86] & 1):
            sube = (self.ram[0x81] >> 6) & 1
            if (n == 1 and sube) or (n == 0 and not sube): self.ram[0x0B] |= 0x02
        self.rb0 = n

    def escribir(self, a, v):
        v &= 0xFF; r = self.ram
        if a == 0x02:
            r[2] = v; self.pc = ((r[0x0A] & 0x1F) << 8) | v; return 1
        if a == 0x03: v = (r[3] & 0x18) | (v & 0xE7)
        if a == 0x01: self.pre0 = 0
        r[a] = v
        if 5 <= a <= 9 or 0x85 <= a <= 0x89:
            p = PUERTOS[(a & 0x7F) - 5]; n = self.salida(p); ant = self.salidas[p]
            if n != ant:
                self.salidas[p] = n
                for e in self.escuchas: e(self, p, ant, n)
            if p == 'B': self.revisar_int()
        elif a == 0x1F:
            if (v & 0x05) == 0x05 and self.adc_fin is None:
                adcs = ((r[0x9F] >> 4) & 4) | ((v >> 6) & 3)
                div = {0: 2, 1: 8, 2: 32, 3: 0, 4: 4, 5: 16, 6: 64, 7: 0}[adcs]
                tad = 4e-6 if div == 0 else div / self.fosc
                self.adc_fin = self.cy + int(12 * tad * self.fosc / 4) + 1
        elif a == 0x19:
            if self.tsr is None: self.tsr = v; self.tsr_fin = self.cy + self.ciclos_char()
            else: self.txreg_lleno = True; r[0x0C] &= ~0x10
        elif a == 0x13 and (r[0x14] & 0x20) and (r[0x14] & 0x0F) == 0x08:
            ack = self.i2c.escribir(v) if self.i2c else False
            self.i2c_op = ('TX', self.cy + 9 * (r[0x93] + 1), ack)
        elif a == 0x91 and (r[0x14] & 0x2F) == 0x28 and self.i2c_op is None:
            per = r[0x93] + 1
            if v & 0x01: self.i2c_op = ('SEN', self.cy + per, None)
            elif v & 0x02: self.i2c_op = ('RSEN', self.cy + per, None)
            elif v & 0x04: self.i2c_op = ('PEN', self.cy + per, None)
            elif v & 0x08: self.i2c_op = ('RCEN', self.cy + 8 * per, None)
            elif v & 0x10: self.i2c_op = ('ACKEN', self.cy + per, None)
        elif a == 0x13 and (r[0x14] & 0x20):
            for e in self.spi: e(self, v)
            div = {0: 4, 1: 16, 2: 64}.get(r[0x14] & 0x0F, 4)
            self.ssp_fin = self.cy + 2 * div
        return 0

    def ciclos_char(self):
        brgh = (self.ram[0x98] >> 2) & 1
        baud = self.fosc / ((16 if brgh else 64) * (self.ram[0x99] + 1))
        return int(10 * (self.fosc / 4) / baud)

    def recibir(self, ch):
        self.rxq.extend(ch if isinstance(ch, (bytes, list)) else [ch])

    # ---------- periféricos ----------
    def tick(self, n):
        r = self.ram
        self.cy += n
        # TMR0
        op = r[0x81]
        if not (op & 0x20):
            pres = 1 if op & 0x08 else 2 << (op & 7)
            self.pre0 += n
            while self.pre0 >= pres:
                self.pre0 -= pres; r[1] = (r[1] + 1) & 0xFF
                if r[1] == 0: r[0x0B] |= 0x04
        # TMR1
        t1 = r[0x10]
        if t1 & 1:
            pres = 1 << ((t1 >> 4) & 3); self.pre1 += n
            while self.pre1 >= pres:
                self.pre1 -= pres; v = ((r[0x0F] << 8) | r[0x0E]) + 1
                if v > 0xFFFF: v = 0; r[0x0C] |= 0x01
                r[0x0E] = v & 0xFF; r[0x0F] = v >> 8
        # TMR2
        t2 = r[0x12]
        if t2 & 4:
            pres = (1, 4, 16, 16)[t2 & 3]; self.pre2 += n
            while self.pre2 >= pres:
                self.pre2 -= pres
                if r[0x11] == r[0x92]:
                    r[0x11] = 0; self.post2 += 1
                    if self.post2 >= ((t2 >> 3) & 15) + 1: self.post2 = 0; r[0x0C] |= 0x02
                else: r[0x11] = (r[0x11] + 1) & 0xFF
        # ADC
        if self.adc_fin is not None and self.cy >= self.adc_fin:
            ch = (r[0x1F] >> 3) & 7
            res = max(0, min(1023, round(self.analog[ch] / 5.0 * 1023)))
            if r[0x9F] & 0x80: r[0x1E] = res >> 8; r[0x9E] = res & 0xFF
            else: r[0x1E] = res >> 2; r[0x9E] = (res & 3) << 6
            r[0x1F] &= ~0x04; r[0x0C] |= 0x40; self.adc_fin = None
        # MSSP
        if self.ssp_fin is not None and self.cy >= self.ssp_fin:
            r[0x94] |= 0x01; r[0x0C] |= 0x08; self.ssp_fin = None
        # MSSP en I2C maestro
        if self.i2c_op is not None and self.cy >= self.i2c_op[1]:
            op, _, ack = self.i2c_op; self.i2c_op = None; b = self.i2c
            if op == 'SEN': r[0x91] &= ~0x01; b and b.start()
            elif op == 'RSEN': r[0x91] &= ~0x02; b and b.start()
            elif op == 'PEN': r[0x91] &= ~0x04; b and b.stop()
            elif op == 'TX': r[0x91] = (r[0x91] & ~0x40) | (0 if ack else 0x40)
            elif op == 'RCEN': r[0x91] &= ~0x08; r[0x13] = (b.leer() if b else 0xFF) & 0xFF; r[0x94] |= 0x01
            elif op == 'ACKEN': r[0x91] &= ~0x10; b and b.ack(not (r[0x91] & 0x20))
            r[0x0C] |= 0x08
        # USART
        if self.tsr is not None:
            r[0x98] &= ~0x02
            if self.cy >= self.tsr_fin:
                self.tx.append(self.tsr); self.tsr = None
                if self.txreg_lleno:
                    self.txreg_lleno = False; self.tsr = r[0x19]; self.tsr_fin = self.cy + self.ciclos_char()
                r[0x0C] |= 0x10
                if self.tsr is None: r[0x98] |= 0x02
        if self.rxq and (r[0x18] & 0x90) == 0x90 and not (r[0x0C] & 0x20) and self.cy >= self.rx_listo:
            r[0x1A] = self.rxq.pop(0); r[0x0C] |= 0x20; self.rx_listo = self.cy + self.ciclos_char()

    def interrupcion(self):
        r = self.ram; ic = r[0x0B]
        pend = (ic >> 3) & ic & 0x07
        if (ic & 0x40) and (r[0x8C] & r[0x0C]): pend = 1
        if not pend: return
        self.dormido = False
        if ic & 0x80:
            if len(self.pila) == 8: self.pila.pop(0)
            self.pila.append(self.pc); self.pc = 4; r[0x0B] &= ~0x80
            self.tick(2)

    # ---------- núcleo ----------
    def flags(self, z=None, c=None, dc=None):
        s = self.ram[3]
        if z is not None: s = (s | 4) if z else (s & ~4)
        if c is not None: s = (s | 1) if c else (s & ~1)
        if dc is not None: s = (s | 2) if dc else (s & ~2)
        self.ram[3] = s & 0xFF

    def paso(self):
        if self.dormido:
            self.tick(1); self.interrupcion(); return
        n = self.ejecutar()
        self.tick(n)
        self.interrupcion()

    def ejecutar(self):
        op = self.prog[self.pc]
        self.pc = (self.pc + 1) & 0x1FFF
        top = op & 0x3000
        if top == 0:
            f = op & 0x7F; d = (op >> 7) & 1; o = op & 0x0F00
            if o == 0:
                if op & 0x80: return 1 + self.escribir(self.dirf(f), self.w)
                if op in (0x08, 0x09):
                    self.pc = self.pila.pop() if self.pila else 0
                    if op == 0x09: self.ram[0x0B] |= 0x80
                    return 2
                if op == 0x63: self.dormido = True
                return 1
            if o == 0x0100:
                if op & 0x80: self.escribir(self.dirf(f), 0)
                else: self.w = 0
                self.flags(z=True); return 1
            a = self.dirf(f); v = self.leer(a); w = self.w; skip = False
            if o == 0x0700: r = v + w; self.flags(z=(r & 0xFF) == 0, c=r > 255, dc=((v & 15) + (w & 15)) > 15)
            elif o == 0x0500: r = v & w; self.flags(z=r == 0)
            elif o == 0x0900: r = ~v & 0xFF; self.flags(z=r == 0)
            elif o == 0x0300: r = (v - 1) & 0xFF; self.flags(z=r == 0)
            elif o == 0x0B00: r = (v - 1) & 0xFF; skip = r == 0
            elif o == 0x0A00: r = (v + 1) & 0xFF; self.flags(z=r == 0)
            elif o == 0x0F00: r = (v + 1) & 0xFF; skip = r == 0
            elif o == 0x0400: r = v | w; self.flags(z=r == 0)
            elif o == 0x0800: r = v; self.flags(z=r == 0)
            elif o == 0x0D00: r = ((v << 1) | (self.ram[3] & 1)) & 0xFF; self.flags(c=bool(v & 0x80))
            elif o == 0x0C00: r = (v >> 1) | ((self.ram[3] & 1) << 7); self.flags(c=bool(v & 1))
            elif o == 0x0200: r = (v - w) & 0xFF; self.flags(z=r == 0, c=v >= w, dc=(v & 15) >= (w & 15))
            elif o == 0x0E00: r = ((v << 4) | (v >> 4)) & 0xFF
            elif o == 0x0600: r = v ^ w; self.flags(z=r == 0)
            else: raise ValueError(f'código {op:04X} inválido en {self.pc - 1:04X}')
            r &= 0xFF; extra = 0
            if d: extra = self.escribir(a, r)
            else: self.w = r
            if skip: self.pc = (self.pc + 1) & 0x1FFF; return 2
            return 1 + extra
        if top == 0x1000:
            f = op & 0x7F; b = (op >> 7) & 7; o = op & 0x0C00; a = self.dirf(f)
            if o == 0: self.escribir(a, self.ram[a] & ~(1 << b) if not 5 <= a <= 9 else self.leer(a) & ~(1 << b)); return 1
            if o == 0x400: self.escribir(a, self.ram[a] | (1 << b) if not 5 <= a <= 9 else self.leer(a) | (1 << b)); return 1
            bit = (self.leer(a) >> b) & 1
            if (o == 0x800 and not bit) or (o == 0xC00 and bit): self.pc = (self.pc + 1) & 0x1FFF; return 2
            return 1
        if top == 0x2000:
            k = op & 0x7FF
            if not op & 0x800:
                if len(self.pila) == 8: self.pila.pop(0)
                self.pila.append(self.pc)
            self.pc = ((self.ram[0x0A] & 0x18) << 8) | k
            return 2
        k = op & 0xFF; s = op & 0x3F00
        if s & 0x3C00 == 0x3000: self.w = k
        elif s & 0x3C00 == 0x3400: self.w = k; self.pc = self.pila.pop() if self.pila else 0; return 2
        elif s == 0x3800: self.w |= k; self.flags(z=self.w == 0)
        elif s == 0x3900: self.w &= k; self.flags(z=self.w == 0)
        elif s == 0x3A00: self.w ^= k; self.flags(z=self.w == 0)
        elif s & 0x3E00 == 0x3C00:
            r = k - self.w; self.flags(c=r >= 0, dc=(k & 15) >= (self.w & 15)); self.w = r & 0xFF; self.flags(z=self.w == 0)
        else:
            r = k + self.w; self.flags(c=r > 255, dc=((k & 15) + (self.w & 15)) > 15); self.w = r & 0xFF; self.flags(z=self.w == 0)
        return 1

    def correr(self, ciclos=None, segundos=None):
        if segundos is not None: ciclos = int(segundos * self.fosc / 4)
        fin = self.cy + ciclos
        while self.cy < fin: self.paso()

    def tiempo(self):
        return self.cy * 4 / self.fosc

    def puerto(self, p):
        return self.leer(5 + PUERTOS.index(p))

    def pwm(self, n=1):
        ccpcon, ccprl, pin = (0x17, 0x15, 2) if n == 1 else (0x1D, 0x1B, 1)
        t2 = self.ram[0x12]
        if (self.ram[ccpcon] & 0x0C) != 0x0C or not (t2 & 0x04): return None
        pres = (1, 4, 16, 16)[t2 & 3]
        per = (self.ram[0x92] + 1) * 4 * pres / self.fosc
        dc = ((self.ram[ccprl] << 2) | ((self.ram[ccpcon] >> 4) & 3)) * pres / self.fosc
        return {'f': 1 / per, 'D': min(1.0, dc / per), 'salida': not (self.ram[0x87] >> pin) & 1}

In [ ]:
%%writefile dispositivos.py
"""Modelos de los dispositivos externos que usan los ejemplos (equivalentes a los de Proteus)."""


def bit(v, b): return (v >> b) & 1


class LCD16x2:
    """HD44780 (LM016L en Proteus) en modo de 4 bits: D4-D7 = bits 4-7 del puerto, RS y E en el mismo puerto. RW a tierra."""
    def __init__(self, pic, puerto='D', rs=0, e=1):
        self.p, self.rs, self.e = puerto, rs, e
        self.ddram = [0x20] * 128; self.cgram = [0] * 64
        self.addr = 0; self.cg = False; self.modo8 = True; self.mitad = None; self.inc = 1
        self.encendido = False; self.ocupado = 0.0; self.errores = []; self.comandos = 0
        pic.escuchas.append(self.cambio)

    def cambio(self, pic, p, ant, n):
        if p != self.p or not (bit(ant, self.e) and not bit(n, self.e)): return
        t = pic.tiempo(); rs = bit(n, self.rs); nib = n >> 4
        if t < 0.015 and not self.errores: self.errores.append(f'{t*1e3:.2f} ms: escritura antes de 15 ms del encendido')
        if self.modo8:
            self.ejecutar(rs, nib << 4, t); return
        if self.mitad is None:
            if t < self.ocupado: self.errores.append(f'{t*1e3:.3f} ms: LCD ocupado (faltan {(self.ocupado - t)*1e6:.0f} µs)')
            self.mitad = nib; return
        b = (self.mitad << 4) | nib; self.mitad = None
        self.ejecutar(rs, b, t)

    def ejecutar(self, rs, b, t):
        self.comandos += 1
        dur = 43e-6
        if rs:
            if self.cg: self.cgram[self.addr & 63] = b; self.addr = (self.addr + self.inc) & 63
            else: self.ddram[self.addr & 127] = b; self.addr = (self.addr + self.inc) & 127
        elif b & 0x80: self.addr = b & 0x7F; self.cg = False
        elif b & 0x40: self.addr = b & 0x3F; self.cg = True
        elif b & 0x20:
            if not (b & 0x10): self.modo8 = False
            else: self.modo8 = True
        elif b & 0x10: pass
        elif b & 0x08: self.encendido = bool(b & 0x04)
        elif b & 0x04: self.inc = 1 if b & 2 else -1
        elif b & 0x02: self.addr = 0; self.cg = False; dur = 1.52e-3
        elif b & 0x01: self.ddram = [0x20] * 128; self.addr = 0; self.cg = False; dur = 1.52e-3
        self.ocupado = t + dur

    def linea(self, i):
        base = 0 if i == 0 else 0x40
        return ''.join(chr(c) if 32 <= c < 127 else ('°' if c == 0xDF else '█' if c < 8 else '?') for c in self.ddram[base:base + 16])

    def texto(self):
        return [self.linea(0), self.linea(1)] if self.encendido else ['', '']


class OLED:
    """SSD1306 128x64 por SPI (UG-2864HSWEG01 en Proteus): CS, D/C y RES en el puerto indicado."""
    def __init__(self, pic, puerto='C', cs=0, dc=1, res=2):
        self.p, self.cs, self.dc, self.res = puerto, cs, dc, res
        self.mem = [[0] * 128 for _ in range(8)]
        self.col = 0; self.pag = 0; self.modo = 2; self.c0, self.c1, self.p0, self.p1 = 0, 127, 0, 7
        self.esperando = []; self.cmd = None; self.encendido = False; self.bytes = 0
        pic.spi.append(self.byte)

    def byte(self, pic, v):
        o = pic.salidas[self.p]
        if bit(o, self.cs) or not bit(o, self.res): return
        self.bytes += 1
        if bit(o, self.dc): self.dato(v)
        else: self.comando(v)

    def dato(self, v):
        self.mem[self.pag & 7][self.col & 127] = v
        if self.modo == 0:
            self.col += 1
            if self.col > self.c1:
                self.col = self.c0; self.pag += 1
                if self.pag > self.p1: self.pag = self.p0
        else:
            self.col = min(self.col + 1, 127)

    def comando(self, v):
        if self.cmd is not None:
            self.esperando.append(v)
            c, n = self.cmd
            if len(self.esperando) == n:
                a = self.esperando
                if c == 0x20: self.modo = a[0] & 3
                elif c == 0x21: self.c0, self.c1 = a[0] & 127, a[1] & 127; self.col = self.c0
                elif c == 0x22: self.p0, self.p1 = a[0] & 7, a[1] & 7; self.pag = self.p0
                self.cmd = None; self.esperando = []
            return
        if v in (0x20, 0x81, 0x8D, 0xA8, 0xD3, 0xD5, 0xD9, 0xDA, 0xDB): self.cmd = (v, 1)
        elif v in (0x21, 0x22): self.cmd = (v, 2)
        elif v == 0xAF: self.encendido = True
        elif v == 0xAE: self.encendido = False
        elif 0xB0 <= v <= 0xB7: self.pag = v & 7
        elif v <= 0x0F: self.col = (self.col & 0xF0) | v
        elif v <= 0x1F: self.col = ((v & 0x0F) << 4) | (self.col & 0x0F)

    def pixel(self, x, y): return bit(self.mem[y >> 3][x], y & 7)

    def ascii(self, paso=2):
        filas = []
        for y in range(0, 64, paso):
            filas.append(''.join('#' if any(self.pixel(x + i, y + j) for i in range(paso) for j in range(paso) if x + i < 128 and y + j < 64) else '.' for x in range(0, 128, paso)))
        return '\n'.join(filas)


class Teclado4x4:
    """KEYPAD-SMALLCALC: filas en RB0-RB3 (salidas), columnas en RB4-RB7 (entradas con pull-up)."""
    TECLAS = ['789/', '456*', '123-', 'C0=+']

    def __init__(self, pic):
        self.tecla = None
        pic.fn_ent['B'] = self.entradas

    def entradas(self, pic):
        v = 0xFF
        if self.tecla:
            for f, fila in enumerate(self.TECLAS):
                c = fila.find(self.tecla)
                if c >= 0 and not bit(pic.salidas['B'], f): v &= ~(1 << (4 + c))
        return v & 0xFF


class Pulso:
    """Mide el ancho y el periodo de los pulsos en un pin (servomotor)."""
    def __init__(self, pic, puerto, b):
        self.p, self.b = puerto, b; self.subida = None; self.ancho = None; self.periodo = None
        pic.escuchas.append(self.cambio)

    def cambio(self, pic, p, ant, n):
        if p != self.p or bit(ant, self.b) == bit(n, self.b): return
        t = pic.tiempo()
        if bit(n, self.b):
            if self.subida is not None: self.periodo = t - self.subida
            self.subida = t
        elif self.subida is not None: self.ancho = t - self.subida


class Paso:
    """Motor paso a paso unipolar por ULN2003: bobinas en los bits 0-3 del puerto (secuencia de medio paso)."""
    SEQ = [0b0001, 0b0011, 0b0010, 0b0110, 0b0100, 0b1100, 0b1000, 0b1001]

    def __init__(self, pic, puerto='D', grados=3.75):
        self.p = puerto; self.idx = None; self.pasos = 0; self.grados = grados; self.errores = 0
        pic.escuchas.append(self.cambio)

    def cambio(self, pic, p, ant, n):
        if p != self.p or (ant & 15) == (n & 15): return
        v = n & 15
        if v not in self.SEQ: return
        i = self.SEQ.index(v)
        if self.idx is not None:
            d = (i - self.idx) % 8
            if d == 1: self.pasos += 1
            elif d == 7: self.pasos -= 1
            elif d: self.errores += 1
        self.idx = i

    def angulo(self): return self.pasos * self.grados


class Multiplex7:
    """Display de 4 dígitos (7SEG-MPX4-CC): segmentos en PORTD (a = bit 0, dp = bit 7), comunes en RA0-RA3 activos en 0."""
    def __init__(self, pic, seg='D', dig='A'):
        self.seg, self.dig = seg, dig; self.vista = [0] * 4; self.cambios = 0
        pic.escuchas.append(self.cambio)

    def cambio(self, pic, p, ant, n):
        if p not in (self.seg, self.dig): return
        d = pic.salidas[self.dig]; s = pic.salidas[self.seg]
        for k in range(4):
            if not bit(d, k): self.vista[k] = s; self.cambios += 1

    SEGS = {0x3F: '0', 0x06: '1', 0x5B: '2', 0x4F: '3', 0x66: '4', 0x6D: '5', 0x7D: '6', 0x07: '7', 0x7F: '8', 0x6F: '9', 0: ' '}

    def texto(self):
        return ''.join(self.SEGS.get(v & 0x7F, '?') + ('.' if v & 0x80 else '') for v in self.vista)


# ---------------- buses y periféricos de comunicación ----------------
class BusI2C:
    """Bus I2C con esclavos por dirección de 7 bits. Guarda un registro de las transacciones."""
    def __init__(self, pic):
        self.dev = {}; self.actual = None; self.esperando_dir = False; self.leyendo = False; self.log = []
        pic.i2c = self

    def agregar(self, d): self.dev[d.dir] = d; return d
    def start(self): self.esperando_dir = True; self.log.append('S')
    def stop(self):
        if self.actual: self.actual.stop()
        self.actual = None; self.log.append('P')

    def escribir(self, b):
        if self.esperando_dir:
            self.esperando_dir = False; d = self.dev.get(b >> 1); self.leyendo = bool(b & 1)
            self.actual = d; self.log.append(f'{b:02X}{"+" if d else "-"}')
            if d: d.inicio(self.leyendo)
            return d is not None
        self.log.append(f'{b:02X}')
        return self.actual.escribir(b) if self.actual else False

    def leer(self):
        v = self.actual.leer() if self.actual else 0xFF
        self.log.append(f'r{v:02X}'); return v

    def ack(self, a): self.log.append('A' if a else 'N')


class PCF8574:
    """Expansor de 8 bits I2C (dirección 0x20 + A2..A0). Salidas cuasi bidireccionales."""
    def __init__(self, dir7=0x20):
        self.dir = dir7; self.salida = 0xFF; self.entradas = 0xFF; self.escrituras = 0
    def inicio(self, leer): pass
    def stop(self): pass
    def escribir(self, b): self.salida = b; self.escrituras += 1; return True
    def leer(self): return self.salida & self.entradas


class DS1307:
    """Reloj de tiempo real I2C (dirección 0x68). Registros en BCD; avanza con el tiempo simulado."""
    def __init__(self, pic):
        import datetime
        self.pic = pic; self.dir = 0x68; self.ram = [0] * 64; self.ptr = 0; self.primero = False
        self.dt = datetime.datetime(2000, 1, 1, 0, 0, 0); self.ch = True; self.t_ult = 0.0; self.ram[7] = 0x03
    def _avanzar(self):
        import datetime
        now = self.pic.tiempo(); d = now - self.t_ult
        if not self.ch and d >= 1:
            n = int(d); self.dt += datetime.timedelta(seconds=n); self.t_ult += n
        elif self.ch: self.t_ult = now
    def _regs(self):
        b = lambda v: ((v // 10) << 4) | (v % 10); d = self.dt
        return [b(d.second) | (0x80 if self.ch else 0), b(d.minute), b(d.hour), d.isoweekday() % 7 + 1, b(d.day), b(d.month), b(d.year % 100)]
    def inicio(self, leer): self.primero = not leer; self._avanzar()
    def stop(self): pass
    def escribir(self, v):
        import datetime
        if self.primero: self.ptr = v & 63; self.primero = False; return True
        self._avanzar()
        if self.ptr < 7:
            regs = self._regs(); regs[self.ptr] = v
            u = lambda x: (x >> 4) * 10 + (x & 15)
            self.ch = bool(regs[0] & 0x80)
            try: self.dt = datetime.datetime(2000 + u(regs[6]), max(1, u(regs[5])), max(1, u(regs[4])), u(regs[2] & 0x3F), u(regs[1]), u(regs[0] & 0x7F))
            except ValueError: pass
            self.t_ult = self.pic.tiempo()
        else: self.ram[self.ptr] = v
        self.ptr = (self.ptr + 1) & 63; return True
    def leer(self):
        self._avanzar()
        v = self._regs()[self.ptr] if self.ptr < 7 else self.ram[self.ptr]
        self.ptr = (self.ptr + 1) & 63; return v
    def texto(self):
        self._avanzar(); return self.dt.strftime('%H:%M:%S %d/%m/%y') + (' (detenido)' if self.ch else '')


class HC595:
    """74HC595: recibe bytes por SPI y los pasa a las salidas Q0..Q7 en el flanco de subida de ST_CP (latch)."""
    def __init__(self, pic, puerto='C', latch=0):
        self.p, self.l = puerto, latch; self.reg = 0; self.salida = 0; self.latches = 0
        pic.spi.append(lambda pi, v: setattr(self, 'reg', v)); pic.escuchas.append(self.cambio)
    def cambio(self, pic, p, ant, n):
        if p == self.p and not bit(ant, self.l) and bit(n, self.l): self.salida = self.reg; self.latches += 1


class MAX7219:
    """Controlador de 8 dígitos de 7 segmentos: 16 bits (dirección, dato) y LOAD (CS) que sube para guardar."""
    def __init__(self, pic, puerto='C', load=0):
        self.p, self.l = puerto, load; self.bytes = []; self.dig = [0] * 8
        self.decode = 0; self.intens = 0; self.limite = 7; self.encendido = False; self.prueba = False
        pic.spi.append(lambda pi, v: self.bytes.append(v)); pic.escuchas.append(self.cambio)
    def cambio(self, pic, p, ant, n):
        if p != self.p or bit(ant, self.l) or not bit(n, self.l) or len(self.bytes) < 2: return
        a, d = self.bytes[-2] & 0x0F, self.bytes[-1]; self.bytes = []
        if 1 <= a <= 8: self.dig[a - 1] = d
        elif a == 9: self.decode = d
        elif a == 10: self.intens = d & 15
        elif a == 11: self.limite = d & 7
        elif a == 12: self.encendido = bool(d & 1)
        elif a == 15: self.prueba = bool(d & 1)
    def texto(self):
        if not self.encendido: return '(apagado)'
        s = ''
        for k in range(7, -1, -1):     # dígito 7 a la izquierda
            if k > self.limite: s += ' '; continue
            v = self.dig[k]
            if self.decode >> k & 1: s += '0123456789-EHLP '[v & 15]
            else: s += Multiplex7.SEGS.get(v & 0x7F, '?') if False else '?'
            if v & 0x80: s += '.'
        return s

## 2. Los programas en ensamblador
Cada archivo es autónomo (incluye sus rutinas de USART, I2C, SPI, LCD, retardos y aritmética): se abre directamente en MPLAB X o se ensambla con gpasm.

In [ ]:
%%writefile ej01_usart_comandos.asm
;===========================================================================
; Ejemplo 1 · USART: terminal de comandos con interrupción de recepción
; PIC16F877A · cristal 4 MHz · 9600 baudios, 8N1
;   RC6/TX y RC7/RX al VIRTUAL TERMINAL (o a un conversor USB-serie)
;   RB0..RB7: 8 LED
;   Comandos: 1..8 invierte el LED · A enciende todos · B apaga todos · ? informa el estado
;===========================================================================
        LIST    P=16F877A, R=DEC
        #include <p16f877a.inc>
        __CONFIG _XT_OSC & _WDT_OFF & _PWRTE_ON & _BOREN_OFF & _LVP_OFF & _CP_OFF

        CBLOCK  0x20
        DATO, LLEGO, MASCARA, AUX, I
        ENDC
        CBLOCK  0x70                    ; RAM común a los cuatro bancos
        W_TEMP, S_TEMP, P_TEMP
        ENDC

        ORG     0x000
        goto    INICIO

        ORG     0x004                   ;---------- rutina de interrupción (ISR)
        movwf   W_TEMP                  ; guarda W, STATUS y PCLATH
        swapf   STATUS,W
        clrf    STATUS
        movwf   S_TEMP
        movf    PCLATH,W
        movwf   P_TEMP
        clrf    PCLATH
        btfss   PIR1,RCIF               ; ¿llegó un carácter?
        goto    FIN_ISR
        movf    RCREG,W                 ; leer RCREG limpia RCIF
        movwf   DATO
        bsf     LLEGO,0                 ; avisa al programa principal
FIN_ISR movf    P_TEMP,W
        movwf   PCLATH
        swapf   S_TEMP,W
        movwf   STATUS
        swapf   W_TEMP,F
        swapf   W_TEMP,W
        retfie

INICIO
        BANKSEL TRISB
        clrf    TRISB
        BANKSEL PORTB
        clrf    PORTB
        clrf    LLEGO
        call    UART_INIT
        BANKSEL PIE1
        bsf     PIE1,RCIE               ; interrupción de recepción
        BANKSEL PORTB
        movlw   b'11000000'             ; GIE y PEIE
        movwf   INTCON
        movlw   HIGH TXT_MENU
        movwf   PTR_H
        movlw   LOW TXT_MENU
        movwf   PTR_L
        call    TX_CADENA

LAZO    btfss   RCSTA,OERR              ; desbordamiento: reinicia el receptor
        goto    LAZO1
        bcf     RCSTA,CREN
        bsf     RCSTA,CREN
LAZO1   btfss   LLEGO,0                 ; espera un comando
        goto    LAZO
        bcf     LLEGO,0
        movf    DATO,W
        xorlw   '?'
        btfsc   STATUS,Z
        goto    INFORMAR
        movf    DATO,W
        andlw   0xDF                    ; minúscula -> mayúscula
        xorlw   'A'
        btfsc   STATUS,Z
        goto    TODOS
        movf    DATO,W
        andlw   0xDF
        xorlw   'B'
        btfsc   STATUS,Z
        goto    NINGUNO
        movlw   '1'                     ; ¿es un dígito '1'..'8'?
        subwf   DATO,W
        btfss   STATUS,C
        goto    LAZO
        movwf   AUX                     ; AUX = DATO - '1' (0..7)
        movlw   8
        subwf   AUX,W
        btfsc   STATUS,C
        goto    LAZO
        movlw   1                       ; MASCARA = 1 << AUX
        movwf   MASCARA
DESPL   movf    AUX,F
        btfsc   STATUS,Z
        goto    INVERTIR
        bcf     STATUS,C
        rlf     MASCARA,F
        decf    AUX,F
        goto    DESPL
INVERTIR
        movf    MASCARA,W
        xorwf   PORTB,F
        goto    INFORMAR
TODOS   movlw   0xFF
        movwf   PORTB
        goto    INFORMAR
NINGUNO clrf    PORTB

INFORMAR                                ; envía "LEDS = b7..b0"
        movlw   HIGH TXT_LEDS
        movwf   PTR_H
        movlw   LOW TXT_LEDS
        movwf   PTR_L
        call    TX_CADENA
        movf    PORTB,W
        movwf   AUX
        movlw   8
        movwf   I
INF1    rlf     AUX,F                   ; el bit 7 pasa a C
        movlw   '0'
        btfsc   STATUS,C
        movlw   '1'
        call    TX_BYTE
        decfsz  I,F
        goto    INF1
        call    TX_CRLF
        goto    LAZO

TXT_MENU DT     "PIC16F877A - USART 9600 8N1",13,10
         DT     "1-8: invierte un LED  A: todos  B: ninguno  ?: estado",13,10,0
TXT_LEDS DT     "LEDS = ",0

;---------------------------------------------------------------------------
; Lectura de tablas DT en la memoria de programa (cadenas terminadas en 0)
;---------------------------------------------------------------------------
        CBLOCK  0x6C
        PTR_H, PTR_L
        ENDC
LEER_TABLA                      ; W = byte de la tabla en PTR_H:PTR_L
        movf    PTR_H,W
        movwf   PCLATH
        movf    PTR_L,W
        movwf   PCL             ; salta a la RETLW, que vuelve a quien llamó a LEER_TABLA

;---------------------------------------------------------------------------
; USART asíncrona a 9600 baudios (cristal de 4 MHz, BRGH = 1, SPBRG = 25)
;   RC6/TX y RC7/RX
;---------------------------------------------------------------------------
        CBLOCK  0x61
        TX_TMP
        ENDC
UART_INIT
        BANKSEL SPBRG
        movlw   25                      ; 4 MHz / (16 x 26) = 9615 baudios (0,16 % de error)
        movwf   SPBRG
        movlw   b'00100100'             ; TXEN = 1, BRGH = 1, asíncrono
        movwf   TXSTA
        BANKSEL RCSTA
        movlw   b'10010000'             ; SPEN = 1, CREN = 1
        movwf   RCSTA
        return
TX_BYTE                                 ; transmite W (espera a que TXREG esté libre)
        btfss   PIR1,TXIF
        goto    TX_BYTE
        movwf   TXREG
        return
TX_CADENA                               ; transmite la cadena PTR_H:PTR_L
        call    LEER_TABLA
        iorlw   0
        btfsc   STATUS,Z
        return
        call    TX_BYTE
        incf    PTR_L,F
        btfsc   STATUS,Z
        incf    PTR_H,F
        goto    TX_CADENA
TX_NUM                                  ; transmite el dígito 0..9 en W
        addlw   '0'
        goto    TX_BYTE
TX_CRLF movlw   13
        call    TX_BYTE
        movlw   10
        goto    TX_BYTE

        END

In [ ]:
%%writefile ej02_usart_telemetria.asm
;===========================================================================
; Ejemplo 2 · USART: telemetría (el PIC envía mediciones al PC cada 250 ms)
; PIC16F877A · cristal 4 MHz · 9600 baudios, 8N1
;   Potenciómetro en AN0 (RA0) · LM35 en AN1 (RA1) · RB0 = LED de actividad
;   Formato de cada línea:  AN0=0512 V=2.50 T=025.3C
;   (fácil de leer y graficar en Python con pyserial: ver graficar_serie.py)
;===========================================================================
        LIST    P=16F877A, R=DEC
        #include <p16f877a.inc>
        __CONFIG _XT_OSC & _WDT_OFF & _PWRTE_ON & _BOREN_OFF & _LVP_OFF & _CP_OFF

        CBLOCK  0x20
        ADC_H, ADC_L
        ENDC

        ORG     0x000
        goto    INICIO
        ORG     0x005
INICIO
        BANKSEL TRISB
        clrf    TRISB
        movlw   0x84                    ; AN0, AN1 y AN3 analógicos, justificado a la derecha
        movwf   ADCON1
        BANKSEL PORTB
        clrf    PORTB
        call    UART_INIT
        movlw   HIGH TXT_HOLA
        movwf   PTR_H
        movlw   LOW TXT_HOLA
        movwf   PTR_L
        call    TX_CADENA

LAZO    ;----- AN0: código del ADC y voltaje
        movlw   0x41                    ; Fosc/8, canal 0, ADC encendido
        call    LEER_ADC
        movlw   HIGH TXT_AN0
        movwf   PTR_H
        movlw   LOW TXT_AN0
        movwf   PTR_L
        call    TX_CADENA
        movf    ADC_H,W
        movwf   NUM_H
        movf    ADC_L,W
        movwf   NUM_L
        call    BCD16
        movf    DIG3,W
        call    TX_NUM
        movf    DIG2,W
        call    TX_NUM
        movf    DIG1,W
        call    TX_NUM
        movf    DIG0,W
        call    TX_NUM
        call    A_MILIVOLTIOS           ; NUM = ADC x 625 / 128 = mV
        call    BCD16
        movlw   HIGH TXT_V
        movwf   PTR_H
        movlw   LOW TXT_V
        movwf   PTR_L
        call    TX_CADENA
        movf    DIG3,W
        call    TX_NUM
        movlw   '.'
        call    TX_BYTE
        movf    DIG2,W
        call    TX_NUM
        movf    DIG1,W
        call    TX_NUM
        ;----- AN1: LM35 (10 mV/°C: los mV son décimas de grado)
        movlw   0x49                    ; canal 1
        call    LEER_ADC
        call    A_MILIVOLTIOS
        call    BCD16
        movlw   HIGH TXT_T
        movwf   PTR_H
        movlw   LOW TXT_T
        movwf   PTR_L
        call    TX_CADENA
        movf    DIG3,W
        call    TX_NUM
        movf    DIG2,W
        call    TX_NUM
        movf    DIG1,W
        call    TX_NUM
        movlw   '.'
        call    TX_BYTE
        movf    DIG0,W
        call    TX_NUM
        movlw   'C'
        call    TX_BYTE
        call    TX_CRLF
        movlw   1                       ; LED de actividad
        xorwf   PORTB,F
        movlw   200
        call    RETARDO_MS
        goto    LAZO

LEER_ADC                                ; W = ADCON0; resultado en ADC_H:ADC_L
        movwf   ADCON0
        call    RETARDO_50US
        bsf     ADCON0,GO
ESPERA  btfsc   ADCON0,GO
        goto    ESPERA
        movf    ADRESH,W
        movwf   ADC_H
        BANKSEL ADRESL
        movf    ADRESL,W
        BANKSEL PORTB
        movwf   ADC_L
        return

A_MILIVOLTIOS                           ; NUM_H:NUM_L = ADC x 625 >> 7
        movf    ADC_H,W
        movwf   NUM_H
        movf    ADC_L,W
        movwf   NUM_L
        movlw   HIGH 625
        movwf   K_H
        movlw   LOW 625
        movwf   K_L
        call    MUL16
        rlf     P0,F
        rlf     P1,F
        rlf     P2,F
        movf    P2,W
        movwf   NUM_H
        movf    P1,W
        movwf   NUM_L
        return

TXT_HOLA DT     "PIC16F877A telemetria: AN0 (pot) y AN1 (LM35)",13,10,0
TXT_AN0  DT     "AN0=",0
TXT_V    DT     " V=",0
TXT_T    DT     " T=",0

;---------------------------------------------------------------------------
; Lectura de tablas DT en la memoria de programa (cadenas terminadas en 0)
;---------------------------------------------------------------------------
        CBLOCK  0x6C
        PTR_H, PTR_L
        ENDC
LEER_TABLA                      ; W = byte de la tabla en PTR_H:PTR_L
        movf    PTR_H,W
        movwf   PCLATH
        movf    PTR_L,W
        movwf   PCL             ; salta a la RETLW, que vuelve a quien llamó a LEER_TABLA

;---------------------------------------------------------------------------
; USART asíncrona a 9600 baudios (cristal de 4 MHz, BRGH = 1, SPBRG = 25)
;   RC6/TX y RC7/RX
;---------------------------------------------------------------------------
        CBLOCK  0x61
        TX_TMP
        ENDC
UART_INIT
        BANKSEL SPBRG
        movlw   25                      ; 4 MHz / (16 x 26) = 9615 baudios (0,16 % de error)
        movwf   SPBRG
        movlw   b'00100100'             ; TXEN = 1, BRGH = 1, asíncrono
        movwf   TXSTA
        BANKSEL RCSTA
        movlw   b'10010000'             ; SPEN = 1, CREN = 1
        movwf   RCSTA
        return
TX_BYTE                                 ; transmite W (espera a que TXREG esté libre)
        btfss   PIR1,TXIF
        goto    TX_BYTE
        movwf   TXREG
        return
TX_CADENA                               ; transmite la cadena PTR_H:PTR_L
        call    LEER_TABLA
        iorlw   0
        btfsc   STATUS,Z
        return
        call    TX_BYTE
        incf    PTR_L,F
        btfsc   STATUS,Z
        incf    PTR_H,F
        goto    TX_CADENA
TX_NUM                                  ; transmite el dígito 0..9 en W
        addlw   '0'
        goto    TX_BYTE
TX_CRLF movlw   13
        call    TX_BYTE
        movlw   10
        goto    TX_BYTE

;---------------------------------------------------------------------------
; Retardos para cristal de 4 MHz (1 ciclo de instrucción = 1 µs)
;---------------------------------------------------------------------------
        CBLOCK  0x68
        R_MS, R_US
        ENDC
RETARDO_MS                      ; espera W milisegundos (W = 1..255)
        movwf   R_MS
RMS1    movlw   249             ; 2 + 248*4 + 3 + 3 = 1000 ciclos por milisegundo
        movwf   R_US
RMS2    nop
        decfsz  R_US,F
        goto    RMS2
        decfsz  R_MS,F
        goto    RMS1
        return
RETARDO_50US                    ; 50 µs contando el call y el return
        movlw   15
        movwf   R_US
R50     decfsz  R_US,F
        goto    R50
        return

;---------------------------------------------------------------------------
; Aritmética de 16 y 24 bits
;---------------------------------------------------------------------------
        CBLOCK  0x50
        NUM_H, NUM_L, K_H, K_L, TMP_L, CNT
        DIG4, DIG3, DIG2, DIG1, DIG0
        P2, P1, P0, X2, X1, X0
        ENDC
BCD16                           ; NUM_H:NUM_L (0..65535) -> DIG4..DIG0 (decenas de mil .. unidades)
        movlw   HIGH 10000
        movwf   K_H
        movlw   LOW 10000
        movwf   K_L
        call    CONTAR
        movwf   DIG4
        movlw   HIGH 1000
        movwf   K_H
        movlw   LOW 1000
        movwf   K_L
        call    CONTAR
        movwf   DIG3
        clrf    K_H
        movlw   100
        movwf   K_L
        call    CONTAR
        movwf   DIG2
        movlw   10
        movwf   K_L
        call    CONTAR
        movwf   DIG1
        movf    NUM_L,W
        movwf   DIG0
        return
CONTAR                          ; W = cuántas veces cabe K en NUM (y se lo resta)
        clrf    CNT
CT1     movf    K_L,W
        subwf   NUM_L,W         ; W = NUM_L - K_L   (C = 0 si hubo préstamo)
        movwf   TMP_L
        movf    K_H,W
        btfss   STATUS,C
        addlw   1               ; el préstamo se suma al byte alto que se resta
        subwf   NUM_H,W
        btfss   STATUS,C
        goto    CT2             ; ya no cabe
        movwf   NUM_H
        movf    TMP_L,W
        movwf   NUM_L
        incf    CNT,F
        goto    CT1
CT2     movf    CNT,W
        return

MUL16                           ; P2:P1:P0 = NUM_H:NUM_L x K_H:K_L  (resultado menor que 2^24)
        clrf    P0
        clrf    P1
        clrf    P2
        clrf    X2
        movf    NUM_H,W
        movwf   X1
        movf    NUM_L,W
        movwf   X0
        movlw   16
        movwf   CNT
ML1     bcf     STATUS,C
        rrf     K_H,F
        rrf     K_L,F           ; bit menos significativo de K en C
        btfss   STATUS,C
        goto    ML2
        movf    X0,W            ; P = P + X (24 bits)
        addwf   P0,F
        movf    X1,W
        btfsc   STATUS,C
        incfsz  X1,W
        addwf   P1,F
        movf    X2,W
        btfsc   STATUS,C
        incfsz  X2,W
        addwf   P2,F
ML2     bcf     STATUS,C        ; X = X * 2
        rlf     X0,F
        rlf     X1,F
        rlf     X2,F
        decfsz  CNT,F
        goto    ML1
        return

        END

In [ ]:
%%writefile ej03_i2c_pcf8574.asm
;===========================================================================
; Ejemplo 3 · I2C: 8 LED en un expansor PCF8574 (dirección 0x20 → 0x40 para escribir)
; PIC16F877A · cristal 4 MHz · I2C a 100 kHz
;   SCL = RC3, SDA = RC4 con pull-ups de 4,7 kΩ · A0, A1, A2 del PCF8574 a tierra
;   LED: +5 V → 330 Ω → LED → P0..P7 (el PCF8574 enciende el LED con un 0)
;   RB0 = cambia el efecto (barrido, contador, alternado) · RC0 = LED de error (el esclavo no responde)
;===========================================================================
        LIST    P=16F877A, R=DEC
        #include <p16f877a.inc>
        __CONFIG _XT_OSC & _WDT_OFF & _PWRTE_ON & _BOREN_OFF & _LVP_OFF & _CP_OFF

        CBLOCK  0x20
        MODO, PATRON, DIR, CUENTA, ANTES
        ENDC
PCF_ESCRIBIR EQU 0x40                   ; 0100 A2 A1 A0 R/W

        ORG     0x000
        goto    INICIO
        ORG     0x005
INICIO
        BANKSEL OPTION_REG
        bcf     OPTION_REG,NOT_RBPU     ; pull-up de RB0
        bcf     TRISC,0                 ; LED de error
        BANKSEL PORTC
        bcf     PORTC,0
        call    I2C_INIT
        clrf    MODO
        clrf    CUENTA
        clrf    DIR
        movlw   1
        movwf   PATRON
        movlw   1
        movwf   ANTES

LAZO    call    ENVIAR
        call    SIGUIENTE
        btfsc   PORTB,0                 ; ¿se presionó RB0? (flanco de bajada)
        goto    SUELTO
        btfss   ANTES,0
        goto    ESPERAR
        clrf    ANTES
        incf    MODO,F                  ; siguiente efecto (0, 1, 2)
        movlw   3
        xorwf   MODO,W
        btfsc   STATUS,Z
        clrf    MODO
        movlw   1
        movwf   PATRON
        goto    ESPERAR
SUELTO  bsf     ANTES,0
ESPERAR movlw   120
        call    RETARDO_MS
        goto    LAZO

ENVIAR                                  ; START · 0x40 · ~PATRON · STOP
        call    I2C_START
        movlw   PCF_ESCRIBIR
        call    I2C_ESCRIBIR
        BANKSEL SSPCON2
        btfsc   SSPCON2,ACKSTAT         ; ACKSTAT = 1: nadie respondió
        goto    SIN_ACK
        BANKSEL PORTC
        bcf     PORTC,0
        comf    PATRON,W                ; el LED enciende con 0
        call    I2C_ESCRIBIR
        goto    I2C_STOP
SIN_ACK BANKSEL PORTC
        bsf     PORTC,0
        goto    I2C_STOP

SIGUIENTE                               ; calcula el próximo patrón según MODO
        movf    MODO,W
        btfsc   STATUS,Z
        goto    BARRIDO
        xorlw   1
        btfsc   STATUS,Z
        goto    CONTADOR
        movlw   0x55                    ; alternado 01010101 / 10101010
        xorwf   PATRON,W
        btfsc   STATUS,Z
        goto    ALT2
        movlw   0x55
        movwf   PATRON
        return
ALT2    movlw   0xAA
        movwf   PATRON
        return
CONTADOR
        incf    CUENTA,F
        movf    CUENTA,W
        movwf   PATRON
        return
BARRIDO                                 ; una luz que va y vuelve
        btfsc   DIR,0
        goto    BAR_DER
        bcf     STATUS,C
        rlf     PATRON,F
        btfsc   PATRON,7
        bsf     DIR,0
        movf    PATRON,F
        btfsc   STATUS,Z
        bsf     PATRON,0
        return
BAR_DER bcf     STATUS,C
        rrf     PATRON,F
        btfsc   PATRON,0
        bcf     DIR,0
        movf    PATRON,F
        btfsc   STATUS,Z
        bsf     PATRON,0
        return

;---------------------------------------------------------------------------
; I2C maestro con el MSSP a 100 kHz (cristal de 4 MHz): SCL = RC3, SDA = RC4
;   Resistencias de pull-up de 4,7 kΩ a +5 V en SCL y SDA
;---------------------------------------------------------------------------
I2C_INIT
        BANKSEL TRISC
        bsf     TRISC,3                 ; SCL y SDA como entradas: el MSSP los maneja
        bsf     TRISC,4
        movlw   9                       ; SSPADD = Fosc / (4 x 100 kHz) - 1
        movwf   SSPADD
        movlw   b'10000000'             ; SMP = 1 (velocidad estándar)
        movwf   SSPSTAT
        clrf    SSPCON2
        BANKSEL SSPCON
        movlw   b'00101000'             ; SSPEN = 1, modo I2C maestro
        movwf   SSPCON
        return
I2C_ESPERA                              ; espera a que termine la operación (SSPIF = 1)
        btfss   PIR1,SSPIF
        goto    I2C_ESPERA
        bcf     PIR1,SSPIF
        return
I2C_START
        BANKSEL SSPCON2
        bsf     SSPCON2,SEN
        BANKSEL PIR1
        goto    I2C_ESPERA
I2C_RESTART
        BANKSEL SSPCON2
        bsf     SSPCON2,RSEN
        BANKSEL PIR1
        goto    I2C_ESPERA
I2C_STOP
        BANKSEL SSPCON2
        bsf     SSPCON2,PEN
        BANKSEL PIR1
        goto    I2C_ESPERA
I2C_ESCRIBIR                            ; envía W; al volver, ACKSTAT (SSPCON2,6) = 0 si el esclavo respondió
        movwf   SSPBUF
        goto    I2C_ESPERA
I2C_LEER_ACK                            ; W = byte leído y responde ACK (habrá más bytes)
        BANKSEL SSPCON2
        bcf     SSPCON2,ACKDT
        goto    I2C_LEER
I2C_LEER_NACK                           ; W = byte leído y responde NACK (último byte)
        BANKSEL SSPCON2
        bsf     SSPCON2,ACKDT
I2C_LEER
        bsf     SSPCON2,RCEN
        BANKSEL PIR1
        call    I2C_ESPERA
        BANKSEL SSPCON2
        bsf     SSPCON2,ACKEN
        BANKSEL PIR1
        call    I2C_ESPERA
        movf    SSPBUF,W
        return

;---------------------------------------------------------------------------
; Retardos para cristal de 4 MHz (1 ciclo de instrucción = 1 µs)
;---------------------------------------------------------------------------
        CBLOCK  0x68
        R_MS, R_US
        ENDC
RETARDO_MS                      ; espera W milisegundos (W = 1..255)
        movwf   R_MS
RMS1    movlw   249             ; 2 + 248*4 + 3 + 3 = 1000 ciclos por milisegundo
        movwf   R_US
RMS2    nop
        decfsz  R_US,F
        goto    RMS2
        decfsz  R_MS,F
        goto    RMS1
        return
RETARDO_50US                    ; 50 µs contando el call y el return
        movlw   15
        movwf   R_US
R50     decfsz  R_US,F
        goto    R50
        return

        END

In [ ]:
%%writefile ej04_i2c_ds1307.asm
;===========================================================================
; Ejemplo 4 · I2C: reloj de tiempo real DS1307 (dirección 0x68 → 0xD0 / 0xD1) en LCD 16x2
; PIC16F877A · cristal 4 MHz · I2C a 100 kHz
;   SCL = RC3, SDA = RC4 con pull-ups de 4,7 kΩ · DS1307 con cristal de 32,768 kHz y batería de 3 V
;   SOUT/SQW del DS1307: onda de 1 Hz (pull-up de 4,7 kΩ y LED) · RB0 = suma un minuto
;   LCD: RS = RD0, E = RD1, D4..D7 = RD4..RD7
;===========================================================================
        LIST    P=16F877A, R=DEC
        #include <p16f877a.inc>
        __CONFIG _XT_OSC & _WDT_OFF & _PWRTE_ON & _BOREN_OFF & _LVP_OFF & _CP_OFF

        CBLOCK  0x20
        SEG, MIN, HORA, DIA, FECHA, MES, ANO, AUX
        ENDC
DS_W    EQU     0xD0
DS_R    EQU     0xD1

        ORG     0x000
        goto    INICIO
        ORG     0x005
INICIO
        BANKSEL OPTION_REG
        bcf     OPTION_REG,NOT_RBPU
        BANKSEL PORTB
        call    LCD_INIT
        call    I2C_INIT
        movlw   0x80
        call    LCD_CMD
        movlw   HIGH TXT_HORA
        movwf   PTR_H
        movlw   LOW TXT_HORA
        movwf   PTR_L
        call    LCD_CADENA
        movlw   0xC0
        call    LCD_CMD
        movlw   HIGH TXT_FECHA
        movwf   PTR_H
        movlw   LOW TXT_FECHA
        movwf   PTR_L
        call    LCD_CADENA
        ;----- si el oscilador está detenido (CH = 1), pone la hora inicial
        call    I2C_START
        movlw   DS_W
        call    I2C_ESCRIBIR
        movlw   0x00                    ; registro 0: segundos
        call    I2C_ESCRIBIR
        call    I2C_RESTART
        movlw   DS_R
        call    I2C_ESCRIBIR
        call    I2C_LEER_NACK
        movwf   AUX
        call    I2C_STOP
        btfsc   AUX,7
        call    PONER_HORA
        ;----- salida SQW de 1 Hz (registro de control 0x07 = 0x10)
        call    I2C_START
        movlw   DS_W
        call    I2C_ESCRIBIR
        movlw   0x07
        call    I2C_ESCRIBIR
        movlw   0x10
        call    I2C_ESCRIBIR
        call    I2C_STOP

LAZO    call    LEER_RELOJ
        call    MOSTRAR
        btfsc   PORTB,0                 ; RB0: suma un minuto
        goto    PAUSA
        call    SUMAR_MINUTO
SOLTAR  btfss   PORTB,0
        goto    SOLTAR
PAUSA   movlw   200
        call    RETARDO_MS
        goto    LAZO

PONER_HORA                              ; 12:00:00, jueves 19/11/26 (en BCD)
        call    I2C_START
        movlw   DS_W
        call    I2C_ESCRIBIR
        movlw   0x00
        call    I2C_ESCRIBIR
        movlw   0x00                    ; segundos (CH = 0: el reloj arranca)
        call    I2C_ESCRIBIR
        movlw   0x00                    ; minutos
        call    I2C_ESCRIBIR
        movlw   0x12                    ; horas (formato de 24 h)
        call    I2C_ESCRIBIR
        movlw   0x05                    ; día de la semana
        call    I2C_ESCRIBIR
        movlw   0x19                    ; fecha
        call    I2C_ESCRIBIR
        movlw   0x11                    ; mes
        call    I2C_ESCRIBIR
        movlw   0x26                    ; año
        call    I2C_ESCRIBIR
        goto    I2C_STOP

LEER_RELOJ                              ; lectura secuencial de los registros 0 a 6
        call    I2C_START
        movlw   DS_W
        call    I2C_ESCRIBIR
        movlw   0x00
        call    I2C_ESCRIBIR
        call    I2C_RESTART
        movlw   DS_R
        call    I2C_ESCRIBIR
        call    I2C_LEER_ACK
        andlw   0x7F
        movwf   SEG
        call    I2C_LEER_ACK
        movwf   MIN
        call    I2C_LEER_ACK
        andlw   0x3F
        movwf   HORA
        call    I2C_LEER_ACK
        movwf   DIA
        call    I2C_LEER_ACK
        movwf   FECHA
        call    I2C_LEER_ACK
        movwf   MES
        call    I2C_LEER_NACK           ; último byte: NACK
        movwf   ANO
        goto    I2C_STOP

SUMAR_MINUTO                            ; MIN = MIN + 1 en BCD (59 → 00) y se escribe
        incf    MIN,F
        movf    MIN,W
        andlw   0x0F
        xorlw   0x0A
        btfss   STATUS,Z
        goto    SM1
        movlw   6                       ; ajuste decimal: x9 + 1 = (x+1)0
        addwf   MIN,F
SM1     movf    MIN,W
        xorlw   0x60
        btfsc   STATUS,Z
        clrf    MIN
        call    I2C_START
        movlw   DS_W
        call    I2C_ESCRIBIR
        movlw   0x01
        call    I2C_ESCRIBIR
        movf    MIN,W
        call    I2C_ESCRIBIR
        goto    I2C_STOP

MOSTRAR movlw   0x87
        call    LCD_CMD
        movf    HORA,W
        call    LCD_BCD
        movlw   ':'
        call    LCD_DATO
        movf    MIN,W
        call    LCD_BCD
        movlw   ':'
        call    LCD_DATO
        movf    SEG,W
        call    LCD_BCD
        movlw   0xC7
        call    LCD_CMD
        movf    FECHA,W
        call    LCD_BCD
        movlw   '/'
        call    LCD_DATO
        movf    MES,W
        call    LCD_BCD
        movlw   '/'
        call    LCD_DATO
        movf    ANO,W
        goto    LCD_BCD

LCD_BCD                                 ; escribe los dos dígitos del byte BCD en W
        movwf   AUX
        swapf   AUX,W
        andlw   0x0F
        call    LCD_NUM
        movf    AUX,W
        andlw   0x0F
        goto    LCD_NUM

TXT_HORA  DT    "Hora:",0
TXT_FECHA DT    "Fecha:",0

;---------------------------------------------------------------------------
; Lectura de tablas DT en la memoria de programa (cadenas terminadas en 0)
;---------------------------------------------------------------------------
        CBLOCK  0x6C
        PTR_H, PTR_L
        ENDC
LEER_TABLA                      ; W = byte de la tabla en PTR_H:PTR_L
        movf    PTR_H,W
        movwf   PCLATH
        movf    PTR_L,W
        movwf   PCL             ; salta a la RETLW, que vuelve a quien llamó a LEER_TABLA

;---------------------------------------------------------------------------
; LCD 16x2 (HD44780 / LM016L) en modo de 4 bits sobre PORTD
;   RS = RD0, E = RD1, D4..D7 = RD4..RD7, RW a tierra
;---------------------------------------------------------------------------
        CBLOCK  0x6A
        LCD_TMP, LCD_DAT
        ENDC
LCD_INIT
        BANKSEL TRISD
        clrf    TRISD
        BANKSEL PORTD
        clrf    PORTD
        movlw   20
        call    RETARDO_MS      ; más de 15 ms después del encendido
        movlw   0x30
        call    LCD_NIB
        movlw   5
        call    RETARDO_MS
        movlw   0x30
        call    LCD_NIB
        call    RETARDO_50US
        call    RETARDO_50US
        movlw   0x30
        call    LCD_NIB
        call    RETARDO_50US
        movlw   0x20            ; pasa a modo de 4 bits
        call    LCD_NIB
        call    RETARDO_50US
        movlw   0x28            ; 4 bits, 2 líneas, caracteres de 5x8
        call    LCD_CMD
        movlw   0x0C            ; display encendido, sin cursor
        call    LCD_CMD
        movlw   0x06            ; la dirección aumenta después de cada carácter
        call    LCD_CMD
        movlw   0x01            ; borrar pantalla
        goto    LCD_CMD

LCD_CMD                         ; envía el comando que está en W
        bcf     PORTD,0         ; RS = 0
        goto    LCD_BYTE
LCD_NUM                         ; escribe el dígito 0..9 que está en W
        addlw   '0'
LCD_DATO                        ; escribe el carácter que está en W
        bsf     PORTD,0         ; RS = 1
LCD_BYTE
        movwf   LCD_DAT
        call    LCD_NIB         ; nibble alto
        swapf   LCD_DAT,W
        call    LCD_NIB         ; nibble bajo
        call    RETARDO_50US
        btfsc   PORTD,0         ; datos: listo
        return
        movlw   3               ; borrar (1) e inicio (2) tardan 1,52 ms
        subwf   LCD_DAT,W
        btfsc   STATUS,C
        return
        movlw   2
        goto    RETARDO_MS

LCD_NIB                         ; pone W<7:4> en D7..D4 y da un pulso en E
        andlw   0xF0
        movwf   LCD_TMP
        movf    PORTD,W
        andlw   0x0F
        iorwf   LCD_TMP,W
        movwf   PORTD
        bsf     PORTD,1         ; E = 1
        nop
        bcf     PORTD,1         ; E = 0: la LCD lee el dato en el flanco de bajada
        return

LCD_CADENA                      ; escribe la cadena (DT "...",0) que empieza en PTR_H:PTR_L
        call    LEER_TABLA
        iorlw   0
        btfsc   STATUS,Z
        return
        call    LCD_DATO
        incf    PTR_L,F
        btfsc   STATUS,Z
        incf    PTR_H,F
        goto    LCD_CADENA

;---------------------------------------------------------------------------
; I2C maestro con el MSSP a 100 kHz (cristal de 4 MHz): SCL = RC3, SDA = RC4
;   Resistencias de pull-up de 4,7 kΩ a +5 V en SCL y SDA
;---------------------------------------------------------------------------
I2C_INIT
        BANKSEL TRISC
        bsf     TRISC,3                 ; SCL y SDA como entradas: el MSSP los maneja
        bsf     TRISC,4
        movlw   9                       ; SSPADD = Fosc / (4 x 100 kHz) - 1
        movwf   SSPADD
        movlw   b'10000000'             ; SMP = 1 (velocidad estándar)
        movwf   SSPSTAT
        clrf    SSPCON2
        BANKSEL SSPCON
        movlw   b'00101000'             ; SSPEN = 1, modo I2C maestro
        movwf   SSPCON
        return
I2C_ESPERA                              ; espera a que termine la operación (SSPIF = 1)
        btfss   PIR1,SSPIF
        goto    I2C_ESPERA
        bcf     PIR1,SSPIF
        return
I2C_START
        BANKSEL SSPCON2
        bsf     SSPCON2,SEN
        BANKSEL PIR1
        goto    I2C_ESPERA
I2C_RESTART
        BANKSEL SSPCON2
        bsf     SSPCON2,RSEN
        BANKSEL PIR1
        goto    I2C_ESPERA
I2C_STOP
        BANKSEL SSPCON2
        bsf     SSPCON2,PEN
        BANKSEL PIR1
        goto    I2C_ESPERA
I2C_ESCRIBIR                            ; envía W; al volver, ACKSTAT (SSPCON2,6) = 0 si el esclavo respondió
        movwf   SSPBUF
        goto    I2C_ESPERA
I2C_LEER_ACK                            ; W = byte leído y responde ACK (habrá más bytes)
        BANKSEL SSPCON2
        bcf     SSPCON2,ACKDT
        goto    I2C_LEER
I2C_LEER_NACK                           ; W = byte leído y responde NACK (último byte)
        BANKSEL SSPCON2
        bsf     SSPCON2,ACKDT
I2C_LEER
        bsf     SSPCON2,RCEN
        BANKSEL PIR1
        call    I2C_ESPERA
        BANKSEL SSPCON2
        bsf     SSPCON2,ACKEN
        BANKSEL PIR1
        call    I2C_ESPERA
        movf    SSPBUF,W
        return

;---------------------------------------------------------------------------
; Retardos para cristal de 4 MHz (1 ciclo de instrucción = 1 µs)
;---------------------------------------------------------------------------
        CBLOCK  0x68
        R_MS, R_US
        ENDC
RETARDO_MS                      ; espera W milisegundos (W = 1..255)
        movwf   R_MS
RMS1    movlw   249             ; 2 + 248*4 + 3 + 3 = 1000 ciclos por milisegundo
        movwf   R_US
RMS2    nop
        decfsz  R_US,F
        goto    RMS2
        decfsz  R_MS,F
        goto    RMS1
        return
RETARDO_50US                    ; 50 µs contando el call y el return
        movlw   15
        movwf   R_US
R50     decfsz  R_US,F
        goto    R50
        return

        END

In [ ]:
%%writefile ej05_spi_74hc595.asm
;===========================================================================
; Ejemplo 5 · SPI: 8 LED con un registro de desplazamiento 74HC595
; PIC16F877A · cristal 4 MHz · SPI maestro a 1 MHz
;   RC5/SDO → DS (14) · RC3/SCK → SH_CP (11) · RC0 → ST_CP (12, latch)
;   OE (13) a tierra · MR (10) a +5 V · Q0..Q7 → 330 Ω → LED → tierra
;   RB0 = cambia de modo: contador binario ↔ barra con el potenciómetro de AN0
;===========================================================================
        LIST    P=16F877A, R=DEC
        #include <p16f877a.inc>
        __CONFIG _XT_OSC & _WDT_OFF & _PWRTE_ON & _BOREN_OFF & _LVP_OFF & _CP_OFF

        CBLOCK  0x20
        MODO, CUENTA, ANTES, DATO
        ENDC

        ORG     0x000
        goto    INICIO
        ORG     0x005
BARRA                                   ; 0..8 LED encendidos
        addwf   PCL,F
        DT      0x00,0x01,0x03,0x07,0x0F,0x1F,0x3F,0x7F,0xFF

INICIO
        BANKSEL OPTION_REG
        bcf     OPTION_REG,NOT_RBPU
        bcf     TRISC,0                 ; RC0 = latch del 74HC595
        movlw   0x0E                    ; AN0 analógico, justificado a la izquierda
        movwf   ADCON1
        BANKSEL PORTC
        bsf     PORTC,0
        movlw   0x41
        movwf   ADCON0
        call    SPI_INIT
        clrf    MODO
        clrf    CUENTA
        movlw   1
        movwf   ANTES

LAZO    btfsc   PORTB,0                 ; flanco de bajada en RB0: cambia de modo
        goto    SUELTO
        btfss   ANTES,0
        goto    CALCULAR
        clrf    ANTES
        movlw   1
        xorwf   MODO,F
        goto    CALCULAR
SUELTO  bsf     ANTES,0
CALCULAR
        btfsc   MODO,0
        goto    MODO_BARRA
        incf    CUENTA,F
        movf    CUENTA,W
        goto    MOSTRAR
MODO_BARRA
        call    RETARDO_50US
        bsf     ADCON0,GO
ESPERA  btfsc   ADCON0,GO
        goto    ESPERA
        movf    ADRESH,W                ; LED encendidos = (ADRESH + 16) / 32  →  0..8
        addlw   16
        btfsc   STATUS,C
        goto    LLENO
        movwf   DATO
        swapf   DATO,W
        andlw   0x0F
        movwf   DATO
        bcf     STATUS,C
        rrf     DATO,W
        call    BARRA
        goto    MOSTRAR
LLENO   movlw   8
        call    BARRA
MOSTRAR call    HC595_ENVIAR
        movlw   150
        call    RETARDO_MS
        goto    LAZO

HC595_ENVIAR                            ; W → Q7..Q0
        bcf     PORTC,0                 ; ST_CP en 0
        call    SPI_TX                  ; 8 bits por SDO con reloj en SCK
        bsf     PORTC,0                 ; flanco de subida: las salidas toman el dato
        return

;---------------------------------------------------------------------------
; SPI maestro con el MSSP: SCK = RC3, SDO = RC5 (SDI = RC4 no se usa)
;   Modo 0: el dato cambia en el flanco de bajada y se lee en el de subida
;---------------------------------------------------------------------------
SPI_INIT
        BANKSEL TRISC
        bcf     TRISC,3                 ; SCK salida
        bcf     TRISC,5                 ; SDO salida
        movlw   b'01000000'             ; CKE = 1
        movwf   SSPSTAT
        BANKSEL SSPCON
        movlw   b'00100000'             ; SSPEN = 1, SPI maestro a Fosc/4 (1 MHz)
        movwf   SSPCON
        return
SPI_TX                                  ; envía W y espera a que termine
        movwf   SSPBUF
SPI1    btfss   PIR1,SSPIF
        goto    SPI1
        bcf     PIR1,SSPIF
        movf    SSPBUF,W                ; leer SSPBUF limpia BF
        return

;---------------------------------------------------------------------------
; Retardos para cristal de 4 MHz (1 ciclo de instrucción = 1 µs)
;---------------------------------------------------------------------------
        CBLOCK  0x68
        R_MS, R_US
        ENDC
RETARDO_MS                      ; espera W milisegundos (W = 1..255)
        movwf   R_MS
RMS1    movlw   249             ; 2 + 248*4 + 3 + 3 = 1000 ciclos por milisegundo
        movwf   R_US
RMS2    nop
        decfsz  R_US,F
        goto    RMS2
        decfsz  R_MS,F
        goto    RMS1
        return
RETARDO_50US                    ; 50 µs contando el call y el return
        movlw   15
        movwf   R_US
R50     decfsz  R_US,F
        goto    R50
        return

        END

In [ ]:
%%writefile ej06_spi_max7219.asm
;===========================================================================
; Ejemplo 6 · SPI: display de 8 dígitos con MAX7219
; PIC16F877A · cristal 4 MHz · SPI maestro a 1 MHz
;   RC5/SDO → DIN (1) · RC3/SCK → CLK (13) · RC0 → LOAD/CS (12)
;   DIG0..DIG7 y SEG A..G, DP del MAX7219 → display 7SEG-MPX8-CC · RSET de 10 kΩ a +5 V
;   Izquierda: segundos desde el encendido (0000-9999) · derecha: lectura del ADC en AN0 (0000-1023)
;===========================================================================
        LIST    P=16F877A, R=DEC
        #include <p16f877a.inc>
        __CONFIG _XT_OSC & _WDT_OFF & _PWRTE_ON & _BOREN_OFF & _LVP_OFF & _CP_OFF

        CBLOCK  0x20
        REG, DATO_MAX, DIEZ, SEG_H, SEG_L
        ENDC

        ORG     0x000
        goto    INICIO
        ORG     0x005
INICIO
        BANKSEL TRISC
        bcf     TRISC,0                 ; RC0 = LOAD
        movlw   0x8E                    ; AN0 analógico, justificado a la derecha (10 bits)
        movwf   ADCON1
        BANKSEL PORTC
        bsf     PORTC,0
        movlw   0x41
        movwf   ADCON0
        call    SPI_INIT
        movlw   0x0F                    ; prueba de display: no
        movwf   REG
        movlw   0x00
        call    MAX_ESCRIBIR
        movlw   0x09                    ; decodificación BCD en los 8 dígitos
        movwf   REG
        movlw   0xFF
        call    MAX_ESCRIBIR
        movlw   0x0A                    ; brillo 8/15
        movwf   REG
        movlw   0x08
        call    MAX_ESCRIBIR
        movlw   0x0B                    ; mostrar los dígitos 0 a 7
        movwf   REG
        movlw   0x07
        call    MAX_ESCRIBIR
        movlw   0x0C                    ; salir del modo de apagado
        movwf   REG
        movlw   0x01
        call    MAX_ESCRIBIR
        clrf    SEG_H
        clrf    SEG_L
        movlw   10
        movwf   DIEZ

LAZO    ;----- segundos en los dígitos 7..4 (izquierda)
        movf    SEG_H,W
        movwf   NUM_H
        movf    SEG_L,W
        movwf   NUM_L
        call    BCD16
        movlw   8
        call    ENVIAR_4
        ;----- ADC en los dígitos 3..0 (derecha)
        call    RETARDO_50US
        bsf     ADCON0,GO
ESPERA  btfsc   ADCON0,GO
        goto    ESPERA
        movf    ADRESH,W
        movwf   NUM_H
        BANKSEL ADRESL
        movf    ADRESL,W
        BANKSEL PORTC
        movwf   NUM_L
        call    BCD16
        movlw   4
        call    ENVIAR_4
        movlw   100                     ; 10 vueltas de 100 ms = 1 s
        call    RETARDO_MS
        decfsz  DIEZ,F
        goto    LAZO
        movlw   10
        movwf   DIEZ
        incf    SEG_L,F
        btfsc   STATUS,Z
        incf    SEG_H,F
        goto    LAZO

ENVIAR_4                                ; W = registro del dígito más significativo (8 o 4); DIG3..DIG0
        movwf   REG
        movf    DIG3,W
        call    MAX_ESCRIBIR
        decf    REG,F
        movf    DIG2,W
        call    MAX_ESCRIBIR
        decf    REG,F
        movf    DIG1,W
        call    MAX_ESCRIBIR
        decf    REG,F
        movlw   0x80                    ; el último dígito de cada grupo lleva punto
        iorwf   DIG0,W
        goto    MAX_ESCRIBIR

MAX_ESCRIBIR                            ; REG ← W (16 bits: dirección y dato; LOAD sube al final)
        movwf   DATO_MAX
        bcf     PORTC,0
        movf    REG,W
        call    SPI_TX
        movf    DATO_MAX,W
        call    SPI_TX
        bsf     PORTC,0                 ; flanco de subida en LOAD: el MAX7219 guarda el dato
        return

;---------------------------------------------------------------------------
; SPI maestro con el MSSP: SCK = RC3, SDO = RC5 (SDI = RC4 no se usa)
;   Modo 0: el dato cambia en el flanco de bajada y se lee en el de subida
;---------------------------------------------------------------------------
SPI_INIT
        BANKSEL TRISC
        bcf     TRISC,3                 ; SCK salida
        bcf     TRISC,5                 ; SDO salida
        movlw   b'01000000'             ; CKE = 1
        movwf   SSPSTAT
        BANKSEL SSPCON
        movlw   b'00100000'             ; SSPEN = 1, SPI maestro a Fosc/4 (1 MHz)
        movwf   SSPCON
        return
SPI_TX                                  ; envía W y espera a que termine
        movwf   SSPBUF
SPI1    btfss   PIR1,SSPIF
        goto    SPI1
        bcf     PIR1,SSPIF
        movf    SSPBUF,W                ; leer SSPBUF limpia BF
        return

;---------------------------------------------------------------------------
; Retardos para cristal de 4 MHz (1 ciclo de instrucción = 1 µs)
;---------------------------------------------------------------------------
        CBLOCK  0x68
        R_MS, R_US
        ENDC
RETARDO_MS                      ; espera W milisegundos (W = 1..255)
        movwf   R_MS
RMS1    movlw   249             ; 2 + 248*4 + 3 + 3 = 1000 ciclos por milisegundo
        movwf   R_US
RMS2    nop
        decfsz  R_US,F
        goto    RMS2
        decfsz  R_MS,F
        goto    RMS1
        return
RETARDO_50US                    ; 50 µs contando el call y el return
        movlw   15
        movwf   R_US
R50     decfsz  R_US,F
        goto    R50
        return

;---------------------------------------------------------------------------
; Aritmética de 16 y 24 bits
;---------------------------------------------------------------------------
        CBLOCK  0x50
        NUM_H, NUM_L, K_H, K_L, TMP_L, CNT
        DIG4, DIG3, DIG2, DIG1, DIG0
        P2, P1, P0, X2, X1, X0
        ENDC
BCD16                           ; NUM_H:NUM_L (0..65535) -> DIG4..DIG0 (decenas de mil .. unidades)
        movlw   HIGH 10000
        movwf   K_H
        movlw   LOW 10000
        movwf   K_L
        call    CONTAR
        movwf   DIG4
        movlw   HIGH 1000
        movwf   K_H
        movlw   LOW 1000
        movwf   K_L
        call    CONTAR
        movwf   DIG3
        clrf    K_H
        movlw   100
        movwf   K_L
        call    CONTAR
        movwf   DIG2
        movlw   10
        movwf   K_L
        call    CONTAR
        movwf   DIG1
        movf    NUM_L,W
        movwf   DIG0
        return
CONTAR                          ; W = cuántas veces cabe K en NUM (y se lo resta)
        clrf    CNT
CT1     movf    K_L,W
        subwf   NUM_L,W         ; W = NUM_L - K_L   (C = 0 si hubo préstamo)
        movwf   TMP_L
        movf    K_H,W
        btfss   STATUS,C
        addlw   1               ; el préstamo se suma al byte alto que se resta
        subwf   NUM_H,W
        btfss   STATUS,C
        goto    CT2             ; ya no cabe
        movwf   NUM_H
        movf    TMP_L,W
        movwf   NUM_L
        incf    CNT,F
        goto    CT1
CT2     movf    CNT,W
        return

MUL16                           ; P2:P1:P0 = NUM_H:NUM_L x K_H:K_L  (resultado menor que 2^24)
        clrf    P0
        clrf    P1
        clrf    P2
        clrf    X2
        movf    NUM_H,W
        movwf   X1
        movf    NUM_L,W
        movwf   X0
        movlw   16
        movwf   CNT
ML1     bcf     STATUS,C
        rrf     K_H,F
        rrf     K_L,F           ; bit menos significativo de K en C
        btfss   STATUS,C
        goto    ML2
        movf    X0,W            ; P = P + X (24 bits)
        addwf   P0,F
        movf    X1,W
        btfsc   STATUS,C
        incfsz  X1,W
        addwf   P1,F
        movf    X2,W
        btfsc   STATUS,C
        incfsz  X2,W
        addwf   P2,F
ML2     bcf     STATUS,C        ; X = X * 2
        rlf     X0,F
        rlf     X1,F
        rlf     X2,F
        decfsz  CNT,F
        goto    ML1
        return

        END

In [ ]:
%%writefile ej07_pwm_dos_canales.asm
;===========================================================================
; Ejemplo 7 · PWM: dos canales de 10 bits (CCP1 y CCP2) a 1 kHz controlados por potenciómetros
; PIC16F877A · cristal 4 MHz
;   Pot 1 en AN0 → PWM en RC2/CCP1 · Pot 2 en AN1 → PWM en RC1/CCP2 · cada salida: 330 Ω + LED (y osciloscopio)
;   Periodo = (PR2 + 1) · 4 · Tosc · preescalador = 250 · 4 · 0,25 µs · 4 = 1 ms
;   LCD: RS = RD0, E = RD1, D4..D7 = RD4..RD7 muestra el ciclo útil de cada canal
;===========================================================================
        LIST    P=16F877A, R=DEC
        #include <p16f877a.inc>
        __CONFIG _XT_OSC & _WDT_OFF & _PWRTE_ON & _BOREN_OFF & _LVP_OFF & _CP_OFF

        CBLOCK  0x20
        ADC_H, ADC_L, AUX
        ENDC

        ORG     0x000
        goto    INICIO
        ORG     0x005
INICIO
        BANKSEL TRISC
        bcf     TRISC,1                 ; RC1/CCP2 y RC2/CCP1 salidas
        bcf     TRISC,2
        movlw   249                     ; PR2 = 249 → 1 kHz con preescalador 4
        movwf   PR2
        movlw   0x84                    ; AN0, AN1 analógicos, justificado a la derecha
        movwf   ADCON1
        BANKSEL PORTC
        clrf    CCPR1L
        clrf    CCPR2L
        movlw   0x0C                    ; CCP1 y CCP2 en modo PWM
        movwf   CCP1CON
        movwf   CCP2CON
        movlw   b'00000101'             ; TMR2 encendido, preescalador 1:4
        movwf   T2CON
        call    LCD_INIT
        movlw   0x80
        call    LCD_CMD
        movlw   HIGH TXT_P1
        movwf   PTR_H
        movlw   LOW TXT_P1
        movwf   PTR_L
        call    LCD_CADENA
        movlw   0xC0
        call    LCD_CMD
        movlw   HIGH TXT_P2
        movwf   PTR_H
        movlw   LOW TXT_P2
        movwf   PTR_L
        call    LCD_CADENA

LAZO    ;----- canal 1
        movlw   0x41
        call    LEER_ADC
        call    LIMITAR                 ; 0..1000 (1000 = 100 % con PR2 = 249)
        movf    AUX,W                   ; 2 bits bajos → CCP1CON<5:4>
        iorlw   0x0C
        movwf   CCP1CON
        movf    ADC_L,W                 ; 8 bits altos → CCPR1L
        movwf   CCPR1L
        movlw   0x86
        call    MOSTRAR_PCT
        ;----- canal 2
        movlw   0x49
        call    LEER_ADC
        call    LIMITAR
        movf    AUX,W
        iorlw   0x0C
        movwf   CCP2CON
        movf    ADC_L,W
        movwf   CCPR2L
        movlw   0xC6
        call    MOSTRAR_PCT
        movlw   50
        call    RETARDO_MS
        goto    LAZO

LEER_ADC                                ; W = ADCON0; resultado de 10 bits en ADC_H:ADC_L
        movwf   ADCON0
        call    RETARDO_50US
        bsf     ADCON0,GO
ESPERA  btfsc   ADCON0,GO
        goto    ESPERA
        movf    ADRESH,W
        movwf   ADC_H
        BANKSEL ADRESL
        movf    ADRESL,W
        BANKSEL PORTC
        movwf   ADC_L
        return

LIMITAR                                 ; ADC (10 bits) → ciclo útil D; deja D>>2 en ADC_L y (D & 3) << 4 en AUX
        movf    ADC_H,W                 ; si ADC > 1000 (0x3E8) usa 1000
        movwf   NUM_H
        movf    ADC_L,W
        movwf   NUM_L
        movlw   LOW 1001
        subwf   ADC_L,W
        movlw   HIGH 1001
        btfss   STATUS,C
        addlw   1
        subwf   ADC_H,W
        btfss   STATUS,C
        goto    LIM1
        movlw   HIGH 1000
        movwf   ADC_H
        movwf   NUM_H
        movlw   LOW 1000
        movwf   ADC_L
        movwf   NUM_L
LIM1    swapf   ADC_L,W                 ; AUX = (D & 3) << 4
        andlw   0x30
        movwf   AUX
        rrf     ADC_H,F                 ; D >> 2
        rrf     ADC_L,F
        rrf     ADC_H,F
        rrf     ADC_L,F
        return

MOSTRAR_PCT                             ; W = posición; escribe D / 10 en % (D ya está en NUM)
        call    LCD_CMD
        clrf    K_H
        movlw   10
        movwf   K_L
        call    CONTAR                  ; NUM / 10 = porcentaje (0..100)
        clrf    NUM_H
        movwf   NUM_L
        call    BCD16
        movf    DIG2,W
        btfsc   STATUS,Z
        movlw   ' ' - '0'
        call    LCD_NUM
        movf    DIG2,W
        iorwf   DIG1,W
        btfsc   STATUS,Z
        goto    SIN_DEC
        movf    DIG1,W
        goto    CON_DEC
SIN_DEC movlw   ' ' - '0'
CON_DEC call    LCD_NUM
        movf    DIG0,W
        call    LCD_NUM
        movlw   '%'
        goto    LCD_DATO

TXT_P1  DT      "PWM1:      1 kHz",0
TXT_P2  DT      "PWM2:      RC1",0

;---------------------------------------------------------------------------
; Lectura de tablas DT en la memoria de programa (cadenas terminadas en 0)
;---------------------------------------------------------------------------
        CBLOCK  0x6C
        PTR_H, PTR_L
        ENDC
LEER_TABLA                      ; W = byte de la tabla en PTR_H:PTR_L
        movf    PTR_H,W
        movwf   PCLATH
        movf    PTR_L,W
        movwf   PCL             ; salta a la RETLW, que vuelve a quien llamó a LEER_TABLA

;---------------------------------------------------------------------------
; LCD 16x2 (HD44780 / LM016L) en modo de 4 bits sobre PORTD
;   RS = RD0, E = RD1, D4..D7 = RD4..RD7, RW a tierra
;---------------------------------------------------------------------------
        CBLOCK  0x6A
        LCD_TMP, LCD_DAT
        ENDC
LCD_INIT
        BANKSEL TRISD
        clrf    TRISD
        BANKSEL PORTD
        clrf    PORTD
        movlw   20
        call    RETARDO_MS      ; más de 15 ms después del encendido
        movlw   0x30
        call    LCD_NIB
        movlw   5
        call    RETARDO_MS
        movlw   0x30
        call    LCD_NIB
        call    RETARDO_50US
        call    RETARDO_50US
        movlw   0x30
        call    LCD_NIB
        call    RETARDO_50US
        movlw   0x20            ; pasa a modo de 4 bits
        call    LCD_NIB
        call    RETARDO_50US
        movlw   0x28            ; 4 bits, 2 líneas, caracteres de 5x8
        call    LCD_CMD
        movlw   0x0C            ; display encendido, sin cursor
        call    LCD_CMD
        movlw   0x06            ; la dirección aumenta después de cada carácter
        call    LCD_CMD
        movlw   0x01            ; borrar pantalla
        goto    LCD_CMD

LCD_CMD                         ; envía el comando que está en W
        bcf     PORTD,0         ; RS = 0
        goto    LCD_BYTE
LCD_NUM                         ; escribe el dígito 0..9 que está en W
        addlw   '0'
LCD_DATO                        ; escribe el carácter que está en W
        bsf     PORTD,0         ; RS = 1
LCD_BYTE
        movwf   LCD_DAT
        call    LCD_NIB         ; nibble alto
        swapf   LCD_DAT,W
        call    LCD_NIB         ; nibble bajo
        call    RETARDO_50US
        btfsc   PORTD,0         ; datos: listo
        return
        movlw   3               ; borrar (1) e inicio (2) tardan 1,52 ms
        subwf   LCD_DAT,W
        btfsc   STATUS,C
        return
        movlw   2
        goto    RETARDO_MS

LCD_NIB                         ; pone W<7:4> en D7..D4 y da un pulso en E
        andlw   0xF0
        movwf   LCD_TMP
        movf    PORTD,W
        andlw   0x0F
        iorwf   LCD_TMP,W
        movwf   PORTD
        bsf     PORTD,1         ; E = 1
        nop
        bcf     PORTD,1         ; E = 0: la LCD lee el dato en el flanco de bajada
        return

LCD_CADENA                      ; escribe la cadena (DT "...",0) que empieza en PTR_H:PTR_L
        call    LEER_TABLA
        iorlw   0
        btfsc   STATUS,Z
        return
        call    LCD_DATO
        incf    PTR_L,F
        btfsc   STATUS,Z
        incf    PTR_H,F
        goto    LCD_CADENA

;---------------------------------------------------------------------------
; Retardos para cristal de 4 MHz (1 ciclo de instrucción = 1 µs)
;---------------------------------------------------------------------------
        CBLOCK  0x68
        R_MS, R_US
        ENDC
RETARDO_MS                      ; espera W milisegundos (W = 1..255)
        movwf   R_MS
RMS1    movlw   249             ; 2 + 248*4 + 3 + 3 = 1000 ciclos por milisegundo
        movwf   R_US
RMS2    nop
        decfsz  R_US,F
        goto    RMS2
        decfsz  R_MS,F
        goto    RMS1
        return
RETARDO_50US                    ; 50 µs contando el call y el return
        movlw   15
        movwf   R_US
R50     decfsz  R_US,F
        goto    R50
        return

;---------------------------------------------------------------------------
; Aritmética de 16 y 24 bits
;---------------------------------------------------------------------------
        CBLOCK  0x50
        NUM_H, NUM_L, K_H, K_L, TMP_L, CNT
        DIG4, DIG3, DIG2, DIG1, DIG0
        P2, P1, P0, X2, X1, X0
        ENDC
BCD16                           ; NUM_H:NUM_L (0..65535) -> DIG4..DIG0 (decenas de mil .. unidades)
        movlw   HIGH 10000
        movwf   K_H
        movlw   LOW 10000
        movwf   K_L
        call    CONTAR
        movwf   DIG4
        movlw   HIGH 1000
        movwf   K_H
        movlw   LOW 1000
        movwf   K_L
        call    CONTAR
        movwf   DIG3
        clrf    K_H
        movlw   100
        movwf   K_L
        call    CONTAR
        movwf   DIG2
        movlw   10
        movwf   K_L
        call    CONTAR
        movwf   DIG1
        movf    NUM_L,W
        movwf   DIG0
        return
CONTAR                          ; W = cuántas veces cabe K en NUM (y se lo resta)
        clrf    CNT
CT1     movf    K_L,W
        subwf   NUM_L,W         ; W = NUM_L - K_L   (C = 0 si hubo préstamo)
        movwf   TMP_L
        movf    K_H,W
        btfss   STATUS,C
        addlw   1               ; el préstamo se suma al byte alto que se resta
        subwf   NUM_H,W
        btfss   STATUS,C
        goto    CT2             ; ya no cabe
        movwf   NUM_H
        movf    TMP_L,W
        movwf   NUM_L
        incf    CNT,F
        goto    CT1
CT2     movf    CNT,W
        return

MUL16                           ; P2:P1:P0 = NUM_H:NUM_L x K_H:K_L  (resultado menor que 2^24)
        clrf    P0
        clrf    P1
        clrf    P2
        clrf    X2
        movf    NUM_H,W
        movwf   X1
        movf    NUM_L,W
        movwf   X0
        movlw   16
        movwf   CNT
ML1     bcf     STATUS,C
        rrf     K_H,F
        rrf     K_L,F           ; bit menos significativo de K en C
        btfss   STATUS,C
        goto    ML2
        movf    X0,W            ; P = P + X (24 bits)
        addwf   P0,F
        movf    X1,W
        btfsc   STATUS,C
        incfsz  X1,W
        addwf   P1,F
        movf    X2,W
        btfsc   STATUS,C
        incfsz  X2,W
        addwf   P2,F
ML2     bcf     STATUS,C        ; X = X * 2
        rlf     X0,F
        rlf     X1,F
        rlf     X2,F
        decfsz  CNT,F
        goto    ML1
        return

        END

In [ ]:
%%writefile ej08_pwm_melodia.asm
;===========================================================================
; Ejemplo 8 · PWM: melodía con un parlante (la frecuencia del PWM es la nota)
; PIC16F877A · cristal 4 MHz
;   RC2/CCP1 → 100 Ω → parlante (SPEAKER en Proteus) · RB0 = tocar · RC0 = LED "sonando"
;   Nota: f = Fosc / (4 · 16 · (PR2 + 1)) → PR2 = 62500 / f − 1, ciclo útil del 50 %
;   Melodía: "Estrellita, ¿dónde estás?" (tradicional, dominio público)
;===========================================================================
        LIST    P=16F877A, R=DEC
        #include <p16f877a.inc>
        __CONFIG _XT_OSC & _WDT_OFF & _PWRTE_ON & _BOREN_OFF & _LVP_OFF & _CP_OFF

        CBLOCK  0x20
        IDX, NOTA, DUR, VECES
        ENDC

        ORG     0x000
        goto    INICIO
        ORG     0x005
PR2_NOTA                                ; Do4 Re4 Mi4 Fa4 Sol4 La4 Si4 Do5
        addwf   PCL,F
        DT      237, 212, 188, 178, 158, 141, 125, 118
MELODIA                                 ; nota (0..7) y duración (en corcheas de 250 ms) alternadas; 0xFF = fin
        addwf   PCL,F
        DT      0,1, 0,1, 4,1, 4,1, 5,1, 5,1, 4,2
        DT      3,1, 3,1, 2,1, 2,1, 1,1, 1,1, 0,2
        DT      4,1, 4,1, 3,1, 3,1, 2,1, 2,1, 1,2
        DT      4,1, 4,1, 3,1, 3,1, 2,1, 2,1, 1,2
        DT      0,1, 0,1, 4,1, 4,1, 5,1, 5,1, 4,2
        DT      3,1, 3,1, 2,1, 2,1, 1,1, 1,1, 0,2
        DT      0xFF

INICIO
        BANKSEL OPTION_REG
        bcf     OPTION_REG,NOT_RBPU
        bcf     TRISC,2                 ; RC2/CCP1 salida
        bcf     TRISC,0
        BANKSEL PORTC
        clrf    PORTC
        clrf    CCPR1L
        movlw   0x0C                    ; CCP1 en PWM
        movwf   CCP1CON
        movlw   b'00000110'             ; TMR2 encendido, preescalador 1:16
        movwf   T2CON

ESPERAR btfsc   PORTB,0                 ; espera el botón
        goto    ESPERAR
        bsf     PORTC,0
        clrf    IDX
TOCAR   movf    IDX,W
        call    MELODIA
        movwf   NOTA
        xorlw   0xFF
        btfsc   STATUS,Z
        goto    FIN
        incf    IDX,F
        movf    IDX,W
        call    MELODIA
        movwf   DUR
        incf    IDX,F
        movf    NOTA,W                  ; frecuencia de la nota
        call    PR2_NOTA
        BANKSEL PR2
        movwf   PR2
        BANKSEL CCPR1L
        movwf   CCPR1L                  ; 50 %: CCPR1L = (PR2 + 1) / 2
        incf    CCPR1L,F
        bcf     STATUS,C
        rrf     CCPR1L,F
SONAR   movlw   230                     ; duración: DUR x 250 ms (230 sonando + 20 de silencio)
        call    RETARDO_MS
        decfsz  DUR,F
        goto    SONAR_MAS
        clrf    CCPR1L                  ; silencio entre notas
        movlw   20
        call    RETARDO_MS
        goto    TOCAR
SONAR_MAS
        movlw   20
        call    RETARDO_MS
        goto    SONAR
FIN     clrf    CCPR1L
        bcf     PORTC,0
        goto    ESPERAR

;---------------------------------------------------------------------------
; Retardos para cristal de 4 MHz (1 ciclo de instrucción = 1 µs)
;---------------------------------------------------------------------------
        CBLOCK  0x68
        R_MS, R_US
        ENDC
RETARDO_MS                      ; espera W milisegundos (W = 1..255)
        movwf   R_MS
RMS1    movlw   249             ; 2 + 248*4 + 3 + 3 = 1000 ciclos por milisegundo
        movwf   R_US
RMS2    nop
        decfsz  R_US,F
        goto    RMS2
        decfsz  R_MS,F
        goto    RMS1
        return
RETARDO_50US                    ; 50 µs contando el call y el return
        movlw   15
        movwf   R_US
R50     decfsz  R_US,F
        goto    R50
        return

        END

In [ ]:
%%writefile ej09_isr_fuentes.asm
;===========================================================================
; Ejemplo 9 · Interrupciones: cuatro fuentes atendidas por una sola ISR
; PIC16F877A · cristal 4 MHz
;   INT (RB0, flanco de bajada): cuenta pulsaciones
;   Cambio en RB4..RB7 (RBIF): indica qué tecla se presionó
;   TMR0 (cada 65,5 ms): inicia una conversión del ADC y hace parpadear el LED de RC0
;   ADC terminado (ADIF): guarda la lectura de AN0
;   LCD: RS = RD0, E = RD1, D4..D7 = RD4..RD7
;   El orden en que la ISR revisa las banderas define la prioridad: INT, RB, ADC, TMR0
;===========================================================================
        LIST    P=16F877A, R=DEC
        #include <p16f877a.inc>
        __CONFIG _XT_OSC & _WDT_OFF & _PWRTE_ON & _BOREN_OFF & _LVP_OFF & _CP_OFF

        CBLOCK  0x20
        PULSOS, TECLA, ADC_H, ADC_L, T0CNT, CAMBIO, AUX
        ENDC
        CBLOCK  0x70
        W_TEMP, S_TEMP, P_TEMP
        ENDC

        ORG     0x000
        goto    INICIO

        ORG     0x004                   ;================= ISR
        movwf   W_TEMP
        swapf   STATUS,W
        clrf    STATUS                  ; banco 0
        movwf   S_TEMP
        movf    PCLATH,W
        movwf   P_TEMP
        clrf    PCLATH
        ;----- 1) INT: pulsador en RB0
        btfss   INTCON,INTE
        goto    ISR_RB
        btfss   INTCON,INTF
        goto    ISR_RB
        bcf     INTCON,INTF
        incf    PULSOS,F
        bsf     CAMBIO,0
        ;----- 2) cambio en RB4..RB7
ISR_RB  btfss   INTCON,RBIF
        goto    ISR_AD
        movf    PORTB,W                 ; leer PORTB termina la condición de cambio
        movwf   AUX
        bcf     INTCON,RBIF
        movlw   4
        btfss   AUX,4
        movwf   TECLA
        movlw   5
        btfss   AUX,5
        movwf   TECLA
        movlw   6
        btfss   AUX,6
        movwf   TECLA
        movlw   7
        btfss   AUX,7
        movwf   TECLA
        bsf     CAMBIO,0
        ;----- 3) fin de conversión del ADC
ISR_AD  btfss   PIR1,ADIF
        goto    ISR_T0
        bcf     PIR1,ADIF
        movf    ADRESH,W
        movwf   ADC_H
        BANKSEL ADRESL
        movf    ADRESL,W
        BANKSEL PIR1
        movwf   ADC_L
        bsf     CAMBIO,0
        ;----- 4) desbordamiento de TMR0
ISR_T0  btfss   INTCON,T0IF
        goto    FIN_ISR
        bcf     INTCON,T0IF
        bsf     ADCON0,GO               ; nueva conversión (la adquisición ya pasó)
        incf    T0CNT,F
        movlw   4                       ; 4 x 65,5 ms ≈ 262 ms
        subwf   T0CNT,W
        btfss   STATUS,C
        goto    FIN_ISR
        clrf    T0CNT
        movlw   1
        xorwf   PORTC,F                 ; LED de latido
FIN_ISR movf    P_TEMP,W
        movwf   PCLATH
        swapf   S_TEMP,W
        movwf   STATUS
        swapf   W_TEMP,F
        swapf   W_TEMP,W
        retfie

INICIO
        BANKSEL TRISC
        bcf     TRISC,0
        movlw   b'00000111'             ; pull-ups, INT en bajada, TMR0 interno con preescalador 1:256
        movwf   OPTION_REG
        movlw   0x8E                    ; AN0 analógico, justificado a la derecha
        movwf   ADCON1
        bsf     PIE1,ADIE               ; interrupción del ADC
        BANKSEL PORTC
        clrf    PORTC
        clrf    PULSOS
        clrf    TECLA
        clrf    ADC_H
        clrf    ADC_L
        clrf    T0CNT
        movlw   0x41
        movwf   ADCON0
        call    LCD_INIT
        movf    PORTB,W                 ; referencia para el cambio en RB4..RB7
        bcf     INTCON,RBIF
        movlw   b'11111000'             ; GIE, PEIE, T0IE, INTE, RBIE
        movwf   INTCON
        movlw   1
        movwf   CAMBIO

LAZO    btfss   CAMBIO,0                ; el programa principal solo muestra
        goto    LAZO
        bcf     CAMBIO,0
        movlw   0x80
        call    LCD_CMD
        movlw   HIGH TXT_L1
        movwf   PTR_H
        movlw   LOW TXT_L1
        movwf   PTR_L
        call    LCD_CADENA
        clrf    NUM_H
        movf    PULSOS,W
        movwf   NUM_L
        call    BCD16
        movf    DIG2,W
        call    LCD_NUM
        movf    DIG1,W
        call    LCD_NUM
        movf    DIG0,W
        call    LCD_NUM
        movlw   HIGH TXT_TEC
        movwf   PTR_H
        movlw   LOW TXT_TEC
        movwf   PTR_L
        call    LCD_CADENA
        movf    TECLA,W
        btfsc   STATUS,Z
        movlw   '-' - '0'
        call    LCD_NUM
        movlw   0xC0
        call    LCD_CMD
        movlw   HIGH TXT_L2
        movwf   PTR_H
        movlw   LOW TXT_L2
        movwf   PTR_L
        call    LCD_CADENA
        bcf     INTCON,GIE              ; copia atómica de los 16 bits del ADC
        movf    ADC_H,W
        movwf   NUM_H
        movf    ADC_L,W
        movwf   NUM_L
        bsf     INTCON,GIE
        call    BCD16
        movf    DIG3,W
        call    LCD_NUM
        movf    DIG2,W
        call    LCD_NUM
        movf    DIG1,W
        call    LCD_NUM
        movf    DIG0,W
        call    LCD_NUM
        goto    LAZO

TXT_L1  DT      "INT=",0
TXT_TEC DT      "  RB=",0
TXT_L2  DT      "ADC=",0

;---------------------------------------------------------------------------
; Lectura de tablas DT en la memoria de programa (cadenas terminadas en 0)
;---------------------------------------------------------------------------
        CBLOCK  0x6C
        PTR_H, PTR_L
        ENDC
LEER_TABLA                      ; W = byte de la tabla en PTR_H:PTR_L
        movf    PTR_H,W
        movwf   PCLATH
        movf    PTR_L,W
        movwf   PCL             ; salta a la RETLW, que vuelve a quien llamó a LEER_TABLA

;---------------------------------------------------------------------------
; LCD 16x2 (HD44780 / LM016L) en modo de 4 bits sobre PORTD
;   RS = RD0, E = RD1, D4..D7 = RD4..RD7, RW a tierra
;---------------------------------------------------------------------------
        CBLOCK  0x6A
        LCD_TMP, LCD_DAT
        ENDC
LCD_INIT
        BANKSEL TRISD
        clrf    TRISD
        BANKSEL PORTD
        clrf    PORTD
        movlw   20
        call    RETARDO_MS      ; más de 15 ms después del encendido
        movlw   0x30
        call    LCD_NIB
        movlw   5
        call    RETARDO_MS
        movlw   0x30
        call    LCD_NIB
        call    RETARDO_50US
        call    RETARDO_50US
        movlw   0x30
        call    LCD_NIB
        call    RETARDO_50US
        movlw   0x20            ; pasa a modo de 4 bits
        call    LCD_NIB
        call    RETARDO_50US
        movlw   0x28            ; 4 bits, 2 líneas, caracteres de 5x8
        call    LCD_CMD
        movlw   0x0C            ; display encendido, sin cursor
        call    LCD_CMD
        movlw   0x06            ; la dirección aumenta después de cada carácter
        call    LCD_CMD
        movlw   0x01            ; borrar pantalla
        goto    LCD_CMD

LCD_CMD                         ; envía el comando que está en W
        bcf     PORTD,0         ; RS = 0
        goto    LCD_BYTE
LCD_NUM                         ; escribe el dígito 0..9 que está en W
        addlw   '0'
LCD_DATO                        ; escribe el carácter que está en W
        bsf     PORTD,0         ; RS = 1
LCD_BYTE
        movwf   LCD_DAT
        call    LCD_NIB         ; nibble alto
        swapf   LCD_DAT,W
        call    LCD_NIB         ; nibble bajo
        call    RETARDO_50US
        btfsc   PORTD,0         ; datos: listo
        return
        movlw   3               ; borrar (1) e inicio (2) tardan 1,52 ms
        subwf   LCD_DAT,W
        btfsc   STATUS,C
        return
        movlw   2
        goto    RETARDO_MS

LCD_NIB                         ; pone W<7:4> en D7..D4 y da un pulso en E
        andlw   0xF0
        movwf   LCD_TMP
        movf    PORTD,W
        andlw   0x0F
        iorwf   LCD_TMP,W
        movwf   PORTD
        bsf     PORTD,1         ; E = 1
        nop
        bcf     PORTD,1         ; E = 0: la LCD lee el dato en el flanco de bajada
        return

LCD_CADENA                      ; escribe la cadena (DT "...",0) que empieza en PTR_H:PTR_L
        call    LEER_TABLA
        iorlw   0
        btfsc   STATUS,Z
        return
        call    LCD_DATO
        incf    PTR_L,F
        btfsc   STATUS,Z
        incf    PTR_H,F
        goto    LCD_CADENA

;---------------------------------------------------------------------------
; Retardos para cristal de 4 MHz (1 ciclo de instrucción = 1 µs)
;---------------------------------------------------------------------------
        CBLOCK  0x68
        R_MS, R_US
        ENDC
RETARDO_MS                      ; espera W milisegundos (W = 1..255)
        movwf   R_MS
RMS1    movlw   249             ; 2 + 248*4 + 3 + 3 = 1000 ciclos por milisegundo
        movwf   R_US
RMS2    nop
        decfsz  R_US,F
        goto    RMS2
        decfsz  R_MS,F
        goto    RMS1
        return
RETARDO_50US                    ; 50 µs contando el call y el return
        movlw   15
        movwf   R_US
R50     decfsz  R_US,F
        goto    R50
        return

;---------------------------------------------------------------------------
; Aritmética de 16 y 24 bits
;---------------------------------------------------------------------------
        CBLOCK  0x50
        NUM_H, NUM_L, K_H, K_L, TMP_L, CNT
        DIG4, DIG3, DIG2, DIG1, DIG0
        P2, P1, P0, X2, X1, X0
        ENDC
BCD16                           ; NUM_H:NUM_L (0..65535) -> DIG4..DIG0 (decenas de mil .. unidades)
        movlw   HIGH 10000
        movwf   K_H
        movlw   LOW 10000
        movwf   K_L
        call    CONTAR
        movwf   DIG4
        movlw   HIGH 1000
        movwf   K_H
        movlw   LOW 1000
        movwf   K_L
        call    CONTAR
        movwf   DIG3
        clrf    K_H
        movlw   100
        movwf   K_L
        call    CONTAR
        movwf   DIG2
        movlw   10
        movwf   K_L
        call    CONTAR
        movwf   DIG1
        movf    NUM_L,W
        movwf   DIG0
        return
CONTAR                          ; W = cuántas veces cabe K en NUM (y se lo resta)
        clrf    CNT
CT1     movf    K_L,W
        subwf   NUM_L,W         ; W = NUM_L - K_L   (C = 0 si hubo préstamo)
        movwf   TMP_L
        movf    K_H,W
        btfss   STATUS,C
        addlw   1               ; el préstamo se suma al byte alto que se resta
        subwf   NUM_H,W
        btfss   STATUS,C
        goto    CT2             ; ya no cabe
        movwf   NUM_H
        movf    TMP_L,W
        movwf   NUM_L
        incf    CNT,F
        goto    CT1
CT2     movf    CNT,W
        return

MUL16                           ; P2:P1:P0 = NUM_H:NUM_L x K_H:K_L  (resultado menor que 2^24)
        clrf    P0
        clrf    P1
        clrf    P2
        clrf    X2
        movf    NUM_H,W
        movwf   X1
        movf    NUM_L,W
        movwf   X0
        movlw   16
        movwf   CNT
ML1     bcf     STATUS,C
        rrf     K_H,F
        rrf     K_L,F           ; bit menos significativo de K en C
        btfss   STATUS,C
        goto    ML2
        movf    X0,W            ; P = P + X (24 bits)
        addwf   P0,F
        movf    X1,W
        btfsc   STATUS,C
        incfsz  X1,W
        addwf   P1,F
        movf    X2,W
        btfsc   STATUS,C
        incfsz  X2,W
        addwf   P2,F
ML2     bcf     STATUS,C        ; X = X * 2
        rlf     X0,F
        rlf     X1,F
        rlf     X2,F
        decfsz  CNT,F
        goto    ML1
        return

        END

In [ ]:
%%writefile ej10_isr_reloj.asm
;===========================================================================
; Ejemplo 10 · Interrupciones: reloj HH:MM:SS con TMR1 y ajuste por INT
; PIC16F877A · cristal 4 MHz
;   TMR1 con preescalador 1:8 y precarga 65536 - 62500 → interrumpe cada 0,5 s
;   RB0/INT: suma un minuto · RB1 (leído en el lazo): suma una hora · RC0: LED de segundos
;   LCD: RS = RD0, E = RD1, D4..D7 = RD4..RD7
;===========================================================================
        LIST    P=16F877A, R=DEC
        #include <p16f877a.inc>
        __CONFIG _XT_OSC & _WDT_OFF & _PWRTE_ON & _BOREN_OFF & _LVP_OFF & _CP_OFF

        CBLOCK  0x20
        HH, MM, SS, MEDIO, NUEVO, DEC, ANTES
        ENDC
        CBLOCK  0x70
        W_TEMP, S_TEMP, P_TEMP
        ENDC
PRECARGA EQU    65536 - 62500           ; 62500 x 8 µs = 0,5 s

        ORG     0x000
        goto    INICIO

        ORG     0x004                   ;================= ISR
        movwf   W_TEMP
        swapf   STATUS,W
        clrf    STATUS
        movwf   S_TEMP
        movf    PCLATH,W
        movwf   P_TEMP
        clrf    PCLATH
        btfss   PIR1,TMR1IF
        goto    ISR_INT
        bcf     PIR1,TMR1IF
        bcf     T1CON,TMR1ON            ; recarga el temporizador
        movlw   HIGH PRECARGA
        movwf   TMR1H
        movlw   LOW PRECARGA
        movwf   TMR1L
        bsf     T1CON,TMR1ON
        movlw   1
        xorwf   PORTC,F                 ; LED que cambia cada 0,5 s
        incf    MEDIO,F
        btfss   MEDIO,1                 ; dos medios segundos = 1 s
        goto    ISR_INT
        clrf    MEDIO
        call    SUMAR_SEG
        bsf     NUEVO,0
ISR_INT btfss   INTCON,INTF
        goto    FIN_ISR
        bcf     INTCON,INTF
        call    SUMAR_MIN
        bsf     NUEVO,0
FIN_ISR movf    P_TEMP,W
        movwf   PCLATH
        swapf   S_TEMP,W
        movwf   STATUS
        swapf   W_TEMP,F
        swapf   W_TEMP,W
        retfie

SUMAR_SEG                               ; SS, MM, HH en binario
        incf    SS,F
        movlw   60
        subwf   SS,W
        btfss   STATUS,C
        return
        clrf    SS
SUMAR_MIN
        incf    MM,F
        movlw   60
        subwf   MM,W
        btfss   STATUS,C
        return
        clrf    MM
SUMAR_HORA
        incf    HH,F
        movlw   24
        subwf   HH,W
        btfsc   STATUS,C
        clrf    HH
        return

INICIO
        BANKSEL TRISC
        bcf     TRISC,0
        movlw   b'00000000'             ; pull-ups, INT en flanco de bajada
        movwf   OPTION_REG
        bsf     PIE1,TMR1IE
        BANKSEL PORTC
        clrf    PORTC
        movlw   12
        movwf   HH
        clrf    MM
        clrf    SS
        clrf    MEDIO
        movlw   1
        movwf   ANTES
        call    LCD_INIT
        movlw   0x80
        call    LCD_CMD
        movlw   HIGH TXT_T
        movwf   PTR_H
        movlw   LOW TXT_T
        movwf   PTR_L
        call    LCD_CADENA
        movlw   HIGH PRECARGA
        movwf   TMR1H
        movlw   LOW PRECARGA
        movwf   TMR1L
        movlw   b'00110001'             ; TMR1: 1:8, reloj interno, encendido
        movwf   T1CON
        movlw   b'11010000'             ; GIE, PEIE, INTE
        movwf   INTCON
        bsf     NUEVO,0

LAZO    btfsc   PORTB,1                 ; RB1: suma una hora (por sondeo, con flanco)
        goto    RB1_SUELTO
        btfss   ANTES,0
        goto    MOSTRAR
        clrf    ANTES
        bcf     INTCON,GIE
        call    SUMAR_HORA
        bsf     INTCON,GIE
        bsf     NUEVO,0
        goto    MOSTRAR
RB1_SUELTO
        bsf     ANTES,0
MOSTRAR btfss   NUEVO,0
        goto    LAZO
        bcf     NUEVO,0
        movlw   0xC4
        call    LCD_CMD
        movf    HH,W
        call    LCD_2DIG
        movlw   ':'
        call    LCD_DATO
        movf    MM,W
        call    LCD_2DIG
        movlw   ':'
        call    LCD_DATO
        movf    SS,W
        call    LCD_2DIG
        goto    LAZO

LCD_2DIG                                ; W (0..99) en dos dígitos
        clrf    DEC
        movwf   NUM_L
D2      movlw   10
        subwf   NUM_L,W
        btfss   STATUS,C
        goto    D3
        movwf   NUM_L
        incf    DEC,F
        goto    D2
D3      movf    DEC,W
        call    LCD_NUM
        movf    NUM_L,W
        goto    LCD_NUM

TXT_T   DT      "Reloj con TMR1",0

;---------------------------------------------------------------------------
; Lectura de tablas DT en la memoria de programa (cadenas terminadas en 0)
;---------------------------------------------------------------------------
        CBLOCK  0x6C
        PTR_H, PTR_L
        ENDC
LEER_TABLA                      ; W = byte de la tabla en PTR_H:PTR_L
        movf    PTR_H,W
        movwf   PCLATH
        movf    PTR_L,W
        movwf   PCL             ; salta a la RETLW, que vuelve a quien llamó a LEER_TABLA

;---------------------------------------------------------------------------
; LCD 16x2 (HD44780 / LM016L) en modo de 4 bits sobre PORTD
;   RS = RD0, E = RD1, D4..D7 = RD4..RD7, RW a tierra
;---------------------------------------------------------------------------
        CBLOCK  0x6A
        LCD_TMP, LCD_DAT
        ENDC
LCD_INIT
        BANKSEL TRISD
        clrf    TRISD
        BANKSEL PORTD
        clrf    PORTD
        movlw   20
        call    RETARDO_MS      ; más de 15 ms después del encendido
        movlw   0x30
        call    LCD_NIB
        movlw   5
        call    RETARDO_MS
        movlw   0x30
        call    LCD_NIB
        call    RETARDO_50US
        call    RETARDO_50US
        movlw   0x30
        call    LCD_NIB
        call    RETARDO_50US
        movlw   0x20            ; pasa a modo de 4 bits
        call    LCD_NIB
        call    RETARDO_50US
        movlw   0x28            ; 4 bits, 2 líneas, caracteres de 5x8
        call    LCD_CMD
        movlw   0x0C            ; display encendido, sin cursor
        call    LCD_CMD
        movlw   0x06            ; la dirección aumenta después de cada carácter
        call    LCD_CMD
        movlw   0x01            ; borrar pantalla
        goto    LCD_CMD

LCD_CMD                         ; envía el comando que está en W
        bcf     PORTD,0         ; RS = 0
        goto    LCD_BYTE
LCD_NUM                         ; escribe el dígito 0..9 que está en W
        addlw   '0'
LCD_DATO                        ; escribe el carácter que está en W
        bsf     PORTD,0         ; RS = 1
LCD_BYTE
        movwf   LCD_DAT
        call    LCD_NIB         ; nibble alto
        swapf   LCD_DAT,W
        call    LCD_NIB         ; nibble bajo
        call    RETARDO_50US
        btfsc   PORTD,0         ; datos: listo
        return
        movlw   3               ; borrar (1) e inicio (2) tardan 1,52 ms
        subwf   LCD_DAT,W
        btfsc   STATUS,C
        return
        movlw   2
        goto    RETARDO_MS

LCD_NIB                         ; pone W<7:4> en D7..D4 y da un pulso en E
        andlw   0xF0
        movwf   LCD_TMP
        movf    PORTD,W
        andlw   0x0F
        iorwf   LCD_TMP,W
        movwf   PORTD
        bsf     PORTD,1         ; E = 1
        nop
        bcf     PORTD,1         ; E = 0: la LCD lee el dato en el flanco de bajada
        return

LCD_CADENA                      ; escribe la cadena (DT "...",0) que empieza en PTR_H:PTR_L
        call    LEER_TABLA
        iorlw   0
        btfsc   STATUS,Z
        return
        call    LCD_DATO
        incf    PTR_L,F
        btfsc   STATUS,Z
        incf    PTR_H,F
        goto    LCD_CADENA

;---------------------------------------------------------------------------
; Retardos para cristal de 4 MHz (1 ciclo de instrucción = 1 µs)
;---------------------------------------------------------------------------
        CBLOCK  0x68
        R_MS, R_US
        ENDC
RETARDO_MS                      ; espera W milisegundos (W = 1..255)
        movwf   R_MS
RMS1    movlw   249             ; 2 + 248*4 + 3 + 3 = 1000 ciclos por milisegundo
        movwf   R_US
RMS2    nop
        decfsz  R_US,F
        goto    RMS2
        decfsz  R_MS,F
        goto    RMS1
        return
RETARDO_50US                    ; 50 µs contando el call y el return
        movlw   15
        movwf   R_US
R50     decfsz  R_US,F
        goto    R50
        return

;---------------------------------------------------------------------------
; Aritmética de 16 y 24 bits
;---------------------------------------------------------------------------
        CBLOCK  0x50
        NUM_H, NUM_L, K_H, K_L, TMP_L, CNT
        DIG4, DIG3, DIG2, DIG1, DIG0
        P2, P1, P0, X2, X1, X0
        ENDC
BCD16                           ; NUM_H:NUM_L (0..65535) -> DIG4..DIG0 (decenas de mil .. unidades)
        movlw   HIGH 10000
        movwf   K_H
        movlw   LOW 10000
        movwf   K_L
        call    CONTAR
        movwf   DIG4
        movlw   HIGH 1000
        movwf   K_H
        movlw   LOW 1000
        movwf   K_L
        call    CONTAR
        movwf   DIG3
        clrf    K_H
        movlw   100
        movwf   K_L
        call    CONTAR
        movwf   DIG2
        movlw   10
        movwf   K_L
        call    CONTAR
        movwf   DIG1
        movf    NUM_L,W
        movwf   DIG0
        return
CONTAR                          ; W = cuántas veces cabe K en NUM (y se lo resta)
        clrf    CNT
CT1     movf    K_L,W
        subwf   NUM_L,W         ; W = NUM_L - K_L   (C = 0 si hubo préstamo)
        movwf   TMP_L
        movf    K_H,W
        btfss   STATUS,C
        addlw   1               ; el préstamo se suma al byte alto que se resta
        subwf   NUM_H,W
        btfss   STATUS,C
        goto    CT2             ; ya no cabe
        movwf   NUM_H
        movf    TMP_L,W
        movwf   NUM_L
        incf    CNT,F
        goto    CT1
CT2     movf    CNT,W
        return

MUL16                           ; P2:P1:P0 = NUM_H:NUM_L x K_H:K_L  (resultado menor que 2^24)
        clrf    P0
        clrf    P1
        clrf    P2
        clrf    X2
        movf    NUM_H,W
        movwf   X1
        movf    NUM_L,W
        movwf   X0
        movlw   16
        movwf   CNT
ML1     bcf     STATUS,C
        rrf     K_H,F
        rrf     K_L,F           ; bit menos significativo de K en C
        btfss   STATUS,C
        goto    ML2
        movf    X0,W            ; P = P + X (24 bits)
        addwf   P0,F
        movf    X1,W
        btfsc   STATUS,C
        incfsz  X1,W
        addwf   P1,F
        movf    X2,W
        btfsc   STATUS,C
        incfsz  X2,W
        addwf   P2,F
ML2     bcf     STATUS,C        ; X = X * 2
        rlf     X0,F
        rlf     X1,F
        rlf     X2,F
        decfsz  CNT,F
        goto    ML1
        return

        END

In [ ]:
%%writefile ej11_multitarea_planificador.asm
;===========================================================================
; Ejemplo 11 · Multitarea cooperativa: planificador con un tick de 10 ms (TMR0 + ISR)
; PIC16F877A · cristal 4 MHz
;   La ISR de TMR0 cuenta milisegundos; cada 10 ms suma un "tick". El lazo principal reparte
;   los ticks entre 5 tareas cortas que nunca esperan (no hay retardos bloqueantes):
;     LED1 (RC0) cada 250 ms · LED2 (RC1) cada 400 ms · botón RB0 cada 20 ms (cuenta pulsaciones)
;     ADC (AN0) → PWM en RC2/CCP1 cada 50 ms · LCD cada 500 ms
;   LCD: RS = RD0, E = RD1, D4..D7 = RD4..RD7
;===========================================================================
        LIST    P=16F877A, R=DEC
        #include <p16f877a.inc>
        __CONFIG _XT_OSC & _WDT_OFF & _PWRTE_ON & _BOREN_OFF & _LVP_OFF & _CP_OFF

        CBLOCK  0x20
        TICKS, MS10, C_LED1, C_LED2, C_BOT, C_ADC, C_LCD
        PULSOS, ANTES, POT
        ENDC
        CBLOCK  0x70
        W_TEMP, S_TEMP
        ENDC

        ORG     0x000
        goto    INICIO

        ORG     0x004                   ;================= ISR: base de tiempo
        movwf   W_TEMP
        swapf   STATUS,W
        clrf    STATUS
        movwf   S_TEMP
        bcf     INTCON,T0IF
        movlw   6                       ; 250 cuentas x 4 µs = 1 ms
        addwf   TMR0,F
        incf    MS10,F
        movlw   10
        subwf   MS10,W
        btfss   STATUS,C
        goto    FIN_ISR
        clrf    MS10
        incf    TICKS,F                 ; un tick más de 10 ms pendiente
FIN_ISR swapf   S_TEMP,W
        movwf   STATUS
        swapf   W_TEMP,F
        swapf   W_TEMP,W
        retfie

INICIO
        BANKSEL TRISC
        movlw   b'11111000'             ; RC0, RC1, RC2 salidas
        movwf   TRISC
        movlw   b'00000001'             ; pull-ups, TMR0 interno, preescalador 1:4
        movwf   OPTION_REG
        movlw   0x0E                    ; AN0 analógico, justificado a la izquierda
        movwf   ADCON1
        movlw   255
        movwf   PR2
        BANKSEL PORTC
        clrf    PORTC
        movlw   0x0C
        movwf   CCP1CON
        movlw   0x04
        movwf   T2CON
        movlw   0x41
        movwf   ADCON0
        call    LCD_INIT
        movlw   0x80
        call    LCD_CMD
        movlw   HIGH TXT_L1
        movwf   PTR_H
        movlw   LOW TXT_L1
        movwf   PTR_L
        call    LCD_CADENA
        movlw   0xC0
        call    LCD_CMD
        movlw   HIGH TXT_L2
        movwf   PTR_H
        movlw   LOW TXT_L2
        movwf   PTR_L
        call    LCD_CADENA
        clrf    TICKS
        clrf    MS10
        clrf    PULSOS
        clrf    POT
        movlw   1
        movwf   ANTES
        movwf   C_LED1
        movwf   C_LED2
        movwf   C_BOT
        movwf   C_ADC
        movwf   C_LCD
        bsf     ADCON0,GO
        movlw   b'10100000'             ; GIE, T0IE
        movwf   INTCON

;================= planificador
LAZO    movf    TICKS,W                 ; espera un tick de 10 ms
        btfsc   STATUS,Z
        goto    LAZO
        decf    TICKS,F
        decfsz  C_LED1,F
        goto    T2
        movlw   25                      ; 25 x 10 ms = 250 ms
        movwf   C_LED1
        call    TAREA_LED1
T2      decfsz  C_LED2,F
        goto    T3
        movlw   40                      ; 400 ms
        movwf   C_LED2
        call    TAREA_LED2
T3      decfsz  C_BOT,F
        goto    T4
        movlw   2                       ; 20 ms
        movwf   C_BOT
        call    TAREA_BOTON
T4      decfsz  C_ADC,F
        goto    T5
        movlw   5                       ; 50 ms
        movwf   C_ADC
        call    TAREA_ADC
T5      decfsz  C_LCD,F
        goto    LAZO
        movlw   50                      ; 500 ms
        movwf   C_LCD
        call    TAREA_LCD
        goto    LAZO

;================= tareas (cortas y sin esperas)
TAREA_LED1
        movlw   b'00000001'
        xorwf   PORTC,F
        return
TAREA_LED2
        movlw   b'00000010'
        xorwf   PORTC,F
        return
TAREA_BOTON                             ; cuenta flancos de bajada de RB0 (el muestreo cada 20 ms filtra rebotes)
        btfsc   PORTB,0
        goto    SUELTO
        btfss   ANTES,0
        return
        clrf    ANTES
        incf    PULSOS,F
        return
SUELTO  bsf     ANTES,0
        return
TAREA_ADC                               ; lee el resultado anterior e inicia la siguiente conversión
        btfsc   ADCON0,GO
        return
        movf    ADRESH,W
        movwf   POT
        movwf   CCPR1L                  ; brillo del LED de RC2
        bsf     ADCON0,GO
        return
TAREA_LCD
        movlw   0x88
        call    LCD_CMD
        clrf    NUM_H
        movf    PULSOS,W
        movwf   NUM_L
        call    BCD16
        call    TRES_DIG
        movlw   0xC8
        call    LCD_CMD
        clrf    NUM_H
        movf    POT,W
        movwf   NUM_L
        call    BCD16
        goto    TRES_DIG
TRES_DIG
        movf    DIG2,W
        call    LCD_NUM
        movf    DIG1,W
        call    LCD_NUM
        movf    DIG0,W
        goto    LCD_NUM

TXT_L1  DT      "Pulsos:",0
TXT_L2  DT      "PWM   :",0

;---------------------------------------------------------------------------
; Lectura de tablas DT en la memoria de programa (cadenas terminadas en 0)
;---------------------------------------------------------------------------
        CBLOCK  0x6C
        PTR_H, PTR_L
        ENDC
LEER_TABLA                      ; W = byte de la tabla en PTR_H:PTR_L
        movf    PTR_H,W
        movwf   PCLATH
        movf    PTR_L,W
        movwf   PCL             ; salta a la RETLW, que vuelve a quien llamó a LEER_TABLA

;---------------------------------------------------------------------------
; LCD 16x2 (HD44780 / LM016L) en modo de 4 bits sobre PORTD
;   RS = RD0, E = RD1, D4..D7 = RD4..RD7, RW a tierra
;---------------------------------------------------------------------------
        CBLOCK  0x6A
        LCD_TMP, LCD_DAT
        ENDC
LCD_INIT
        BANKSEL TRISD
        clrf    TRISD
        BANKSEL PORTD
        clrf    PORTD
        movlw   20
        call    RETARDO_MS      ; más de 15 ms después del encendido
        movlw   0x30
        call    LCD_NIB
        movlw   5
        call    RETARDO_MS
        movlw   0x30
        call    LCD_NIB
        call    RETARDO_50US
        call    RETARDO_50US
        movlw   0x30
        call    LCD_NIB
        call    RETARDO_50US
        movlw   0x20            ; pasa a modo de 4 bits
        call    LCD_NIB
        call    RETARDO_50US
        movlw   0x28            ; 4 bits, 2 líneas, caracteres de 5x8
        call    LCD_CMD
        movlw   0x0C            ; display encendido, sin cursor
        call    LCD_CMD
        movlw   0x06            ; la dirección aumenta después de cada carácter
        call    LCD_CMD
        movlw   0x01            ; borrar pantalla
        goto    LCD_CMD

LCD_CMD                         ; envía el comando que está en W
        bcf     PORTD,0         ; RS = 0
        goto    LCD_BYTE
LCD_NUM                         ; escribe el dígito 0..9 que está en W
        addlw   '0'
LCD_DATO                        ; escribe el carácter que está en W
        bsf     PORTD,0         ; RS = 1
LCD_BYTE
        movwf   LCD_DAT
        call    LCD_NIB         ; nibble alto
        swapf   LCD_DAT,W
        call    LCD_NIB         ; nibble bajo
        call    RETARDO_50US
        btfsc   PORTD,0         ; datos: listo
        return
        movlw   3               ; borrar (1) e inicio (2) tardan 1,52 ms
        subwf   LCD_DAT,W
        btfsc   STATUS,C
        return
        movlw   2
        goto    RETARDO_MS

LCD_NIB                         ; pone W<7:4> en D7..D4 y da un pulso en E
        andlw   0xF0
        movwf   LCD_TMP
        movf    PORTD,W
        andlw   0x0F
        iorwf   LCD_TMP,W
        movwf   PORTD
        bsf     PORTD,1         ; E = 1
        nop
        bcf     PORTD,1         ; E = 0: la LCD lee el dato en el flanco de bajada
        return

LCD_CADENA                      ; escribe la cadena (DT "...",0) que empieza en PTR_H:PTR_L
        call    LEER_TABLA
        iorlw   0
        btfsc   STATUS,Z
        return
        call    LCD_DATO
        incf    PTR_L,F
        btfsc   STATUS,Z
        incf    PTR_H,F
        goto    LCD_CADENA

;---------------------------------------------------------------------------
; Retardos para cristal de 4 MHz (1 ciclo de instrucción = 1 µs)
;---------------------------------------------------------------------------
        CBLOCK  0x68
        R_MS, R_US
        ENDC
RETARDO_MS                      ; espera W milisegundos (W = 1..255)
        movwf   R_MS
RMS1    movlw   249             ; 2 + 248*4 + 3 + 3 = 1000 ciclos por milisegundo
        movwf   R_US
RMS2    nop
        decfsz  R_US,F
        goto    RMS2
        decfsz  R_MS,F
        goto    RMS1
        return
RETARDO_50US                    ; 50 µs contando el call y el return
        movlw   15
        movwf   R_US
R50     decfsz  R_US,F
        goto    R50
        return

;---------------------------------------------------------------------------
; Aritmética de 16 y 24 bits
;---------------------------------------------------------------------------
        CBLOCK  0x50
        NUM_H, NUM_L, K_H, K_L, TMP_L, CNT
        DIG4, DIG3, DIG2, DIG1, DIG0
        P2, P1, P0, X2, X1, X0
        ENDC
BCD16                           ; NUM_H:NUM_L (0..65535) -> DIG4..DIG0 (decenas de mil .. unidades)
        movlw   HIGH 10000
        movwf   K_H
        movlw   LOW 10000
        movwf   K_L
        call    CONTAR
        movwf   DIG4
        movlw   HIGH 1000
        movwf   K_H
        movlw   LOW 1000
        movwf   K_L
        call    CONTAR
        movwf   DIG3
        clrf    K_H
        movlw   100
        movwf   K_L
        call    CONTAR
        movwf   DIG2
        movlw   10
        movwf   K_L
        call    CONTAR
        movwf   DIG1
        movf    NUM_L,W
        movwf   DIG0
        return
CONTAR                          ; W = cuántas veces cabe K en NUM (y se lo resta)
        clrf    CNT
CT1     movf    K_L,W
        subwf   NUM_L,W         ; W = NUM_L - K_L   (C = 0 si hubo préstamo)
        movwf   TMP_L
        movf    K_H,W
        btfss   STATUS,C
        addlw   1               ; el préstamo se suma al byte alto que se resta
        subwf   NUM_H,W
        btfss   STATUS,C
        goto    CT2             ; ya no cabe
        movwf   NUM_H
        movf    TMP_L,W
        movwf   NUM_L
        incf    CNT,F
        goto    CT1
CT2     movf    CNT,W
        return

MUL16                           ; P2:P1:P0 = NUM_H:NUM_L x K_H:K_L  (resultado menor que 2^24)
        clrf    P0
        clrf    P1
        clrf    P2
        clrf    X2
        movf    NUM_H,W
        movwf   X1
        movf    NUM_L,W
        movwf   X0
        movlw   16
        movwf   CNT
ML1     bcf     STATUS,C
        rrf     K_H,F
        rrf     K_L,F           ; bit menos significativo de K en C
        btfss   STATUS,C
        goto    ML2
        movf    X0,W            ; P = P + X (24 bits)
        addwf   P0,F
        movf    X1,W
        btfsc   STATUS,C
        incfsz  X1,W
        addwf   P1,F
        movf    X2,W
        btfsc   STATUS,C
        incfsz  X2,W
        addwf   P2,F
ML2     bcf     STATUS,C        ; X = X * 2
        rlf     X0,F
        rlf     X1,F
        rlf     X2,F
        decfsz  CNT,F
        goto    ML1
        return

        END

In [ ]:
%%writefile ej12_multitarea_semaforo.asm
;===========================================================================
; Ejemplo 12 · Multitarea: semáforo + latido + comandos por USART al mismo tiempo
; PIC16F877A · cristal 4 MHz · 9600 baudios
;   Tick de 10 ms con TMR0 · la recepción por USART llega por interrupción
;   Tareas: SEMAFORO (máquina de estados, cada 100 ms) · LATIDO (RB7, cada 300 ms) · SERIE (cada 10 ms)
;   RB0 rojo · RB1 amarillo · RB2 verde (vehicular) · RB3 rojo · RB4 verde (peatonal) · RB5 "espere"
;   Comandos: P = solicitud peatonal · S = estado actual. El PIC informa cada cambio de estado.
;===========================================================================
        LIST    P=16F877A, R=DEC
        #include <p16f877a.inc>
        __CONFIG _XT_OSC & _WDT_OFF & _PWRTE_ON & _BOREN_OFF & _LVP_OFF & _CP_OFF

        CBLOCK  0x20
        TICKS, MS10, C_SEM, C_LAT, ESTADO, TIEMPO, SOLICITUD, RXDATO, RXFLAG, AUX
        ENDC
        CBLOCK  0x70
        W_TEMP, S_TEMP, P_TEMP
        ENDC
VERDE    EQU    0
AMARILLO EQU    1
ROJO     EQU    2
DESPEJE  EQU    3

        ORG     0x000
        goto    INICIO

        ORG     0x004                   ;================= ISR
        movwf   W_TEMP
        swapf   STATUS,W
        clrf    STATUS
        movwf   S_TEMP
        movf    PCLATH,W
        movwf   P_TEMP
        clrf    PCLATH
        btfss   INTCON,T0IF
        goto    ISR_RX
        bcf     INTCON,T0IF
        movlw   6
        addwf   TMR0,F
        incf    MS10,F
        movlw   10
        subwf   MS10,W
        btfss   STATUS,C
        goto    ISR_RX
        clrf    MS10
        incf    TICKS,F
ISR_RX  btfss   PIR1,RCIF
        goto    FIN_ISR
        movf    RCREG,W
        movwf   RXDATO
        bsf     RXFLAG,0
FIN_ISR movf    P_TEMP,W
        movwf   PCLATH
        swapf   S_TEMP,W
        movwf   STATUS
        swapf   W_TEMP,F
        swapf   W_TEMP,W
        retfie

LUCES                                   ; LED de cada estado (sin RB5 ni RB7)
        clrf    PCLATH                  ; la tabla está en la página 0x000-0x0FF
        addwf   PCL,F
        DT      b'00001100', b'00001010', b'00010001', b'00001001'

INICIO
        BANKSEL TRISB
        clrf    TRISB
        movlw   b'00000001'             ; TMR0 interno, preescalador 1:4
        movwf   OPTION_REG
        BANKSEL PORTB
        clrf    PORTB
        call    UART_INIT
        BANKSEL PIE1
        bsf     PIE1,RCIE
        BANKSEL PORTB
        clrf    TICKS
        clrf    MS10
        clrf    SOLICITUD
        clrf    RXFLAG
        clrf    TIEMPO
        movlw   VERDE
        movwf   ESTADO
        movlw   10
        movwf   C_SEM
        movlw   30
        movwf   C_LAT
        movlw   HIGH TXT_HOLA
        movwf   PTR_H
        movlw   LOW TXT_HOLA
        movwf   PTR_L
        call    TX_CADENA
        call    PONER_LUCES
        call    ANUNCIAR
        movlw   b'11100000'             ; GIE, PEIE, T0IE
        movwf   INTCON

;================= planificador
LAZO    movf    TICKS,W
        btfsc   STATUS,Z
        goto    LAZO
        decf    TICKS,F
        call    TAREA_SERIE             ; cada 10 ms
        decfsz  C_SEM,F
        goto    L2
        movlw   10                      ; cada 100 ms
        movwf   C_SEM
        call    TAREA_SEMAFORO
L2      decfsz  C_LAT,F
        goto    LAZO
        movlw   30                      ; cada 300 ms
        movwf   C_LAT
        movlw   b'10000000'
        xorwf   PORTB,F
        goto    LAZO

;================= tarea: comandos por la USART
TAREA_SERIE
        btfss   RXFLAG,0
        return
        bcf     RXFLAG,0
        movf    RXDATO,W
        andlw   0xDF
        movwf   AUX
        xorlw   'P'
        btfsc   STATUS,Z
        goto    PEDIR
        movf    AUX,W
        xorlw   'S'
        btfsc   STATUS,Z
        goto    ANUNCIAR
        return
PEDIR   bsf     SOLICITUD,0
        bsf     PORTB,5                 ; LED "espere"
        movlw   HIGH TXT_PED
        movwf   PTR_H
        movlw   LOW TXT_PED
        movwf   PTR_L
        goto    TX_CADENA

;================= tarea: máquina de estados del semáforo (cada 100 ms)
TAREA_SEMAFORO
        incf    TIEMPO,F
        movf    ESTADO,W
        xorlw   VERDE
        btfsc   STATUS,Z
        goto    S_VERDE
        movf    ESTADO,W
        xorlw   AMARILLO
        btfsc   STATUS,Z
        goto    S_AMARILLO
        movf    ESTADO,W
        xorlw   ROJO
        btfsc   STATUS,Z
        goto    S_ROJO
S_DESPEJE                               ; todo en rojo 1 s
        movlw   10
        call    ES_HORA
        btfss   STATUS,C
        return
        movlw   VERDE
        goto    CAMBIAR
S_VERDE                                 ; 8 s, o 3 s si hay solicitud peatonal
        movlw   80
        call    ES_HORA
        btfsc   STATUS,C
        goto    A_AMARILLO
        btfss   SOLICITUD,0
        return
        movlw   30
        call    ES_HORA
        btfss   STATUS,C
        return
A_AMARILLO
        movlw   AMARILLO
        goto    CAMBIAR
S_AMARILLO                              ; 2 s
        movlw   20
        call    ES_HORA
        btfss   STATUS,C
        return
        clrf    SOLICITUD
        bcf     PORTB,5
        movlw   ROJO
        goto    CAMBIAR
S_ROJO                                  ; paso peatonal 5 s
        movlw   50
        call    ES_HORA
        btfss   STATUS,C
        return
        movlw   DESPEJE
CAMBIAR movwf   ESTADO
        clrf    TIEMPO
        call    PONER_LUCES
        goto    ANUNCIAR

ES_HORA                                   ; C = 1 si TIEMPO >= W
        subwf   TIEMPO,W
        return

PONER_LUCES                             ; conserva RB5 (espere) y RB7 (latido)
        movf    PORTB,W
        andlw   b'10100000'
        movwf   AUX
        movf    ESTADO,W
        call    LUCES
        iorwf   AUX,W
        movwf   PORTB
        return

ANUNCIAR                                ; "Semaforo: <estado>"
        movlw   HIGH TXT_SEM
        movwf   PTR_H
        movlw   LOW TXT_SEM
        movwf   PTR_L
        call    TX_CADENA
        movf    ESTADO,W
        xorlw   VERDE
        btfsc   STATUS,Z
        goto    AN_V
        movf    ESTADO,W
        xorlw   AMARILLO
        btfsc   STATUS,Z
        goto    AN_A
        movf    ESTADO,W
        xorlw   ROJO
        btfsc   STATUS,Z
        goto    AN_R
        movlw   HIGH TXT_DESP
        movwf   PTR_H
        movlw   LOW TXT_DESP
        goto    AN_TX
AN_V    movlw   HIGH TXT_VERDE
        movwf   PTR_H
        movlw   LOW TXT_VERDE
        goto    AN_TX
AN_A    movlw   HIGH TXT_AMAR
        movwf   PTR_H
        movlw   LOW TXT_AMAR
        goto    AN_TX
AN_R    movlw   HIGH TXT_ROJO
        movwf   PTR_H
        movlw   LOW TXT_ROJO
AN_TX   movwf   PTR_L
        call    TX_CADENA
        goto    TX_CRLF

TXT_HOLA  DT    "PIC16F877A multitarea: P = solicitud peatonal, S = estado",13,10,0
TXT_SEM   DT    "Semaforo: ",0
TXT_VERDE DT    "VERDE",0
TXT_AMAR  DT    "AMARILLO",0
TXT_ROJO  DT    "ROJO (pasa el peaton)",0
TXT_DESP  DT    "TODO EN ROJO",0
TXT_PED   DT    "Solicitud peatonal registrada",13,10,0

;---------------------------------------------------------------------------
; Lectura de tablas DT en la memoria de programa (cadenas terminadas en 0)
;---------------------------------------------------------------------------
        CBLOCK  0x6C
        PTR_H, PTR_L
        ENDC
LEER_TABLA                      ; W = byte de la tabla en PTR_H:PTR_L
        movf    PTR_H,W
        movwf   PCLATH
        movf    PTR_L,W
        movwf   PCL             ; salta a la RETLW, que vuelve a quien llamó a LEER_TABLA

;---------------------------------------------------------------------------
; USART asíncrona a 9600 baudios (cristal de 4 MHz, BRGH = 1, SPBRG = 25)
;   RC6/TX y RC7/RX
;---------------------------------------------------------------------------
        CBLOCK  0x61
        TX_TMP
        ENDC
UART_INIT
        BANKSEL SPBRG
        movlw   25                      ; 4 MHz / (16 x 26) = 9615 baudios (0,16 % de error)
        movwf   SPBRG
        movlw   b'00100100'             ; TXEN = 1, BRGH = 1, asíncrono
        movwf   TXSTA
        BANKSEL RCSTA
        movlw   b'10010000'             ; SPEN = 1, CREN = 1
        movwf   RCSTA
        return
TX_BYTE                                 ; transmite W (espera a que TXREG esté libre)
        btfss   PIR1,TXIF
        goto    TX_BYTE
        movwf   TXREG
        return
TX_CADENA                               ; transmite la cadena PTR_H:PTR_L
        call    LEER_TABLA
        iorlw   0
        btfsc   STATUS,Z
        return
        call    TX_BYTE
        incf    PTR_L,F
        btfsc   STATUS,Z
        incf    PTR_H,F
        goto    TX_CADENA
TX_NUM                                  ; transmite el dígito 0..9 en W
        addlw   '0'
        goto    TX_BYTE
TX_CRLF movlw   13
        call    TX_BYTE
        movlw   10
        goto    TX_BYTE

        END

### Las mismas aplicaciones en C (MPLAB X + XC8)
Se compilan en MPLAB X con XC8 (proyecto para PIC16F877A, cristal de 4 MHz). Aquí solo se guardan para descargarlas.

In [ ]:
%%writefile ej01_usart_comandos.c
/* Ejemplo 1 · USART: terminal de comandos con interrupción de recepción (MPLAB X + XC8)
   LEDs en RB0..RB7 · 1..8 invierte un LED · A todos · B ninguno · ? estado */
#include <xc.h>
#include <stdint.h>
#pragma config FOSC = XT, WDTE = OFF, PWRTE = ON, BOREN = OFF, LVP = OFF, CPD = OFF, WRT = OFF, CP = OFF
#define _XTAL_FREQ 4000000UL
/* ---------- USART 9600 baudios (4 MHz, BRGH = 1, SPBRG = 25) ---------- */
void uart_init(void) { SPBRG = 25; TXSTA = 0x24; RCSTA = 0x90; }
void uart_tx(char c) { while (!TXIF); TXREG = c; }
void uart_print(const char *s) { while (*s) uart_tx(*s++); }

volatile char dato;
volatile uint8_t llego = 0;

void __interrupt() isr(void) {
    if (RCIF) { dato = RCREG; llego = 1; }   /* leer RCREG limpia RCIF */
}

void informar(void) {
    uart_print("LEDS = ");
    for (int8_t b = 7; b >= 0; b--) uart_tx((PORTB >> b) & 1 ? '1' : '0');
    uart_print("\r\n");
}

void main(void) {
    TRISB = 0x00; PORTB = 0x00;
    uart_init();
    RCIE = 1; PEIE = 1; GIE = 1;
    uart_print("PIC16F877A - USART 9600 8N1\r\n");
    uart_print("1-8: invierte un LED  A: todos  B: ninguno  ?: estado\r\n");
    while (1) {
        if (OERR) { CREN = 0; CREN = 1; }
        if (!llego) continue;
        llego = 0;
        char c = dato;
        if (c == '?') { informar(); continue; }
        if ((c & 0xDF) == 'A') { PORTB = 0xFF; informar(); continue; }
        if ((c & 0xDF) == 'B') { PORTB = 0x00; informar(); continue; }
        if (c >= '1' && c <= '8') { PORTB ^= (uint8_t)(1 << (c - '1')); informar(); }
    }
}

In [ ]:
%%writefile ej02_usart_telemetria.c
/* Ejemplo 2 · USART: telemetría cada 250 ms (MPLAB X + XC8)
   Pot en AN0, LM35 en AN1. Línea: AN0=0512 V=2.50 T=025.3C */
#include <xc.h>
#include <stdint.h>
#pragma config FOSC = XT, WDTE = OFF, PWRTE = ON, BOREN = OFF, LVP = OFF, CPD = OFF, WRT = OFF, CP = OFF
#define _XTAL_FREQ 4000000UL
/* ---------- USART 9600 baudios (4 MHz, BRGH = 1, SPBRG = 25) ---------- */
void uart_init(void) { SPBRG = 25; TXSTA = 0x24; RCSTA = 0x90; }
void uart_tx(char c) { while (!TXIF); TXREG = c; }
void uart_print(const char *s) { while (*s) uart_tx(*s++); }

uint16_t leer_adc(uint8_t canal) {
    ADCON0 = (uint8_t)(0x41 | (canal << 3));
    __delay_us(50);
    GO_nDONE = 1; while (GO_nDONE);
    return (uint16_t)((ADRESH << 8) | ADRESL);
}

void uart_num(uint16_t n, uint8_t digitos) {           /* n con 'digitos' cifras */
    char b[5];
    for (int8_t i = (int8_t)digitos - 1; i >= 0; i--) { b[i] = '0' + n % 10; n /= 10; }
    for (uint8_t i = 0; i < digitos; i++) uart_tx(b[i]);
}

void main(void) {
    TRISB = 0x00; PORTB = 0;
    ADCON1 = 0x84;                                       /* AN0, AN1, AN3; justificado a la derecha */
    uart_init();
    uart_print("PIC16F877A telemetria: AN0 (pot) y AN1 (LM35)\r\n");
    while (1) {
        uint16_t a0 = leer_adc(0);
        uint16_t mv = (uint16_t)(((uint32_t)a0 * 625) >> 7);
        uart_print("AN0="); uart_num(a0, 4);
        uart_print(" V="); uart_num(mv / 1000, 1); uart_tx('.'); uart_num((mv / 10) % 100, 2);
        uint16_t t = (uint16_t)(((uint32_t)leer_adc(1) * 625) >> 7);   /* décimas de °C */
        uart_print(" T="); uart_num(t / 10, 3); uart_tx('.'); uart_num(t % 10, 1);
        uart_print("C\r\n");
        PORTB ^= 1;
        __delay_ms(200);
    }
}

In [ ]:
%%writefile ej03_i2c_pcf8574.c
/* Ejemplo 3 · I2C: 8 LED en un PCF8574 (dirección 0x20) (MPLAB X + XC8)
   LED de +5 V a cada salida (se encienden con 0) · RB0 cambia el efecto · RC0 = error de bus */
#include <xc.h>
#include <stdint.h>
#pragma config FOSC = XT, WDTE = OFF, PWRTE = ON, BOREN = OFF, LVP = OFF, CPD = OFF, WRT = OFF, CP = OFF
#define _XTAL_FREQ 4000000UL
/* ---------- I2C maestro a 100 kHz (4 MHz): SCL = RC3, SDA = RC4, pull-ups de 4,7 kΩ ---------- */
void i2c_espera(void) { while (!SSPIF); SSPIF = 0; }
void i2c_init(void) { TRISC3 = 1; TRISC4 = 1; SSPADD = 9; SSPSTAT = 0x80; SSPCON2 = 0; SSPCON = 0x28; }
void i2c_start(void) { SEN = 1; i2c_espera(); }
void i2c_restart(void) { RSEN = 1; i2c_espera(); }
void i2c_stop(void) { PEN = 1; i2c_espera(); }
uint8_t i2c_escribir(uint8_t b) { SSPBUF = b; i2c_espera(); return !ACKSTAT; }   /* 1 = el esclavo respondió */
uint8_t i2c_leer(uint8_t ack) {
    RCEN = 1; i2c_espera();
    ACKDT = ack ? 0 : 1; ACKEN = 1; i2c_espera();
    return SSPBUF;
}

#define PCF8574_W 0x40

void enviar(uint8_t patron) {
    i2c_start();
    if (i2c_escribir(PCF8574_W)) { RC0 = 0; i2c_escribir((uint8_t)~patron); }
    else RC0 = 1;                                        /* nadie respondió (NACK) */
    i2c_stop();
}

void main(void) {
    uint8_t modo = 0, patron = 1, dir = 0, cuenta = 0, antes = 1;
    OPTION_REGbits.nRBPU = 0;
    TRISC0 = 0; RC0 = 0;
    i2c_init();
    while (1) {
        enviar(patron);
        switch (modo) {
            case 0:                                      /* barrido */
                patron = dir ? patron >> 1 : (uint8_t)(patron << 1);
                if (patron & 0x80) dir = 1;
                if (patron & 0x01) dir = 0;
                if (!patron) patron = 1;
                break;
            case 1: patron = ++cuenta; break;            /* contador binario */
            default: patron = (patron == 0x55) ? 0xAA : 0x55;
        }
        if (!RB0 && antes) { modo = (modo + 1) % 3; patron = 1; }
        antes = RB0;
        __delay_ms(120);
    }
}

In [ ]:
%%writefile ej04_i2c_ds1307.c
/* Ejemplo 4 · I2C: reloj DS1307 (dirección 0x68) en LCD 16x2 (MPLAB X + XC8)
   SQW a 1 Hz · RB0 suma un minuto */
#include <xc.h>
#include <stdint.h>
#pragma config FOSC = XT, WDTE = OFF, PWRTE = ON, BOREN = OFF, LVP = OFF, CPD = OFF, WRT = OFF, CP = OFF
#define _XTAL_FREQ 4000000UL
/* ---------- LCD 16x2 (HD44780) en modo de 4 bits: RS = RD0, E = RD1, D4..D7 = RD4..RD7, RW a tierra ---------- */
#define LCD_RS PORTDbits.RD0
#define LCD_E  PORTDbits.RD1

void lcd_nibble(uint8_t n) {            /* pone n<7:4> en D7..D4 y da un pulso en E */
    PORTD = (uint8_t)((PORTD & 0x0F) | (n & 0xF0));
    LCD_E = 1; __delay_us(1); LCD_E = 0;
}
void lcd_cmd(uint8_t c) {
    LCD_RS = 0; lcd_nibble(c); lcd_nibble((uint8_t)(c << 4));
    if (c < 3) __delay_ms(2); else __delay_us(50);
}
void lcd_char(char d) {
    LCD_RS = 1; lcd_nibble((uint8_t)d); lcd_nibble((uint8_t)(d << 4)); __delay_us(50);
}
void lcd_init(void) {
    TRISD = 0x00; PORTD = 0x00;
    __delay_ms(20);
    lcd_nibble(0x30); __delay_ms(5);
    lcd_nibble(0x30); __delay_us(150);
    lcd_nibble(0x30); __delay_us(50);
    lcd_nibble(0x20); __delay_us(50);     /* modo de 4 bits */
    lcd_cmd(0x28);                        /* 2 líneas, 5x8 */
    lcd_cmd(0x0C);                        /* display encendido, sin cursor */
    lcd_cmd(0x06);                        /* incremento automático */
    lcd_cmd(0x01);                        /* borrar */
}
void lcd_goto(uint8_t fila, uint8_t col) { lcd_cmd((uint8_t)((fila ? 0xC0 : 0x80) + col)); }
void lcd_print(const char *s) { while (*s) lcd_char(*s++); }
/* ---------- I2C maestro a 100 kHz (4 MHz): SCL = RC3, SDA = RC4, pull-ups de 4,7 kΩ ---------- */
void i2c_espera(void) { while (!SSPIF); SSPIF = 0; }
void i2c_init(void) { TRISC3 = 1; TRISC4 = 1; SSPADD = 9; SSPSTAT = 0x80; SSPCON2 = 0; SSPCON = 0x28; }
void i2c_start(void) { SEN = 1; i2c_espera(); }
void i2c_restart(void) { RSEN = 1; i2c_espera(); }
void i2c_stop(void) { PEN = 1; i2c_espera(); }
uint8_t i2c_escribir(uint8_t b) { SSPBUF = b; i2c_espera(); return !ACKSTAT; }   /* 1 = el esclavo respondió */
uint8_t i2c_leer(uint8_t ack) {
    RCEN = 1; i2c_espera();
    ACKDT = ack ? 0 : 1; ACKEN = 1; i2c_espera();
    return SSPBUF;
}

#define DS_W 0xD0
#define DS_R 0xD1
uint8_t reg[7];

void ds_escribir(uint8_t dir, const uint8_t *d, uint8_t n) {
    i2c_start(); i2c_escribir(DS_W); i2c_escribir(dir);
    while (n--) i2c_escribir(*d++);
    i2c_stop();
}
void ds_leer(uint8_t dir, uint8_t *d, uint8_t n) {
    i2c_start(); i2c_escribir(DS_W); i2c_escribir(dir);
    i2c_restart(); i2c_escribir(DS_R);
    while (n--) *d++ = i2c_leer(n != 0);                 /* NACK en el último */
    i2c_stop();
}
void lcd_bcd(uint8_t b) { lcd_char('0' + (b >> 4)); lcd_char('0' + (b & 0x0F)); }

void main(void) {
    const uint8_t inicial[7] = {0x00, 0x00, 0x12, 0x05, 0x19, 0x11, 0x26};   /* 12:00:00 19/11/26 */
    const uint8_t sqw = 0x10;
    OPTION_REGbits.nRBPU = 0;
    lcd_init(); i2c_init();
    lcd_goto(0, 0); lcd_print("Hora:");
    lcd_goto(1, 0); lcd_print("Fecha:");
    ds_leer(0, reg, 1);
    if (reg[0] & 0x80) ds_escribir(0, inicial, 7);       /* CH = 1: reloj detenido */
    ds_escribir(7, &sqw, 1);                            /* SQW a 1 Hz */
    while (1) {
        ds_leer(0, reg, 7);
        lcd_goto(0, 7); lcd_bcd(reg[2] & 0x3F); lcd_char(':'); lcd_bcd(reg[1]); lcd_char(':'); lcd_bcd(reg[0] & 0x7F);
        lcd_goto(1, 7); lcd_bcd(reg[4]); lcd_char('/'); lcd_bcd(reg[5]); lcd_char('/'); lcd_bcd(reg[6]);
        if (!RB0) {                                     /* suma un minuto (BCD) */
            uint8_t m = reg[1] + 1;
            if ((m & 0x0F) == 0x0A) m += 6;
            if (m == 0x60) m = 0;
            ds_escribir(1, &m, 1);
            while (!RB0);
        }
        __delay_ms(200);
    }
}

In [ ]:
%%writefile ej05_spi_74hc595.c
/* Ejemplo 5 · SPI: 8 LED con un 74HC595 (MPLAB X + XC8)
   SDO → DS, SCK → SH_CP, RC0 → ST_CP · RB0 alterna contador / barra con el pot de AN0 */
#include <xc.h>
#include <stdint.h>
#pragma config FOSC = XT, WDTE = OFF, PWRTE = ON, BOREN = OFF, LVP = OFF, CPD = OFF, WRT = OFF, CP = OFF
#define _XTAL_FREQ 4000000UL
/* ---------- SPI maestro: SCK = RC3, SDO = RC5, modo 0 a Fosc/4 ---------- */
void spi_init(void) { TRISC3 = 0; TRISC5 = 0; SSPSTAT = 0x40; SSPCON = 0x20; }
uint8_t spi_tx(uint8_t b) { SSPBUF = b; while (!SSPIF); SSPIF = 0; return SSPBUF; }

#define LATCH RC0

void hc595(uint8_t v) { LATCH = 0; spi_tx(v); LATCH = 1; }   /* flanco de subida en ST_CP */

void main(void) {
    uint8_t modo = 0, cuenta = 0, antes = 1;
    OPTION_REGbits.nRBPU = 0;
    TRISC0 = 0; LATCH = 1;
    ADCON1 = 0x0E; ADCON0 = 0x41;
    spi_init();
    while (1) {
        if (!RB0 && antes) modo ^= 1;
        antes = RB0;
        if (modo == 0) hc595(++cuenta);
        else {
            __delay_us(50); GO_nDONE = 1; while (GO_nDONE);
            uint8_t n = (uint8_t)(((uint16_t)ADRESH + 16) >> 5);   /* 0..8 LED */
            hc595((uint8_t)((1u << n) - 1));
        }
        __delay_ms(150);
    }
}

In [ ]:
%%writefile ej06_spi_max7219.c
/* Ejemplo 6 · SPI: display de 8 dígitos con MAX7219 (MPLAB X + XC8)
   SDO → DIN, SCK → CLK, RC0 → LOAD · izquierda: segundos · derecha: ADC de AN0 */
#include <xc.h>
#include <stdint.h>
#pragma config FOSC = XT, WDTE = OFF, PWRTE = ON, BOREN = OFF, LVP = OFF, CPD = OFF, WRT = OFF, CP = OFF
#define _XTAL_FREQ 4000000UL
/* ---------- SPI maestro: SCK = RC3, SDO = RC5, modo 0 a Fosc/4 ---------- */
void spi_init(void) { TRISC3 = 0; TRISC5 = 0; SSPSTAT = 0x40; SSPCON = 0x20; }
uint8_t spi_tx(uint8_t b) { SSPBUF = b; while (!SSPIF); SSPIF = 0; return SSPBUF; }

#define LOAD RC0

void max7219(uint8_t reg, uint8_t dato) { LOAD = 0; spi_tx(reg); spi_tx(dato); LOAD = 1; }

void mostrar4(uint8_t primero, uint16_t n) {           /* 4 dígitos BCD desde el registro 'primero' */
    for (uint8_t i = 0; i < 4; i++) {
        uint8_t d = n % 10; n /= 10;
        max7219((uint8_t)(primero - 3 + i), (uint8_t)(i == 0 ? d | 0x80 : d));   /* punto en el último */
    }
}

void main(void) {
    uint16_t segundos = 0;
    TRISC0 = 0; LOAD = 1;
    ADCON1 = 0x8E; ADCON0 = 0x41;                      /* AN0, 10 bits a la derecha */
    spi_init();
    max7219(0x0F, 0x00);                               /* sin prueba de display */
    max7219(0x09, 0xFF);                               /* decodificación BCD */
    max7219(0x0A, 0x08);                               /* brillo */
    max7219(0x0B, 0x07);                               /* 8 dígitos */
    max7219(0x0C, 0x01);                               /* encendido */
    while (1) {
        for (uint8_t k = 0; k < 10; k++) {
            mostrar4(8, segundos);
            __delay_us(50); GO_nDONE = 1; while (GO_nDONE);
            mostrar4(4, (uint16_t)((ADRESH << 8) | ADRESL));
            __delay_ms(100);
        }
        segundos++;
    }
}

In [ ]:
%%writefile ej07_pwm_dos_canales.c
/* Ejemplo 7 · PWM: dos canales de 10 bits a 1 kHz (CCP1 en RC2 y CCP2 en RC1) (MPLAB X + XC8)
   Pot 1 en AN0 → CCP1 · Pot 2 en AN1 → CCP2 · LCD con el ciclo útil */
#include <xc.h>
#include <stdint.h>
#pragma config FOSC = XT, WDTE = OFF, PWRTE = ON, BOREN = OFF, LVP = OFF, CPD = OFF, WRT = OFF, CP = OFF
#define _XTAL_FREQ 4000000UL
/* ---------- LCD 16x2 (HD44780) en modo de 4 bits: RS = RD0, E = RD1, D4..D7 = RD4..RD7, RW a tierra ---------- */
#define LCD_RS PORTDbits.RD0
#define LCD_E  PORTDbits.RD1

void lcd_nibble(uint8_t n) {            /* pone n<7:4> en D7..D4 y da un pulso en E */
    PORTD = (uint8_t)((PORTD & 0x0F) | (n & 0xF0));
    LCD_E = 1; __delay_us(1); LCD_E = 0;
}
void lcd_cmd(uint8_t c) {
    LCD_RS = 0; lcd_nibble(c); lcd_nibble((uint8_t)(c << 4));
    if (c < 3) __delay_ms(2); else __delay_us(50);
}
void lcd_char(char d) {
    LCD_RS = 1; lcd_nibble((uint8_t)d); lcd_nibble((uint8_t)(d << 4)); __delay_us(50);
}
void lcd_init(void) {
    TRISD = 0x00; PORTD = 0x00;
    __delay_ms(20);
    lcd_nibble(0x30); __delay_ms(5);
    lcd_nibble(0x30); __delay_us(150);
    lcd_nibble(0x30); __delay_us(50);
    lcd_nibble(0x20); __delay_us(50);     /* modo de 4 bits */
    lcd_cmd(0x28);                        /* 2 líneas, 5x8 */
    lcd_cmd(0x0C);                        /* display encendido, sin cursor */
    lcd_cmd(0x06);                        /* incremento automático */
    lcd_cmd(0x01);                        /* borrar */
}
void lcd_goto(uint8_t fila, uint8_t col) { lcd_cmd((uint8_t)((fila ? 0xC0 : 0x80) + col)); }
void lcd_print(const char *s) { while (*s) lcd_char(*s++); }

uint16_t leer_adc(uint8_t canal) {
    ADCON0 = (uint8_t)(0x41 | (canal << 3)); __delay_us(50);
    GO_nDONE = 1; while (GO_nDONE);
    return (uint16_t)((ADRESH << 8) | ADRESL);
}

void pwm1(uint16_t d) { CCPR1L = (uint8_t)(d >> 2); CCP1CON = (uint8_t)(0x0C | ((d & 3) << 4)); }
void pwm2(uint16_t d) { CCPR2L = (uint8_t)(d >> 2); CCP2CON = (uint8_t)(0x0C | ((d & 3) << 4)); }

void pct(uint8_t fila, uint16_t d) {                    /* d de 0 a 1000 → 0..100 % */
    uint8_t p = (uint8_t)(d / 10);
    lcd_goto(fila, 6);
    lcd_char(p >= 100 ? '1' : ' '); lcd_char(p >= 10 ? '0' + (p / 10) % 10 : ' '); lcd_char('0' + p % 10); lcd_char('%');
}

void main(void) {
    TRISC1 = 0; TRISC2 = 0;
    PR2 = 249;                                          /* 1 kHz con preescalador 4 */
    CCP1CON = 0x0C; CCP2CON = 0x0C;
    T2CON = 0x05;
    ADCON1 = 0x84;
    lcd_init();
    lcd_goto(0, 0); lcd_print("PWM1:      1 kHz");
    lcd_goto(1, 0); lcd_print("PWM2:      RC1");
    while (1) {
        uint16_t d1 = leer_adc(0), d2 = leer_adc(1);
        if (d1 > 1000) d1 = 1000;                       /* 1000 = 100 % con PR2 = 249 */
        if (d2 > 1000) d2 = 1000;
        pwm1(d1); pwm2(d2);
        pct(0, d1); pct(1, d2);
        __delay_ms(50);
    }
}

In [ ]:
%%writefile ej08_pwm_melodia.c
/* Ejemplo 8 · PWM: melodía en un parlante (RC2/CCP1) (MPLAB X + XC8)
   RB0 toca "Estrellita" · f = 62500 / (PR2 + 1) con preescalador 16 · 50 % de ciclo útil */
#include <xc.h>
#include <stdint.h>
#pragma config FOSC = XT, WDTE = OFF, PWRTE = ON, BOREN = OFF, LVP = OFF, CPD = OFF, WRT = OFF, CP = OFF
#define _XTAL_FREQ 4000000UL

const uint8_t PR2_NOTA[8] = {237, 212, 188, 178, 158, 141, 125, 118};   /* Do4 .. Do5 */
const uint8_t MELODIA[] = {0,1, 0,1, 4,1, 4,1, 5,1, 5,1, 4,2,  3,1, 3,1, 2,1, 2,1, 1,1, 1,1, 0,2,
                           4,1, 4,1, 3,1, 3,1, 2,1, 2,1, 1,2,  4,1, 4,1, 3,1, 3,1, 2,1, 2,1, 1,2,
                           0,1, 0,1, 4,1, 4,1, 5,1, 5,1, 4,2,  3,1, 3,1, 2,1, 2,1, 1,1, 1,1, 0,2, 0xFF};

void main(void) {
    OPTION_REGbits.nRBPU = 0;
    TRISC2 = 0; TRISC0 = 0; PORTC = 0;
    CCPR1L = 0; CCP1CON = 0x0C;
    T2CON = 0x06;                                       /* TMR2 encendido, 1:16 */
    while (1) {
        while (RB0);                                    /* espera el botón */
        RC0 = 1;
        for (uint8_t i = 0; MELODIA[i] != 0xFF; i += 2) {
            PR2 = PR2_NOTA[MELODIA[i]];
            CCPR1L = (uint8_t)((PR2 + 1) / 2);          /* 50 % */
            for (uint8_t d = MELODIA[i + 1]; d; d--) __delay_ms(250);
            CCPR1L = 0; __delay_ms(20);                 /* silencio entre notas */
        }
        RC0 = 0;
    }
}

In [ ]:
%%writefile ej09_isr_fuentes.c
/* Ejemplo 9 · Interrupciones: INT, cambio en RB4-RB7, ADC y TMR0 en una sola ISR (MPLAB X + XC8) */
#include <xc.h>
#include <stdint.h>
#pragma config FOSC = XT, WDTE = OFF, PWRTE = ON, BOREN = OFF, LVP = OFF, CPD = OFF, WRT = OFF, CP = OFF
#define _XTAL_FREQ 4000000UL
/* ---------- LCD 16x2 (HD44780) en modo de 4 bits: RS = RD0, E = RD1, D4..D7 = RD4..RD7, RW a tierra ---------- */
#define LCD_RS PORTDbits.RD0
#define LCD_E  PORTDbits.RD1

void lcd_nibble(uint8_t n) {            /* pone n<7:4> en D7..D4 y da un pulso en E */
    PORTD = (uint8_t)((PORTD & 0x0F) | (n & 0xF0));
    LCD_E = 1; __delay_us(1); LCD_E = 0;
}
void lcd_cmd(uint8_t c) {
    LCD_RS = 0; lcd_nibble(c); lcd_nibble((uint8_t)(c << 4));
    if (c < 3) __delay_ms(2); else __delay_us(50);
}
void lcd_char(char d) {
    LCD_RS = 1; lcd_nibble((uint8_t)d); lcd_nibble((uint8_t)(d << 4)); __delay_us(50);
}
void lcd_init(void) {
    TRISD = 0x00; PORTD = 0x00;
    __delay_ms(20);
    lcd_nibble(0x30); __delay_ms(5);
    lcd_nibble(0x30); __delay_us(150);
    lcd_nibble(0x30); __delay_us(50);
    lcd_nibble(0x20); __delay_us(50);     /* modo de 4 bits */
    lcd_cmd(0x28);                        /* 2 líneas, 5x8 */
    lcd_cmd(0x0C);                        /* display encendido, sin cursor */
    lcd_cmd(0x06);                        /* incremento automático */
    lcd_cmd(0x01);                        /* borrar */
}
void lcd_goto(uint8_t fila, uint8_t col) { lcd_cmd((uint8_t)((fila ? 0xC0 : 0x80) + col)); }
void lcd_print(const char *s) { while (*s) lcd_char(*s++); }

volatile uint8_t pulsos = 0, tecla = 0, t0cnt = 0, cambio = 1;
volatile uint16_t adc = 0;

void __interrupt() isr(void) {
    if (INTE && INTF) { INTF = 0; pulsos++; cambio = 1; }          /* 1) INT */
    if (RBIF) {                                                    /* 2) cambio en RB4-RB7 */
        uint8_t b = PORTB; RBIF = 0;                               /* leer PORTB termina el cambio */
        for (uint8_t k = 4; k < 8; k++) if (!(b & (1 << k))) tecla = k;
        cambio = 1;
    }
    if (ADIF) { ADIF = 0; adc = (uint16_t)((ADRESH << 8) | ADRESL); cambio = 1; }   /* 3) ADC */
    if (TMR0IF) {                                                  /* 4) TMR0 cada 65,5 ms */
        TMR0IF = 0; GO_nDONE = 1;
        if (++t0cnt >= 4) { t0cnt = 0; RC0 ^= 1; }
    }
}

void main(void) {
    TRISC0 = 0; RC0 = 0;
    OPTION_REG = 0x07;                                  /* pull-ups, INT en bajada, TMR0 1:256 */
    ADCON1 = 0x8E; ADCON0 = 0x41; ADIE = 1;
    lcd_init();
    (void)PORTB; RBIF = 0;
    INTCON = 0xF8;                                      /* GIE PEIE T0IE INTE RBIE */
    while (1) {
        if (!cambio) continue;
        cambio = 0;
        GIE = 0; uint16_t a = adc; uint8_t p = pulsos, t = tecla; GIE = 1;   /* copia atómica */
        lcd_goto(0, 0); lcd_print("INT=");
        lcd_char('0' + p / 100); lcd_char('0' + (p / 10) % 10); lcd_char('0' + p % 10);
        lcd_print("  RB="); lcd_char(t ? '0' + t : '-');
        lcd_goto(1, 0); lcd_print("ADC=");
        lcd_char('0' + a / 1000); lcd_char('0' + (a / 100) % 10); lcd_char('0' + (a / 10) % 10); lcd_char('0' + a % 10);
    }
}

In [ ]:
%%writefile ej10_isr_reloj.c
/* Ejemplo 10 · Interrupciones: reloj HH:MM:SS con TMR1 (0,5 s) y ajuste por INT (MPLAB X + XC8) */
#include <xc.h>
#include <stdint.h>
#pragma config FOSC = XT, WDTE = OFF, PWRTE = ON, BOREN = OFF, LVP = OFF, CPD = OFF, WRT = OFF, CP = OFF
#define _XTAL_FREQ 4000000UL
/* ---------- LCD 16x2 (HD44780) en modo de 4 bits: RS = RD0, E = RD1, D4..D7 = RD4..RD7, RW a tierra ---------- */
#define LCD_RS PORTDbits.RD0
#define LCD_E  PORTDbits.RD1

void lcd_nibble(uint8_t n) {            /* pone n<7:4> en D7..D4 y da un pulso en E */
    PORTD = (uint8_t)((PORTD & 0x0F) | (n & 0xF0));
    LCD_E = 1; __delay_us(1); LCD_E = 0;
}
void lcd_cmd(uint8_t c) {
    LCD_RS = 0; lcd_nibble(c); lcd_nibble((uint8_t)(c << 4));
    if (c < 3) __delay_ms(2); else __delay_us(50);
}
void lcd_char(char d) {
    LCD_RS = 1; lcd_nibble((uint8_t)d); lcd_nibble((uint8_t)(d << 4)); __delay_us(50);
}
void lcd_init(void) {
    TRISD = 0x00; PORTD = 0x00;
    __delay_ms(20);
    lcd_nibble(0x30); __delay_ms(5);
    lcd_nibble(0x30); __delay_us(150);
    lcd_nibble(0x30); __delay_us(50);
    lcd_nibble(0x20); __delay_us(50);     /* modo de 4 bits */
    lcd_cmd(0x28);                        /* 2 líneas, 5x8 */
    lcd_cmd(0x0C);                        /* display encendido, sin cursor */
    lcd_cmd(0x06);                        /* incremento automático */
    lcd_cmd(0x01);                        /* borrar */
}
void lcd_goto(uint8_t fila, uint8_t col) { lcd_cmd((uint8_t)((fila ? 0xC0 : 0x80) + col)); }
void lcd_print(const char *s) { while (*s) lcd_char(*s++); }

volatile uint8_t hh = 12, mm = 0, ss = 0, medio = 0, nuevo = 1;

void sumar_hora(void) { if (++hh >= 24) hh = 0; }
void sumar_min(void) { if (++mm >= 60) { mm = 0; sumar_hora(); } }

void __interrupt() isr(void) {
    if (TMR1IF) {
        TMR1IF = 0;
        TMR1ON = 0; TMR1H = 0x0B; TMR1L = 0xDC; TMR1ON = 1;      /* 65536 - 62500 */
        RC0 ^= 1;
        if (++medio >= 2) { medio = 0; if (++ss >= 60) { ss = 0; sumar_min(); } nuevo = 1; }
    }
    if (INTF) { INTF = 0; sumar_min(); nuevo = 1; }
}

void dos(uint8_t v) { lcd_char('0' + v / 10); lcd_char('0' + v % 10); }

void main(void) {
    uint8_t antes = 1;
    TRISC0 = 0; RC0 = 0;
    OPTION_REG = 0x00;                                  /* pull-ups, INT en bajada */
    lcd_init(); lcd_goto(0, 0); lcd_print("Reloj con TMR1");
    TMR1H = 0x0B; TMR1L = 0xDC; T1CON = 0x31; TMR1IE = 1;
    INTCON = 0xD0;                                      /* GIE PEIE INTE */
    while (1) {
        if (!RB1 && antes) { GIE = 0; sumar_hora(); GIE = 1; nuevo = 1; }
        antes = RB1;
        if (nuevo) { nuevo = 0; lcd_goto(1, 4); dos(hh); lcd_char(':'); dos(mm); lcd_char(':'); dos(ss); }
    }
}

In [ ]:
%%writefile ej11_multitarea_planificador.c
/* Ejemplo 11 · Multitarea cooperativa con tick de 10 ms (MPLAB X + XC8)
   LED1 250 ms · LED2 400 ms · botón 20 ms · ADC→PWM 50 ms · LCD 500 ms */
#include <xc.h>
#include <stdint.h>
#pragma config FOSC = XT, WDTE = OFF, PWRTE = ON, BOREN = OFF, LVP = OFF, CPD = OFF, WRT = OFF, CP = OFF
#define _XTAL_FREQ 4000000UL
/* ---------- LCD 16x2 (HD44780) en modo de 4 bits: RS = RD0, E = RD1, D4..D7 = RD4..RD7, RW a tierra ---------- */
#define LCD_RS PORTDbits.RD0
#define LCD_E  PORTDbits.RD1

void lcd_nibble(uint8_t n) {            /* pone n<7:4> en D7..D4 y da un pulso en E */
    PORTD = (uint8_t)((PORTD & 0x0F) | (n & 0xF0));
    LCD_E = 1; __delay_us(1); LCD_E = 0;
}
void lcd_cmd(uint8_t c) {
    LCD_RS = 0; lcd_nibble(c); lcd_nibble((uint8_t)(c << 4));
    if (c < 3) __delay_ms(2); else __delay_us(50);
}
void lcd_char(char d) {
    LCD_RS = 1; lcd_nibble((uint8_t)d); lcd_nibble((uint8_t)(d << 4)); __delay_us(50);
}
void lcd_init(void) {
    TRISD = 0x00; PORTD = 0x00;
    __delay_ms(20);
    lcd_nibble(0x30); __delay_ms(5);
    lcd_nibble(0x30); __delay_us(150);
    lcd_nibble(0x30); __delay_us(50);
    lcd_nibble(0x20); __delay_us(50);     /* modo de 4 bits */
    lcd_cmd(0x28);                        /* 2 líneas, 5x8 */
    lcd_cmd(0x0C);                        /* display encendido, sin cursor */
    lcd_cmd(0x06);                        /* incremento automático */
    lcd_cmd(0x01);                        /* borrar */
}
void lcd_goto(uint8_t fila, uint8_t col) { lcd_cmd((uint8_t)((fila ? 0xC0 : 0x80) + col)); }
void lcd_print(const char *s) { while (*s) lcd_char(*s++); }

volatile uint8_t ticks = 0, ms10 = 0;
uint8_t pulsos = 0, antes = 1, pot = 0;

void __interrupt() isr(void) {
    if (TMR0IF) { TMR0IF = 0; TMR0 += 6; if (++ms10 >= 10) { ms10 = 0; ticks++; } }
}

void tarea_led1(void) { RC0 ^= 1; }
void tarea_led2(void) { RC1 ^= 1; }
void tarea_boton(void) { if (!RB0 && antes) pulsos++; antes = RB0; }
void tarea_adc(void) { if (!GO_nDONE) { pot = ADRESH; CCPR1L = pot; GO_nDONE = 1; } }
void tres(uint8_t v) { lcd_char('0' + v / 100); lcd_char('0' + (v / 10) % 10); lcd_char('0' + v % 10); }
void tarea_lcd(void) { lcd_goto(0, 8); tres(pulsos); lcd_goto(1, 8); tres(pot); }

typedef struct { uint8_t periodo, cuenta; void (*f)(void); } tarea_t;
tarea_t tareas[] = { {25, 1, tarea_led1}, {40, 1, tarea_led2}, {2, 1, tarea_boton}, {5, 1, tarea_adc}, {50, 1, tarea_lcd} };

void main(void) {
    TRISC = 0xF8; PORTC = 0;
    OPTION_REG = 0x01;                                  /* pull-ups, TMR0 1:4 */
    ADCON1 = 0x0E; ADCON0 = 0x41;
    PR2 = 255; CCP1CON = 0x0C; T2CON = 0x04;
    lcd_init(); lcd_goto(0, 0); lcd_print("Pulsos:"); lcd_goto(1, 0); lcd_print("PWM   :");
    GO_nDONE = 1;
    INTCON = 0xA0;                                      /* GIE T0IE */
    while (1) {
        if (!ticks) continue;                           /* espera un tick */
        GIE = 0; ticks--; GIE = 1;
        for (uint8_t i = 0; i < sizeof(tareas) / sizeof(tareas[0]); i++)
            if (--tareas[i].cuenta == 0) { tareas[i].cuenta = tareas[i].periodo; tareas[i].f(); }
    }
}

In [ ]:
%%writefile ej12_multitarea_semaforo.c
/* Ejemplo 12 · Multitarea: semáforo + latido + comandos por USART (MPLAB X + XC8)
   P = solicitud peatonal · S = estado · RB0..RB4 semáforo · RB5 espere · RB7 latido */
#include <xc.h>
#include <stdint.h>
#pragma config FOSC = XT, WDTE = OFF, PWRTE = ON, BOREN = OFF, LVP = OFF, CPD = OFF, WRT = OFF, CP = OFF
#define _XTAL_FREQ 4000000UL
/* ---------- USART 9600 baudios (4 MHz, BRGH = 1, SPBRG = 25) ---------- */
void uart_init(void) { SPBRG = 25; TXSTA = 0x24; RCSTA = 0x90; }
void uart_tx(char c) { while (!TXIF); TXREG = c; }
void uart_print(const char *s) { while (*s) uart_tx(*s++); }

enum { VERDE, AMARILLO, ROJO, DESPEJE };
const uint8_t LUCES[4] = {0x0C, 0x0A, 0x11, 0x09};
const char *NOMBRE[4] = {"VERDE", "AMARILLO", "ROJO (pasa el peaton)", "TODO EN ROJO"};
volatile uint8_t ticks = 0, ms10 = 0, rxflag = 0;
volatile char rxdato;
uint8_t estado = VERDE, tiempo = 0, solicitud = 0;

void __interrupt() isr(void) {
    if (TMR0IF) { TMR0IF = 0; TMR0 += 6; if (++ms10 >= 10) { ms10 = 0; ticks++; } }
    if (RCIF) { rxdato = RCREG; rxflag = 1; }
}

void anunciar(void) { uart_print("Semaforo: "); uart_print(NOMBRE[estado]); uart_print("\r\n"); }
void cambiar(uint8_t e) {
    estado = e; tiempo = 0;
    PORTB = (uint8_t)((PORTB & 0xA0) | LUCES[estado]);
    anunciar();
}

void tarea_serie(void) {
    if (!rxflag) return;
    rxflag = 0;
    char c = (char)(rxdato & 0xDF);
    if (c == 'P') { solicitud = 1; RB5 = 1; uart_print("Solicitud peatonal registrada\r\n"); }
    if (c == 'S') anunciar();
}

void tarea_semaforo(void) {                            /* cada 100 ms */
    tiempo++;
    switch (estado) {
        case VERDE:    if (tiempo >= 80 || (solicitud && tiempo >= 30)) cambiar(AMARILLO); break;
        case AMARILLO: if (tiempo >= 20) { solicitud = 0; RB5 = 0; cambiar(ROJO); } break;
        case ROJO:     if (tiempo >= 50) cambiar(DESPEJE); break;
        default:       if (tiempo >= 10) cambiar(VERDE);
    }
}

void main(void) {
    uint8_t c_sem = 10, c_lat = 30;
    TRISB = 0; PORTB = 0;
    OPTION_REG = 0x01;
    uart_init(); RCIE = 1;
    uart_print("PIC16F877A multitarea: P = solicitud peatonal, S = estado\r\n");
    cambiar(VERDE);
    INTCON = 0xE0;                                      /* GIE PEIE T0IE */
    while (1) {
        if (!ticks) continue;
        GIE = 0; ticks--; GIE = 1;
        tarea_serie();                                  /* cada 10 ms */
        if (--c_sem == 0) { c_sem = 10; tarea_semaforo(); }
        if (--c_lat == 0) { c_lat = 30; RB7 ^= 1; }
    }
}

## 3. Ensamblar y generar los `.hex`
En Proteus: doble clic sobre el PIC → *Program File* = el `.hex`, *Processor Clock Frequency* = 4 MHz.

In [ ]:
import importlib, picasm, pic16sim, dispositivos
for m in (picasm, pic16sim, dispositivos): importlib.reload(m)
from picasm import Ensamblador, leer_hex, desensamblar
from pic16sim import PIC16F877
from dispositivos import LCD16x2, BusI2C, PCF8574, DS1307, HC595, MAX7219

EJ = ['ej01_usart_comandos', 'ej02_usart_telemetria', 'ej03_i2c_pcf8574', 'ej04_i2c_ds1307', 'ej05_spi_74hc595', 'ej06_spi_max7219', 'ej07_pwm_dos_canales', 'ej08_pwm_melodia', 'ej09_isr_fuentes', 'ej10_isr_reloj', 'ej11_multitarea_planificador', 'ej12_multitarea_semaforo']
PROG, SIM = {}, {}
for n in EJ:
    a = Ensamblador(open(n + '.asm', encoding='utf-8').read()).ensamblar()
    open(n + '.hex', 'w').write(a.hex())
    with open(n + '.lst', 'w', encoding='utf-8') as L:
        for pc, w, t in a.lista: L.write(f'{pc:04X}  {w:04X}  {desensamblar(w):<22s} ; {t}\n')
    PROG[n], conf = leer_hex(a.hex()); SIM[n] = a.simbolos
    print(f'{n + ".hex":34s} {len(PROG[n]):5d} palabras · config 0x{conf:04X}')

def nuevo(n):
    return PIC16F877(PROG[n], 4e6)

In [ ]:
# Descargar todo en un .zip (fuentes, .hex y listados)
import zipfile, glob
with zipfile.ZipFile('pic16f877a_12_ejemplos.zip', 'w') as z:
    for f in sorted(glob.glob('ej[01]*_*.*')):
        if not f.endswith('.zip'): z.write(f)
print('Creado pic16f877a_12_ejemplos.zip')
try:
    from google.colab import files; files.download('pic16f877a_12_ejemplos.zip')
except ImportError:
    print('(fuera de Colab el archivo queda en la carpeta actual)')

## 4. Funciones de apoyo

In [ ]:
import numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

def ver_lcd(lcd, titulo=''):
    l1, l2 = lcd.texto()
    print(('  ' + titulo) if titulo else '')
    print('  ┌' + '─' * 16 + '┐'); print('  │' + l1.ljust(16) + '│'); print('  │' + l2.ljust(16) + '│'); print('  └' + '─' * 16 + '┘')

def pulsar(p, puerto, bit, t=0.06, despues=0.06):
    p.poner_entrada(puerto, 0xFF & ~(1 << bit)); p.correr(segundos=t)
    p.poner_entrada(puerto, 0xFF); p.correr(segundos=despues)

def terminal(p):
    """lo que el PIC transmitió por la USART desde la última llamada"""
    s = bytes(p.tx).decode('ascii', 'replace'); p.tx.clear(); return s

def enviar(p, texto, espera=0.03):
    for c in texto: p.recibir(ord(c)); p.correr(segundos=espera)

## 5. USART
Trama asíncrona 8N1: un bit de inicio (0), 8 datos con el **LSB primero** y un bit de parada (1). A 9600 baudios cada bit dura 104,2 µs.

Con BRGH = 1: $\text{baudios} = \dfrac{F_{osc}}{16\,(\text{SPBRG}+1)}$ → con 4 MHz, SPBRG = 25 da 9615 baudios (error de 0,16 %).

In [ ]:
fosc = 4e6
print('baudios   BRGH=1: SPBRG  real    error   |  BRGH=0: SPBRG  real    error')
for b in (2400, 9600, 19200, 57600):
    fila = f'{b:6d}  '
    for k in (16, 64):
        n = round(fosc / (k * b) - 1)
        if 0 <= n <= 255: r = fosc / (k * (n + 1)); fila += f'      {n:5d} {r:8.0f} {100*(r-b)/b:+6.2f} %  |'
        else: fila += '          -        -         |'
    print(fila)

def trama_uart(byte, ax, baud=9600, color='#1F8A70'):
    bits = [0] + [(byte >> i) & 1 for i in range(8)] + [1]
    t = np.arange(len(bits) + 1) * 1e6 / baud
    ax.step(np.r_[-50, t], np.r_[1, bits, 1], where='post', color=color, lw=2)
    for i, b in enumerate(bits):
        ax.text((i + 0.5) * 1e6 / baud, 1.25, 'inicio' if i == 0 else 'parada' if i == 9 else f'D{i-1}={b}', ha='center', fontsize=8)
    ax.set_ylim(-0.3, 1.5); ax.set_yticks([0, 1]); ax.set_xlabel('µs'); ax.set_title(f"Trama 8N1 del carácter '{chr(byte)}' = 0x{byte:02X} en RC6/TX")
fig, ax = plt.subplots(figsize=(11, 2.4)); trama_uart(ord('A'), ax); plt.show()

### Ejemplo 1 · Terminal de comandos con interrupción de recepción
Cada carácter que llega dispara la interrupción RCIF; la ISR lo guarda y el programa principal lo interpreta.

In [ ]:
p = nuevo('ej01_usart_comandos'); p.correr(segundos=0.15); print(terminal(p))
for c in ['1', '3', '5', '?', 'A', 'B', '8']:
    enviar(p, c); print(f'» {c}   PORTB = {p.puerto("B"):08b}   respuesta: {terminal(p).strip()}')

### Ejemplo 2 · Telemetría al PC
Se mueve el potenciómetro y la temperatura del LM35 mientras corre el programa; las líneas que llegan se interpretan igual que lo haría `pc/graficar_serie.py`.

In [ ]:
import re
p = nuevo('ej02_usart_telemetria'); p.correr(segundos=0.1); terminal(p)
ts, v_real, t_real = [], [], []
for k in range(60):
    t = k * 0.1; p.analog[0] = 2.5 + 2.4 * np.sin(2 * np.pi * t / 4); p.analog[1] = (25 + 10 * t / 6) * 0.01
    p.correr(segundos=0.1); ts.append(p.tiempo()); v_real.append(p.analog[0]); t_real.append(p.analog[1] * 100)
lineas = [m.groups() for m in re.finditer(r'AN0=(\d+) V=([\d.]+) T=([\d.]+)C', terminal(p))]
print(len(lineas), 'líneas recibidas · ejemplo:', lineas[0])
V = [float(l[1]) for l in lineas]; T = [float(l[2]) for l in lineas]; tl = np.arange(len(V)) * 0.25 + ts[0]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
ax[0].plot(ts, v_real, color='#BBB', label='real'); ax[0].plot(tl, V, 'o', ms=4, color='#1F8A70', label='recibido'); ax[0].set_title('AN0 (V)'); ax[0].legend()
ax[1].plot(ts, t_real, color='#BBB', label='real'); ax[1].plot(tl, T, 'o', ms=4, color='#E2453A', label='recibido'); ax[1].set_title('LM35 (°C)'); ax[1].legend()
for a in ax: a.set_xlabel('s')
plt.tight_layout(); plt.show()

## 6. I2C
Dos hilos con resistencias de pull-up: **SCL** (RC3) y **SDA** (RC4). El maestro genera START (SDA baja con SCL en 1), envía la dirección de 7 bits + R/W, el esclavo responde **ACK** (SDA en 0 en el noveno pulso), siguen los datos y termina con STOP.

Con el MSSP: `SSPCON = 0x28` (I2C maestro), `SSPADD = Fosc/(4·100 kHz) − 1 = 9`. Cada paso se pide con un bit de **SSPCON2** (SEN, RSEN, PEN, RCEN, ACKEN) y se espera la bandera **SSPIF**.

In [ ]:
def onda_i2c(bytes_, ax, titulo):
    """dibuja SCL y SDA de una escritura: START, bytes con su ACK, STOP"""
    t, scl, sda = [0, 1], [1, 1], [1, 0]           # START: SDA baja con SCL alto
    x = 1.5
    for b in bytes_:
        for i in range(9):
            bit = (b >> (7 - i)) & 1 if i < 8 else 0   # noveno bit: ACK del esclavo
            t += [x, x, x + 0.5, x + 1]; scl += [0, 0, 1, 0]; sda += [sda[-1], bit, bit, bit]
            ax.text(x + 0.5, -0.9, 'A' if i == 8 else str(bit), ha='center', fontsize=7, color='#E2453A' if i == 8 else 'k'); x += 1
    t += [x, x + 0.5, x + 1]; scl += [0, 1, 1]; sda += [0, 0, 1]   # STOP: SDA sube con SCL alto
    ax.step(t, np.array(scl) + 2, where='post', color='#2F6DB5', lw=1.8); ax.step(t, sda, where='post', color='#1F8A70', lw=1.8)
    ax.set_yticks([0.5, 2.5]); ax.set_yticklabels(['SDA', 'SCL']); ax.set_ylim(-1.3, 3.4); ax.set_xticks([]); ax.set_title(titulo)
    ax.text(0.5, 3.15, 'S', ha='center'); ax.text(x + 0.5, 3.15, 'P', ha='center')

p = nuevo('ej03_i2c_pcf8574'); bus = BusI2C(p); pcf = bus.agregar(PCF8574())
patrones = []
for k in range(16): p.correr(segundos=0.13); patrones.append(~pcf.salida & 0xFF)
print('Registro del bus (S = START, 40+ = dirección con ACK, P = STOP):', ' '.join(bus.log[:12]))
fig, ax = plt.subplots(2, 1, figsize=(11, 5), gridspec_kw={'height_ratios': [1, 1.2]})
onda_i2c([0x40, pcf.salida], ax[0], f'Escritura al PCF8574: dirección 0x40 y dato 0x{pcf.salida:02X}')
ax[1].imshow(np.array([[(v >> b) & 1 for b in range(8)] for v in patrones]).T, cmap='Reds', aspect='auto', interpolation='nearest')
ax[1].set_yticks(range(8)); ax[1].set_yticklabels([f'P{b}' for b in range(8)]); ax[1].set_xlabel('patrón enviado (cada 130 ms)'); ax[1].set_title('LED encendidos: barrido')
plt.tight_layout(); plt.show()
q = nuevo('ej03_i2c_pcf8574'); bq = BusI2C(q); q.correr(segundos=0.2)
print('Sin PCF8574 en el bus:', ' '.join(bq.log[:6]), '→ LED de error RC0 =', q.puerto('C') & 1)

### Ejemplo 4 · Reloj DS1307
Lectura secuencial: START, 0xD0, puntero 0, **RESTART**, 0xD1, 7 bytes con ACK y el último con **NACK**, STOP. Los registros vienen en BCD.

In [ ]:
p = nuevo('ej04_i2c_ds1307'); lcd = LCD16x2(p); bus = BusI2C(p); rtc = bus.agregar(DS1307(p))
p.correr(segundos=0.3); ver_lcd(lcd, 'al encender (el PIC puso el reloj en hora)')
p.correr(segundos=5); ver_lcd(lcd, '5 s después')
pulsar(p, 'B', 0, 0.3, 0.3); ver_lcd(lcd, 'después de RB0 (+1 minuto)')
# una lectura completa del bus
log = bus.log; i = len(log) - 1 - log[::-1].index('S'); j = i - 1 - log[i-1::-1].index('S')
print('Última lectura:', ' '.join(log[j:i + 25]))
regs = [int(x[1:], 16) for x in log[i:] if x.startswith('r')][:7]
bcd = lambda v: (v >> 4) * 10 + (v & 15)
print('registros:', ' '.join(f'{v:02X}' for v in regs), '→', f'{bcd(regs[2]):02d}:{bcd(regs[1]):02d}:{bcd(regs[0] & 0x7F):02d}  {bcd(regs[4]):02d}/{bcd(regs[5]):02d}/{bcd(regs[6]):02d}')

## 7. SPI
Tres hilos y un selector: **SCK** (RC3), **SDO** (RC5), **SDI** (RC4) y un pin de selección o *latch* (RC0). Es síncrono y en *full duplex*: con cada flanco de SCK sale un bit y entra otro.

Modo 0 (CKP = 0, CKE = 1): el dato cambia en el flanco de bajada y se lee en el de subida. `SSPCON = 0x20` (maestro, Fosc/4 = 1 MHz) y `SSPSTAT = 0x40`.

In [ ]:
def onda_spi(bytes_, ax, titulo, latch=True):
    t, sck, sdo = [], [], []; x = 0
    for b in bytes_:
        for i in range(8):
            bit = (b >> (7 - i)) & 1
            t += [x, x + 0.5]; sck += [0, 1]; sdo += [bit, bit]
            ax.text(x + 0.5, -0.8, str(bit), ha='center', fontsize=7); x += 1
        x += 0.5; t += [x - 0.5]; sck += [0]; sdo += [sdo[-1]]
    t.append(x); sck.append(0); sdo.append(sdo[-1])
    ax.step(t, np.array(sck) + 4, where='post', color='#2F6DB5', lw=1.6); ax.step(t, np.array(sdo) + 2, where='post', color='#1F8A70', lw=1.6)
    if latch: ax.step([0, x - 0.3, x - 0.3, x + 0.6], [0, 0, 1, 1], where='post', color='#E2453A', lw=1.6)
    ax.set_yticks([0.5, 2.5, 4.5]); ax.set_yticklabels(['RC0 latch', 'SDO', 'SCK']); ax.set_xticks([]); ax.set_ylim(-1.2, 5.6); ax.set_title(titulo)

p = nuevo('ej05_spi_74hc595'); h = HC595(p)
cont = []
for k in range(6): p.correr(segundos=0.155); cont.append(h.salida)
pulsar(p, 'B', 0, 0.2, 0.05)
volt = np.linspace(0, 5, 21); barra = []
for v in volt: p.analog[0] = v; p.correr(segundos=0.3); barra.append(bin(h.salida).count('1'))
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2), gridspec_kw={'width_ratios': [1.6, 1]})
onda_spi([cont[-1]], ax[0], f'Byte 0x{cont[-1]:02X} al 74HC595 y flanco de latch')
ax[1].step(volt, barra, where='mid', color='#2E9B57'); ax[1].set_xlabel('AN0 (V)'); ax[1].set_ylabel('LED encendidos'); ax[1].set_title('Modo barra')
plt.tight_layout(); plt.show()
print('contador:', [f'{v:08b}' for v in cont])

### Ejemplo 6 · MAX7219
Cada escritura son 16 bits: primero la dirección del registro y después el dato. LOAD sube y el MAX7219 lo guarda.

In [ ]:
p = nuevo('ej06_spi_max7219'); m = MAX7219(p); tramas = []
buf = []
p.spi.append(lambda pi, v: buf.append(v))
def al_latch(pi, pu, ant, n):
    if pu == 'C' and not ant & 1 and n & 1 and len(buf) >= 2: tramas.append((buf[-2], buf[-1])); buf.clear()
p.escuchas.append(al_latch)
p.analog[0] = 3.3; p.correr(segundos=3.2)
NOM = {9: 'decodificación BCD', 10: 'intensidad', 11: 'límite de dígitos', 12: 'apagado/encendido', 15: 'prueba de display'}
print('Primeras tramas (configuración):')
for r, d in tramas[:6]: print(f'   {r:02X} {d:02X}  {NOM.get(r, f"dígito {r-1}")} = {d}')
print('Display:', repr(m.texto()), '· izquierda = segundos, derecha = ADC de AN0')
fig, ax = plt.subplots(figsize=(11, 2.4)); r, d = tramas[-1]; onda_spi([r, d], ax, f'Trama de 16 bits: registro 0x{r:02X}, dato 0x{d:02X}'); plt.show()

## 8. PWM
$T = (\text{PR2}+1)\cdot 4\,T_{osc}\cdot \text{preescalador}$ y el ancho $= (\text{CCPRxL}:\text{CCPxCON}\langle5{:}4\rangle)\cdot T_{osc}\cdot \text{preescalador}$, con 10 bits de resolución.

Con 4 MHz, PR2 = 249 y preescalador 4: $T = 250 \cdot 1\,\mu s \cdot 4 = 1$ ms → 1 kHz. El ciclo útil completo es $4\,(\text{PR2}+1) = 1000$ cuentas.

In [ ]:
p = nuevo('ej07_pwm_dos_canales'); lcd = LCD16x2(p)
volts = np.linspace(0, 5, 11); d1, d2 = [], []
for v in volts:
    p.analog[0] = v; p.analog[1] = 5 - v; p.correr(segundos=0.15)
    d1.append(p.pwm(1)['D'] * 100); d2.append(p.pwm(2)['D'] * 100)
ver_lcd(lcd, 'AN0 = 5 V, AN1 = 0 V')
fig, ax = plt.subplots(1, 2, figsize=(11, 3.3))
ax[0].plot(volts, d1, 'o-', label='CCP1 (AN0)', color='#F2A33A'); ax[0].plot(volts, d2, 's-', label='CCP2 (AN1 = 5 V − AN0)', color='#2E9B57')
ax[0].set_xlabel('AN0 (V)'); ax[0].set_ylabel('ciclo útil (%)'); ax[0].legend(); ax[0].set_title('ADC de 10 bits → ciclo útil')
p.analog[0] = 2.5; p.analog[1] = 1.0; p.correr(segundos=0.15)
t = np.linspace(0, 3e-3, 3000)
for k, (n, c) in enumerate(((1, '#F2A33A'), (2, '#2E9B57'))):
    w = p.pwm(n); ax[1].plot(t * 1e3, ((t * w['f']) % 1 < w['D']) + 1.4 * k, color=c, label=f'CCP{n}: {w["f"]:.0f} Hz, {w["D"]*100:.1f} %')
ax[1].set_xlabel('ms'); ax[1].set_yticks([]); ax[1].legend(loc='upper right', fontsize=8); ax[1].set_title('Osciloscopio: 1 ms de periodo')
plt.tight_layout(); plt.show()

### Ejemplo 8 · Melodía
La nota es la frecuencia del PWM: con preescalador 16, $f = 62\,500/(\text{PR2}+1)$ Hz, y el ciclo útil se fija en 50 %. La celda reconstruye el audio y lo deja escuchar.

In [ ]:
p = nuevo('ej08_pwm_melodia'); p.correr(segundos=0.1); pulsar(p, 'B', 0, 0.05, 0)
dt = 0.01; f = []
for k in range(1500):
    p.correr(segundos=dt); w = p.pwm(1); f.append(w['f'] if w and w['D'] > 0.1 else 0)
f = np.array(f); t = np.arange(len(f)) * dt
NOT = [(262, 'DO'), (294, 'RE'), (330, 'MI'), (349, 'FA'), (392, 'SOL'), (440, 'LA'), (494, 'SI'), (523, "DO'")]
plt.figure(figsize=(11, 3)); plt.scatter(t[f > 0], f[f > 0], s=3, color='#1F8A70')
plt.yticks([n for n, _ in NOT], [f'{s} {n}' for n, s in NOT]); plt.xlabel('s'); plt.title('Estrellita, ¿dónde estás? · frecuencia del PWM en RC2'); plt.show()
sr = 22050; fase = np.cumsum(np.repeat(f, int(sr * dt)) / sr); audio = 0.3 * np.sign(np.sin(2 * np.pi * fase)) * (np.repeat(f, int(sr * dt)) > 0)
from IPython.display import Audio, display
display(Audio(audio, rate=sr))

## 9. Interrupciones e ISR
Cuando una bandera (xxIF) y su habilitación (xxIE) están en 1 y **GIE** = 1 (más **PEIE** para los periféricos), el PIC termina la instrucción, guarda el PC en la pila, borra GIE y salta al **vector 0x004**. La ISR guarda W, STATUS y PCLATH, revisa las banderas una por una (el orden es la prioridad), las borra y termina con `RETFIE`.

La celda mide cuánto tiempo pasa el PIC dentro de la ISR del ejemplo 9 y en qué momentos entra.

In [ ]:
p = nuevo('ej09_isr_fuentes'); lcd = LCD16x2(p); p.analog[0] = 3.3
ini = SIM['ej09_isr_fuentes']['INICIO']; p.correr(segundos=0.2)
entradas, dentro, total, ant = [], 0, 0, False
eventos = {0.05: ('B', 0xFE), 0.08: ('B', 0xFF), 0.15: ('B', 0xBF), 0.18: ('B', 0xFF)}
t0 = p.tiempo(); prox = sorted(eventos)
while p.tiempo() - t0 < 0.25:
    if prox and p.tiempo() - t0 >= prox[0]: p.poner_entrada(*eventos[prox.pop(0)])
    c0 = p.cy; en = 4 <= p.pc < ini; p.paso(); d = p.cy - c0; total += d
    if en: dentro += d
    if en and not ant: entradas.append((p.tiempo() - t0, p.ram[0x0B], p.ram[0x0C]))
    ant = en
ver_lcd(lcd)
print(f'{len(entradas)} entradas a la ISR en 250 ms · CPU dentro de la ISR: {100 * dentro / total:.1f} %')
tipo = lambda ic, pir: 'INT' if ic & 2 else 'RB' if ic & 1 else 'ADC' if pir & 0x40 else 'TMR0' if ic & 4 else '?'
fig, ax = plt.subplots(figsize=(11, 2.6)); fil = {'TMR0': 0, 'ADC': 1, 'RB': 2, 'INT': 3}
for tt, ic, pir in entradas:
    k = tipo(ic, pir)
    if k in fil: ax.plot([tt * 1e3] * 2, [fil[k], fil[k] + 0.7], color=['#888', '#2F6DB5', '#F2A33A', '#E2453A'][fil[k]])
ax.set_yticks([0.35, 1.35, 2.35, 3.35]); ax.set_yticklabels(list(fil)); ax.set_xlabel('ms'); ax.set_title('Cada raya es una entrada a la ISR, según la bandera que la provocó'); plt.show()

### Ejemplo 10 · Reloj con TMR1
Precarga $= 65536 - \dfrac{0{,}5\ \text{s}}{8 \cdot 1\ \mu s} = 3036$ = 0x0BDC. Se mide el periodo real del LED que cambia en cada interrupción.

In [ ]:
p = nuevo('ej10_isr_reloj'); lcd = LCD16x2(p); cambios = []; ant = p.puerto('C') & 1
for k in range(4000):
    p.correr(segundos=0.001); v = p.puerto('C') & 1
    if v != ant: cambios.append(p.tiempo()); ant = v
d = np.diff(cambios)[1:]          # el primer intervalo incluye el arranque (LCD y configuración)
print(f'TMR1: {len(d)} intervalos medidos · periodo medio {d.mean()*1e3:.2f} ms (ideal 500 ms, resolución de la medida 1 ms)')
ver_lcd(lcd, 'después de 4 s')
pulsar(p, 'B', 0); pulsar(p, 'B', 1); ver_lcd(lcd, 'después de +1 min (INT) y +1 hora (RB1)')

## 10. Multitarea cooperativa
El PIC16 no tiene memoria ni pila para un RTOS con desalojo (8 niveles de pila por hardware y 368 bytes de RAM). La solución práctica es un **planificador cooperativo**: la ISR de TMR0 cuenta *ticks* de 10 ms y el lazo principal reparte esos ticks entre tareas cortas que **nunca esperan** (sin retardos bloqueantes): cada una hace un paso y devuelve el control.

In [ ]:
p = nuevo('ej11_multitarea_planificador'); lcd = LCD16x2(p); p.analog[0] = 2.0
p.correr(segundos=0.2); dt = 0.002; reg = []
for k in range(1500):
    if k in (300, 600, 900): p.poner_entrada('B', 0xFE)
    if k in (330, 630, 930): p.poner_entrada('B', 0xFF)
    if k == 750: p.analog[0] = 4.5
    p.correr(segundos=dt); c = p.puerto('C'); reg.append((c & 1, (c >> 1) & 1, p.puerto('B') & 1 ^ 1, p.pwm(1)['D']))
R = np.array(reg); t = np.arange(len(R)) * dt
fig, ax = plt.subplots(figsize=(11, 3.4))
for k, (nm, col) in enumerate((('T1 · LED RC0 (250 ms)', '#2E9B57'), ('T2 · LED RC1 (400 ms)', '#E2453A'), ('pulsador RB0', '#F2A33A'), ('T4 · PWM RC2 (ciclo útil)', '#2F6DB5'))):
    ax.fill_between(t, 3 - k, 3 - k + 0.75 * R[:, k], step='post', color=col, alpha=.85)
ax.set_yticks([3.35, 2.35, 1.35, 0.35]); ax.set_yticklabels(['T1 · RC0 250 ms', 'T2 · RC1 400 ms', 'RB0 (T3 lo cuenta)', 'T4 · PWM']); ax.set_xlabel('s')
ax.set_title('Diagrama de tiempo: las tareas avanzan a la vez'); plt.show()
ver_lcd(lcd, 'T5 actualiza la LCD cada 500 ms')

### Ejemplo 12 · Semáforo + latido + comandos
Tres tareas concurrentes; la del semáforo es una máquina de estados. Se envía `P` por la USART a los 6 s.

In [ ]:
p = nuevo('ej12_multitarea_semaforo'); p.correr(segundos=0.1); print(terminal(p).strip())
hist, buf = [], ''
for k in range(260):
    if k == 60: p.recibir(ord('P'))
    p.correr(segundos=0.1); hist.append(p.puerto('B')); buf += terminal(p)
    while '\r\n' in buf:
        linea, buf = buf.split('\r\n', 1)
        if linea: print(f'{p.tiempo():5.1f} s  {linea}')
H = np.array([[(v >> b) & 1 for v in hist] for b in (0, 1, 2, 3, 4, 5, 7)])
cols = ['#E2453A', '#F2B33A', '#2E9B57', '#E2453A', '#2E9B57', '#888888', '#3060FF']
nom = ['rojo V', 'amarillo V', 'verde V', 'rojo P', 'verde P', 'espere', 'latido']
plt.figure(figsize=(11, 3.2)); tt = np.arange(len(hist)) * 0.1
for b in range(7): plt.fill_between(tt, b, b + 0.8 * H[b], step='post', color=cols[b])
plt.yticks(np.arange(7) + 0.4, nom); plt.xlabel('s'); plt.axvline(6, ls=':', color='k'); plt.title("Semáforo: 'P' (línea punteada) pide el paso peatonal"); plt.show()

## 11. Ejercicios propuestos
1. **USART**: agregue al ejemplo 1 el comando `Tn` que invierta todos los LED cada `n` décimas de segundo.
2. **USART**: cambie el ejemplo 2 a 19200 baudios (¿qué SPBRG? ¿qué error?) y ajuste `graficar_serie.py`.
3. **I2C**: lea los botones del PCF8574 (P4-P7 como entradas) y muestre su estado en los LED P0-P3.
4. **I2C**: agregue una alarma al ejemplo 4 guardada en la RAM del DS1307 (registros 0x08-0x3F).
5. **SPI**: encadene dos 74HC595 (Q7' → DS) y haga un barrido de 16 LED.
6. **PWM**: agregue una segunda melodía y un botón para elegirla.
7. **Interrupciones**: mida con TMR1 el tiempo entre dos pulsaciones de RB0 (cronómetro de reacción).
8. **Multitarea**: agregue una sexta tarea que envíe el contador de pulsos por la USART cada segundo.

Plantilla: escriba un programa, ensámblelo, simúlelo y descargue su `.hex`.

In [ ]:
MI_PROGRAMA = """
        LIST    P=16F877A, R=DEC
        #include <p16f877a.inc>
        __CONFIG _XT_OSC & _WDT_OFF & _PWRTE_ON & _BOREN_OFF & _LVP_OFF & _CP_OFF
        ORG     0
        BANKSEL SPBRG
        movlw   25                      ; 9600 baudios con 4 MHz (BRGH = 1)
        movwf   SPBRG
        movlw   b'00100100'             ; TXEN, BRGH
        movwf   TXSTA
        BANKSEL RCSTA
        bsf     RCSTA,SPEN
LAZO    movlw   'H'
        call    TX
        movlw   'i'
        call    TX
        goto    LAZO
TX      btfss   PIR1,TXIF
        goto    TX
        movwf   TXREG
        return
        END
"""
a = Ensamblador(MI_PROGRAMA).ensamblar(); open('mi_programa.hex', 'w').write(a.hex())
p = PIC16F877(a.mem); p.correr(segundos=0.02)
print('USART:', terminal(p)[:20], '· mi_programa.hex listo para Proteus')

## 12. Simulador interactivo en el navegador
Suba `simulador_pic16f877a_buses_interrupciones.html` (entregado con las diapositivas) y ejecute la celda.

In [ ]:
import os, html as _html
from IPython.display import HTML, display
f = 'simulador_pic16f877a_buses_interrupciones.html'
if not os.path.exists(f):
    try:
        from google.colab import files; files.upload()
    except ImportError:
        print('Copie ' + f + ' a esta carpeta.')
if os.path.exists(f):
    display(HTML('<iframe srcdoc="' + _html.escape(open(f, encoding='utf-8').read(), quote=True) + '" style="width:100%;height:900px;border:1px solid #D5DEE2;border-radius:12px" scrolling="yes"></iframe>'))